# Carnatic Music Processing

Notebook Outline

0. Preprocessing
1. Understanding the Swara
2. Tonic and Label Estimation
3. Contour Segmentation
4. Tempo Estimation
5. Dynamic Programming
6. Summary and Future Work

## 0. Preprocessing

In [ ]:
# FILE PROCESSING
import os
import yaml
import xml.etree.ElementTree as ET
from xml.dom import minidom

# AUDIO PROCESSING
import librosa as lb
import librosa.display as ld
from IPython.display import Audio
import soundfile as sf

# PLOTTING
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import matplotlib.cm as cm
import seaborn as sns

# DATA PROCESSING
import numpy as np
import pandas as pd
from IPython.display import display

from scipy.signal import medfilt
from scipy.interpolate import interp1d
from scipy.ndimage import median_filter

# STATISTICS
import scipy.stats as stats
from scipy.stats import skew, kurtosis
from scipy.stats import vonmises, gaussian_kde
from scipy.special import i0
from scipy.stats import ttest_ind
from scipy.stats import norm

from scipy.integrate import simpson
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress

# OPTIMIZATION
from scipy.signal import find_peaks
from scipy.optimize import minimize
from scipy.optimize import linear_sum_assignment

### Load Dataset

In [ ]:
# FUNCTIONS

def load_dataset(directory_path, ftype = 'mp3'):
    '''
    Inputs:
    1. directory_path (STR)

    Outputs:
    1. data: (DICT) of form {id: TUPLE(file path (STR), audio (ARR), sampling rate (INT)))}
    '''
    data = {}
    try:
        items = [(directory_path + '/' + f) for f in os.listdir(directory_path) if f.endswith(ftype)]
    except FileNotFoundError:
        print(f"Error: Directory '{directory_path}' not found.")
        exit()
    for i in range(len(items)):
        item = items[i]
        x, sr = lb.load(item, sr = 44100)
        data[i] = (item, x, sr)
    return data

In [ ]:
# TEST
dataset = load_dataset('carnatic_varnam_1.0/Audio')

### Spectrogram and Contour Generation

In [ ]:
# FUNCTIONS
# Generate and plot the spectrogram, fundamental frequency contour, and fundamental pitch contour

def clip_sample(x, sr, duration):
    '''
    Inputs:
    1. x: iterable object representing audio (ARR)
    2. sr: sampling rate at which x was taken (samples / sec) (INT)
    3. duration: desired duration (in seconds) of clipped output (FLOAT)

    Outputs:
    1. clip: iterable object representing clipped audio (ARR)
    '''
    # determine number of samples in clipped version
    samples = duration * sr
    if duration > 0 and duration <= (len(x) / sr): # check that clipping duration is valid
        clip = x[:samples]
    elif duration > (len(x) / sr):
        clip = np.pad(x, (0, samples - len(x)))
    else:
        clip = x
    return clip

def plot_spec(spec, sr, hop_length, n_fft = 4096, new = True):
    '''
    Inputs:
    1. spec: spectrogram (output of stft on audio) (2D ARR)
    2. sr: sampling rate (INT)
    3. hop_length: hop_length used when computing stft to generate spec (INT)
    4: n_fft: n_fft (window length?) used when computing stft to generate spec (INT)
    5: new: whether a new plot has to be initialized (BOOL)

    Outputs:
    1. display of spectrogram
    '''
    if new:
        plt.figure(figsize=(12, 6))
    # librosa has 'fft_svara' as an axis type :0
    lb.display.specshow(spec, sr=sr, hop_length = hop_length, n_fft = n_fft, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.show()

def get_spec(x, sr, n_fft = 4096, hop_length = 1024, duration = 25, display = True):
    '''
    Inputs:
    1. x: audio (ARR)
    2. sr: sampling rate (INT)
    3. hop_length: hop_length for stft to generate spec (INT)
    4: n_fft: n_fft (window length?) for stft to generate spec (INT)
    5: duration: length (in seconds) of audio being used (FLOAT)
    6: display: whether or not to output plot (BOOL)

    Outputs:
    1. spec (2D ARR)
    '''
    clip = clip_sample(x, sr, duration)
    spec = lb.amplitude_to_db(np.abs(lb.stft(clip, n_fft = n_fft, hop_length = hop_length)), ref=np.max)

    if display:
        plot_spec(spec, sr, hop_length, n_fft)
    return spec

def plot_f0(f0, sr, hop_length = 1024, mode = 'freq', new = True):
    '''
    Inputs:
    1. f0: fundamental pitch contour (ARR)
    2. sr: sampling rate of audio for which f0 was generated (INT)
    3. mode: freq/midi (STR)
    4. new: whether a new plot has to be initialized (BOOL)

    Outputs:
    1. display of f0 contour
    '''
    if new:
        plt.figure(figsize=(12, 6))
    times = lb.times_like(f0, sr=sr, hop_length=hop_length)
    plt.plot(times, f0, label='f0 contour', color='r')
    plt.xlabel('Time (s)')
    if mode == 'freq': 
        plt.ylabel('Frequency (Hz)')
        plt.title('Fundamental Frequency Contour (f0)')
    if mode == 'midi':
        plt.ylabel('MIDI')
        plt.title('Fundamental Pitch Contour (p0)')
    plt.legend()
    plt.show()

def get_f0(x, sr, tonic = None, n_fft = 4096, hop_length = 1024, duration = -1, display = True):
    '''
    Inputs:
    1. x: audio (ARR)
    2. sr: sampling rate (INT)
    3. hop_length: hop_length for pyin (INT)
    4. duration: length (in seconds) of audio being used (FLOAT)
    5. display: whether or not to output plot (BOOL)

    Outputs:
    1. f0: (ARR)
    2. display of f0 contour
    '''
    clip = clip_sample(x, sr, duration)
    fmin = 10
    fmax = 2000

    if tonic:
        fmin = tonic*0.6
        fmax = tonic*3

    f0, voiced_flag, voiced_probs = lb.pyin(clip, fmin = fmin, fmax = fmax, sr = sr, frame_length = n_fft, hop_length=hop_length)

    if display:
        plot_f0(f0, sr, hop_length, 'freq')
    return f0, voiced_flag, voiced_probs

def get_p0(f0, sr, hop_length = 1024, tonic = None, display = True):
    '''
    Inputs:
    1. f0: freq contour (ARR)
    2. sr: sampling rate (INT)
    3. tonic: frequency of tonic (FLOAT)
    4. display: whether or not to output plot (BOOL)

    Outputs:
    1. p0: (ARR)
    2. display of p0 contour
    '''

    tmidi = 0
    if tonic:
        tmidi = lb.hz_to_midi(tonic)
    p0 = lb.hz_to_midi(f0) - tmidi

    if display:
        plot_f0(p0, sr, hop_length, 'midi')
    
    return p0

def plot_spec_f0(spec, f0, sr, hop_length = 1024):
    '''
    Inputs:
    1. spec: (2D ARR)
    2. f0: (ARR)
    3. sr: sampling rate of original audio (INT)
    4. hop_length: hop_length used to generate f0 and spec (INT)

    Outputs:
    1. plot of spec with f0 overlaid
    '''

    times = lb.times_like(f0, sr=sr, hop_length=hop_length)

    # Plot spectrogram
    plt.figure(figsize=(12, 6))
    lb.display.specshow(spec, sr=sr, hop_length=hop_length, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')

    # Overlay f0 contour
    plt.plot(times, f0, label='f0 contour', color='r', linewidth=2)

    # Labels and title
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.title('Spectrogram with f0 Contour')
    plt.legend()

    plt.show()

In [ ]:
# FUNCTIONS
# Smooth contours using median filtering

def med_filter(signal, kernel_size = 3, mode = 'freq'):
    '''
    Inputs:
    1. signal: contour (ARR)
    2. kernel_size: size of neighborhood being used for smoothing (INT)
    3. lbound: lower bound on signal values for pruning (FLOAT)
    4. ubound: upper bound on signal values for pruning (FLOAT)

    Outputs:
    1. signal_filtered: smoothed (using median filtering) and filtered (using frequency bounds) version of input (ARR)
    '''
    # Determine bounds based on type of contour
    if mode == 'freq':
        lbound = 75
        ubound = 750
    else:
        lbound = -12
        ubound = 24

    # Apply median filtering to remove noise
    signal_filtered = medfilt(signal, kernel_size=kernel_size)

    # Remove out-of-bounds values
    signal_filtered = np.where((signal_filtered > lbound) & (signal_filtered < ubound), signal_filtered, np.nan)
    return signal_filtered

def smooth_signal(signal, kernel_size=5):
    '''
    Inputs:
    1. signal: contour (ARR)
    2. kernel_size: size of neighborhood being used for smoothing (INT)

    Outputs:
    1. signal smoothed using averaging
    '''
    kernel = np.ones(kernel_size) / kernel_size
    return np.convolve(signal, kernel, mode='same')

def smooth_contour(c0, sr, kernel_size = 3, hop_length = 1024, mode = 'freq', display = True, new = True):
    '''
    Inputs:
    1. c0: input contour, either freq or pitch (ARR)
    2. sr: sampling rate of original audio (INT)
    3. kernel_size: size of neighborhood being used for smoothing (INT)
    4. hop_length: hop_length used to generate f0 from original audio with sampling rate sr (INT)
    5. display: (BOOL)
    6. new: (BOOL)

    Outputs:
    1. c0_smooth: smoothened f0 (ARR)
    '''
    # fmin and fmax roughly extrapolated from tonic ranges (110-250) to cover two octave range
    # Apply median filtering to remove noise
    #     
    c0_filtered = med_filter(c0, kernel_size, mode)

    # Interpolate to fill missing values
    times = lb.times_like(c0, sr=sr, hop_length=hop_length)
    valid = ~np.isnan(c0_filtered)
    interp_func = interp1d(times[valid], c0_filtered[valid], kind='linear', fill_value="extrapolate")
    c0_smooth = interp_func(times)

    if display:
        plot_f0(c0_smooth, sr, mode, new)
    return c0_smooth

def smooth_voiced_sections(f0, sr, voiced_flag, filter_size=5, hop_length = 1024, mode = 'freq', display = True, new = True):
    f0_smoothed = f0.copy()

    # Find contiguous voiced regions
    start = None
    for i, v in enumerate(voiced_flag):
        if v:
            if start is None:
                start = i
        elif start is not None:
            # Smooth this voiced segment
            segment = f0[start:i]
            if len(segment) >= filter_size:
                f0_smoothed[start:i] = median_filter(segment, size=filter_size)
            start = None
    # Handle last segment if it ends on a voiced region
    if start is not None:
        segment = f0[start:]
        if len(segment) >= filter_size:
            f0_smoothed[start:] = median_filter(segment, size=filter_size)

    if display:
        plot_f0(f0_smoothed, sr, hop_length, mode, new)
    return f0_smoothed

In [ ]:
# TEST
# Load an example file from the dataset

example = 'carnatic_varnam_1.0/Audio/223586__gopalkoduri__carnatic-varnam-by-badrinarayanan-in-kalyani-raaga.mp3'
ex_audio, ex_sr = lb.load(example, sr = 44100)
display(Audio(data = ex_audio, rate = ex_sr))

In [ ]:
# TEST
# Generate a spectrogram for an audio input
ex_clip = clip_sample(ex_audio, ex_sr, duration = 80)
ex_spec = get_spec(ex_clip, ex_sr, duration = 80)

In [ ]:
# TEST
# Obtain fundamental frequency and pitch contours from input audio
ex_tonic_hz = 138.59
ex_f0, ex_vflags, ex_vprobs = get_f0(ex_audio, ex_sr, ex_tonic_hz)
ex_f0_clip, _, _ = get_f0(ex_clip, ex_sr, ex_tonic_hz)
ex_p0 = get_p0(ex_f0, ex_sr, tonic = ex_tonic_hz)
plot_spec_f0(ex_spec, ex_f0_clip, ex_sr)

In [ ]:
# TEST
# Smooth extracted contours (interpolation only done in voiced sections to avoid generating false data in silent portions, filter size chosen empirically to preserve oscillations while dampening noise)
ex_f0_smooth = smooth_voiced_sections(ex_f0, ex_sr, ex_vflags, filter_size = 7, mode = 'freq')
ex_p0_smooth = smooth_voiced_sections(ex_p0, ex_sr, ex_vflags, filter_size = 7, mode = 'midi')

### Annotation Parsing

In [ ]:
# FUNCTIONS
# Extract raw annotation information regarding note labels and beat locations

def parse_svl(svl_file):
    '''
    Inputs:
    1. svl_file: (STR)

    Outputs:
    1. frames: frame indices of beat annotations (ARR)
    '''

    tree = ET.parse(svl_file)
    root = tree.getroot()

    frames = []
    for point in root.findall(".//point"):  # Iterate over all "point" elements
        frame = int(point.attrib["frame"])  # Extract the "time" attribute
        frames.append(frame)

    for model in root.findall(".//model"):
        sample_rate = model.get('sampleRate')
    
    if sample_rate:
        factor = 44100 / float(sample_rate)
        frames = [int(frame * factor) for frame in frames]
    return frames

def parse_yaml(file_path, mode=1):
    """
    Inputs:
    1. file_path: (STRING)
    2. mode: 1 for first half of varnam, 2 for entire thing (INT)

    Outputs:
    1. notes: list of string annotations for every akshara (LIST)
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    def flatten(nested_list):
        """Recursively flattens a nested list structure."""
        flat_list = []
        for item in nested_list:
            if isinstance(item, list):
                flat_list.extend(flatten(item))
            else:
                flat_list.append(item)
        return flat_list

    # Extract and flatten required sections
    sections = ["pallavi", "anupallavi", "muktayiswaram"]
    notes = flatten([data.get(sec, []) for sec in sections])

    if mode == 1:
        return notes  # Return only pallavi, anupallavi, and muktayiswaram as a flat list

    elif mode == 2:
        notes.append("*")  # Add "*" after muktayiswaram for purvangam second speed
        notes = notes * 2  # already has another "*" built in for return to first speed

        # Process charanam and chittaiswaram in alternating fashion
        charanam = flatten(data.get("charanam", []))
        chittaiswarams = [flatten(swaram) for swaram in data.get("chittiswaram", [])]

        for swaram in chittaiswarams:
            notes.extend(charanam)  # Add charanam (without "*")
            notes.extend(swaram)    # Add chittaiswaram
            notes.append("*")       # Add "*" after chittaiswaram
            notes.extend(charanam)  # Add charanam (without "*")
            lines = len(swaram) // 32
            if lines % 2 == 0:
                notes.extend(charanam)
            notes.extend(swaram)    # Add chittaiswaram
            notes.append("*") 
    return notes

In [ ]:
# FUNCTIONS
# Postprocess extracted information to align note labels with beat locations

def cut_reps(frames):
    avglen = int(np.mean(np.array([frames[i+1] - frames[i] for i in range(4)])))
    edited = []
    for i in range(len(frames) - 1):
        if frames[i+1] - frames[i] <= 2 * avglen:
            edited.append((frames[i], frames[i+1]))
        else:
            edited.append((frames[i], frames[i] + avglen))
    return edited

def subdivide(arr, n, end=None):
    """
    Goal: Given an np array of beat tuples, return a new array where each interval 
    is subdivided into n uniform regions, inserting (n-1) intermediate values.

    Inputs:
    1. arr: (ARR)
    2. n: number of subdivisions / multiplicative factor (INT)

    Outputs:
    1. result: arr with interspersed subdivisions (ARR)
    """
    if len(arr) < 2 or n < 1:
        return arr  # No subdivisions needed if there's 0 or 1 element or n < 1

    result = []
    for i in range(len(arr)):
        start, end = arr[i]

        # Generate n points including start and end
        subdivided = np.linspace(start, end, n + 1, dtype=int)  # Exclude the last point
        result.extend([(subdivided[i], subdivided[i+1]) for i in range(len(subdivided) - 1)])
    
    return np.array(result)

def subdivide_yaml(notes):
    """
    Goal: take yaml annotations and duplicate to account for second speed sections
    """
    doubled = True
    output = []
    for note in notes:
        if note == "*":
            doubled = not doubled
        else:
            output.append(note)
            if doubled:
                output.append(',')
    return output

In [ ]:
# FUNCTIONS
# Save modified annotations to be viewed in Sonic Visualizer

def save_svl_annotation(point_list, filename, sample_rate=22050, resolution=1):
    # Sort and strip just the frames
    frames = [int(a) for a, _ in point_list]
    if not frames:
        raise ValueError("Point list is empty")

    start_frame = min(frames)
    end_frame = max(frames)

    # Create root
    sv = ET.Element("sv")

    # Data block
    data = ET.SubElement(sv, "data")
    model = ET.SubElement(data, "model", {
        "dataset": "2", "dimensions": "1", "end": str(end_frame),
        "id": "3", "name": "", "notifyOnAdd": "true",
        "resolution": str(resolution), "sampleRate": str(sample_rate),
        "start": str(start_frame), "type": "sparse"
    })
    dataset = ET.SubElement(data, "dataset", {"dimensions": "1", "id": "2"})

    # Label counter
    for i, frame in enumerate(frames):
        measure = i // 8 + 1
        beat = i % 8 + 1
        label = f"{measure}.{beat}"
        ET.SubElement(dataset, "point", {"frame": str(frame), "label": label})

    # Display block
    display = ET.SubElement(sv, "display")
    layer = ET.SubElement(display, "layer", {
        "colour": "#c832ff", "colourName": "Purple",
        "darkBackground": "false", "id": "0", "model": "3",
        "name": "Time Instants", "plotStyle": "0", "type": "timeinstants"
    })

    # Prettify and save
    xml_str = minidom.parseString(ET.tostring(sv)).toprettyxml(indent="  ")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(xml_str)

    print(f"SVL file saved to {filename}")


In [ ]:
# TEST
# Parse SVL file to get beat frames and subdivide

ex_svl = 'carnatic_varnam_1.0/Notations_Annotations/annotations/taalas/kalyani/badrinarayanan.svl'
ex_parsed_svl = cut_reps(parse_svl(ex_svl))
ex_beat_frames = subdivide(ex_parsed_svl, 4, len(ex_audio))

print('Beat Frame Subdivision')
print(ex_parsed_svl[0])
print(ex_beat_frames[:4])

generated_svl = save_svl_annotation(ex_beat_frames, 'example_gen.svl')

In [ ]:
# TEST
# Parse yaml file to get note labels and subdivide

ex_yaml = 'carnatic_varnam_1.0/Notations_Annotations/notations/kalyani.yaml'
ex_parsed_yaml = parse_yaml(ex_yaml, 2)
ex_note_labels = subdivide_yaml(ex_parsed_yaml)

print('Note Label Subdivision')
print(ex_parsed_yaml[:8])
print(ex_note_labels[:16])

In [ ]:
# Note: there is a discrepancy in length between the two lists
print('Lengths (Beat Frames, Note Labels)')
print(len(ex_beat_frames), len(ex_note_labels))

## 1. Understanding the Swara

### Segment Dataframe Generation + Analysis

In [ ]:
# FUNCTIONS

def get_segment_df(audio, timestamps, labels, p0, hop_length = 1024):
    """
    Segment audio into chunks corresponding to each note using beat timestamps and pitch contour.

    Inputs:
    1. audio: input signal (ARR)
    2. timestamps: frame indices of audio corresponding to beat divisions (ARR)
    3. labels: list of note labels for each annotated beat, with ',' indicating sustained notes (LIST)
    4. f0: fundamental pitch contour (ARR)
    5. hop_length: hop length used to compute f0 (INT)

    Ouputs:
    1. note_segments: list of dictionaries representing each note as {"audio": (ARR), "label": (STR), "f0": (ARR)} (LIST[DICT])
    """
    df = pd.DataFrame(columns=['id', 'swara', 'audio', 'p0', 'time_start', 'time_end', 'p0_start', 'p0_end'])

    start_idx = 0  # Index of first beat for each note
    segment_idx = 0
    curr_label = labels[0]

    for i, label in enumerate(labels):
        if label != ',' and i > 0:
            frame1 = timestamps[start_idx][0]
            frame2 = timestamps[i-1][1]
            seg_audio = audio[frame1: frame2]
            f0frame1 = int(frame1 // hop_length)
            f0frame2 = int(frame2 // hop_length)
            seg_f0 = p0[f0frame1:f0frame2]
            df.loc[len(df)] = [segment_idx, curr_label, seg_audio, seg_f0, frame1, frame2, f0frame1, f0frame2]
            start_idx = i
            segment_idx += 1
            curr_label = label
        else:
            continue
            
    return df

def add_labels(df, swara_type):
    df['label'] = df['swara'].map(swara_type)
    return df

In [ ]:
def append_stats(df, data_col):
    def safe_skew(x): return skew(x[~np.isnan(x)]) if np.any(~np.isnan(x)) else np.nan
    def safe_kurtosis(x): return kurtosis(x[~np.isnan(x)]) if np.any(~np.isnan(x)) else np.nan

    df['mean'] = df[data_col].apply(lambda x: np.nanmean(x))
    df['median'] = df[data_col].apply(lambda x: np.nanmedian(x))
    df['stdev'] = df[data_col].apply(lambda x: np.nanstd(x))
    df['range'] = df[data_col].apply(lambda x: np.nanmax(x) - np.nanmin(x) if np.any(~np.isnan(x)) else np.nan)
    df['skew'] = df[data_col].apply(safe_skew)
    df['kurtosis'] = df[data_col].apply(safe_kurtosis)
    return df

def oct_inv_col(df, data_col, center_col):
    '''
    '''
    new_col = data_col + '_octinv'
    df.loc[:, new_col] = df.apply(lambda x: (x[data_col] + (6 -x[center_col])) % 12 - (6 -x[center_col]), axis = 1)
    return df

In [ ]:
# FUNCTIONS
# Dataframe debugging helper functions

def plot_actual_vs_theoretical(p0, df, sr=44100, hop_length=1024):
    # Time axis
    times = np.arange(len(p0)) * hop_length / sr

    # Initialize theoretical pitch array
    theoretical_p0 = np.full_like(p0, fill_value=np.nan, dtype=np.float32)

    # Fill theoretical pitch from note labels
    for _, row in df.iterrows():
        label = int(row['label'])  # 0–11 semitone
        mean_pitch = row['mean']
        start = int(row['p0_start'])
        end = int(row['p0_end'])

        # Map label to nearest correct octave
        octave_offset = round((mean_pitch - label) / 12)
        corrected_label = label + 12 * octave_offset

        theoretical_p0[start:end] = corrected_label 
    
    

    # Plot
    plt.figure(figsize=(14, 6))
    
    # Faint horizontal lines for semitone levels 0–11
    for semitone in range(12):
        plt.axhline(y=semitone, color='gray', linestyle='--', alpha=0.2)
        plt.text(times[-1] + 0.1, semitone, f'{semitone}', 
                 va='center', ha='left', fontsize=9, alpha=0.4)

    plt.plot(times, p0, label='Actual Pitch Contour', linewidth=1.5)
    plt.plot(times, theoretical_p0, label='Theoretical (Segment Labels)', 
             linestyle='--', linewidth=2, color='orange')

    plt.xlabel("Time (s)")
    plt.ylabel("Pitch (MIDI)")
    plt.title("Actual vs. Theoretical Pitch Contour")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def hear_seg(segdf, idx, sr = 44100):
    notes = segdf.iloc[idx]["audio"]
    display(Audio(data = notes, rate = sr))
    return notes

def get_context(segdf, idx, sr = 44100, padding = 2):
    notes = np.concatenate(tuple(segdf.iloc[idx-padding:idx+padding]["audio"]))
    display(Audio(data = notes, rate = sr))
    return notes

def get_outliers(segdf, bfunc):
    for index, row in segdf.iterrows():
        if bfunc(row):
            display(segdf.iloc[index])
            hear_seg(segdf, index)
            get_context(segdf, index)
    return None

In [ ]:
# TEST
# Segment dataframe generation and processing for a single example

ex_swara_type = {'S': 0, 'R': 2, 'G': 4, 'M': 6, 'P': 7, 'D': 9, 'N': 11}
ex_segdf = get_segment_df(ex_audio, ex_beat_frames, ex_note_labels, ex_p0_smooth)
ex_segdf = add_labels(ex_segdf, ex_swara_type)

ex_segdf = append_stats(ex_segdf, 'p0')
ex_segdf = oct_inv_col(ex_segdf, 'mean', 'label')
ex_segdf = oct_inv_col(ex_segdf, 'median', 'label')
ex_segdf['tag'] = 'actual'

ex_segdf = ex_segdf.dropna()
ex_segdf.head()

### Dataset-Wide Segment Dataframe

In [ ]:
# GLOBAL VALUES
# Tonic and swara label information for all artists and ragas in the dataset

tonics = {'badrinarayanan': 138.59,
    'dharini': 200.58,
    'prasanna': 146.83,
    'ramakrishnamurthy': 149.4,
    'sreevidya': 210.07,
    'vignesh': 138.59}

ragadicts = {'abhogi': {'S': 0, 'R': 2, 'G': 3, 'M': 5, 'D': 9},
    'begada': {'S': 0, 'R': 2, 'G': 4, 'M': 5, 'P': 7, 'D': 9, 'N': 10},
    'kalyani': {'S': 0, 'R': 2, 'G': 4, 'M': 6, 'P': 7, 'D': 9, 'N': 11},
    'mohanam': {'S': 0, 'R': 2, 'G': 4, 'P': 7, 'D': 9},
    'sahana': {'S': 0, 'R': 2, 'G': 4, 'M': 5, 'P': 7, 'D': 9, 'N': 10},
    'saveri': {'S': 0, 'R': 1, 'G': 4, 'M': 5, 'P': 7, 'D': 8, 'N': 11},
    'sri': {'S': 0, 'R': 2, 'G': 3, 'M': 5, 'P': 7, 'D': 9, 'N': 10}}

In [ ]:
segdfs = []

for key, value in dataset.items():
    # extract metadata from file name
    item, x, sr = value
    artist = item.split("-")[3]
    raga = item.split("-")[5]

    # extract audio content and pitch contour
    xtonic_hz = tonics[artist]
    xf0, xvflags, xvprobs = get_f0(x, sr, xtonic_hz, display = False)
    xp0 = get_p0(xf0, sr, tonic = xtonic_hz, display = False)
    xp0_smooth = smooth_voiced_sections(xp0, sr, xvflags, filter_size = 7, display = False)
    
    # get beat annotations from sonic visualizer
    xsvl = 'carnatic_varnam_1.0/Notations_Annotations/annotations/taalas/' + raga + '/' + artist + '.svl'
    xparsed_svl = cut_reps(parse_svl(xsvl))
    xbeat_frames = subdivide(xparsed_svl, 4, len(x))

    # get label notations from manual annotation
    xyaml = 'carnatic_varnam_1.0/Notations_Annotations/notations/' + raga + '.yaml'
    xparsed_yaml = parse_yaml(xyaml, 2)
    xnote_labels = subdivide_yaml(xparsed_yaml)
    save_svl_annotation(xbeat_frames, artist + '_' + raga + '_gen.svl')
    
    if len(xbeat_frames) < len(xnote_labels):
        xnote_labels = xnote_labels[:len(xbeat_frames)]

    xswara_type = ragadicts[raga]
    xsegdf = get_segment_df(x, xbeat_frames, xnote_labels, xp0_smooth)
    xsegdf = add_labels(xsegdf, xswara_type)
    xsegdf = append_stats(xsegdf, 'p0')
    xsegdf = oct_inv_col(xsegdf, 'mean', 'label')
    xsegdf = oct_inv_col(xsegdf, 'median', 'label')
    xsegdf = xsegdf.dropna()

    xsegdf['raga'] = raga
    xsegdf['artist'] = artist
    segdfs.append(xsegdf)

all_segments = pd.concat(segdfs, ignore_index = True)


### Segment Statistic Plotting

In [ ]:
# GLOBAL VALUES

all_stats = ['mean_octinv','median_octinv', 'range', 'stdev', 'skew', 'kurtosis']

In [ ]:
# FUNCTIONS

def plot_df_stat(df, group_col, data_col, bins=50, alpha=0.5, chart='kde', ax=None, title=None):
    '''
    Plotting helper function for KDE or histogram grouped by a column.

    Parameters:
    - df: pandas DataFrame
    - group_col: column name to group by (e.g., 'label')
    - data_col: column with the numeric data to plot (e.g., 'mean')
    - bins: number of bins for histogram
    - alpha: transparency level for histograms
    - chart: 'kde' or 'hist'
    - ax: matplotlib Axes to draw on (optional)
    - title: plot title (optional)
    '''
    created_fig = False
    if ax is None:
        fig, ax = plt.subplots()
        created_fig = True  # Mark that we created the figure here

    fmin = df[data_col].min()
    fmax = df[data_col].max()
    grouped = df.groupby(group_col)[data_col]

    if chart == 'kde':
        x_values = np.linspace(fmin - 2, fmax + 2, 500)
        for key, values in grouped:
            values = values.dropna()
            if values.size > 1:
                kde = stats.gaussian_kde(values)
                ax.plot(x_values, kde(x_values), label=key)

        ax.set_xlabel('Semitones')
        ax.set_ylabel('Density')
        ax.set_title(title or f'Distribution of Segment {data_col} by {group_col}')

    elif chart == 'hist':
        for key, values in grouped:
            values = values.dropna()
            ax.hist(values, bins=bins, alpha=alpha, label=key)

        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')
        ax.set_title(title or f'Histograms of {data_col} by {group_col}')

    else:
        raise ValueError("chart must be either 'kde' or 'hist'")

    ax.legend()

    if created_fig:
        plt.show()


In [ ]:
def plot_grid_by_raga_and_artist(df, stat):
    """
    Creates a grid of mean-distribution plots where columns = ragas and rows = artists.
    """
    import matplotlib.pyplot as plt

    ragas = sorted(df['raga'].unique())
    artists = sorted(df['artist'].unique())

    nrows = len(artists)
    ncols = len(ragas)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharex=True, sharey=False)

    # If there's only one row or one column, axes may be 1D; convert to 2D for consistency
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes[np.newaxis, :]
    elif ncols == 1:
        axes = axes[:, np.newaxis]

    for i, artist in enumerate(artists):
        for j, raga in enumerate(ragas):
            ax = axes[i, j]
            sub_df = df[(df['artist'] == artist) & (df['raga'] == raga)]
            title = f"{artist} / {raga}"
            if len(sub_df) == 0:
                ax.set_visible(False)
            else:
                plot_df_stat(sub_df, 'label', stat, ax=ax, title=title)

    plt.tight_layout()
    plt.show()

def plot_grid_by_cat_and_stat(df, cat, stats=all_stats, group_col='label', chart='kde'):
    """
    Creates a grid of distribution plots where each row = statistic and each column = category (cat).
    Axes are shared within each row for consistent scales.
    """
    cats = sorted(df[cat].unique())
    n_rows = len(stats)
    n_cols = len(cats)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), squeeze=False)

    for i, stat in enumerate(stats):
        ref_ax = None  # reference axis for this row
        for j, c in enumerate(cats):
            ax = axes[i][j]

            # FIX: filter by category value, not loop index
            sub_df = df[df[cat] == c]

            # Use sharex/sharey (portable across Matplotlib versions)
            if ref_ax is None:
                ref_ax = ax
            else:
                ax.sharex(ref_ax)
                ax.sharey(ref_ax)

            title = f"{c} / {stat}"
            if sub_df.empty:
                ax.set_title(title + " (no data)")
                ax.axis('off')
                continue

            plot_df_stat(sub_df, group_col, stat, ax=ax, title=title, chart=chart)

            # Hide y-axis labels except for the first column
            if j > 0:
                ax.set_ylabel('')
            # Hide x-axis labels except for the last row
            if i < n_rows - 1:
                ax.set_xlabel('')

    plt.tight_layout()
    plt.show()



In [ ]:
# TEST
# Statistics on a single sample
for stat in all_stats:
    plot_df_stat(ex_segdf, 'label', stat)

In [ ]:
for stat in all_stats:
    plot_grid_by_raga_and_artist(all_segments, stat)

In [ ]:
plot_grid_by_cat_and_stat(all_segments, 'raga')

### Evaluating Discriminative Power

#### Weighted Bhattacharyya Distance

In [ ]:
# FUNCTIONS
def compute_kde_bhattacharyya_matrix(df, value_col='mean_octinv', label_col='label', 
                                     wrap_mod_12=True, bandwidth=None, grid_res=500, 
                                     plot=True, compute_weighted=True,
                                     swara_to_pc=None, weight_power=1):
    """
    Compute and visualize Bhattacharyya distances between KDEs grouped by swara.

    Parameters:
    - df: DataFrame with one row per segment and columns for swara label and a value
    - value_col: Column name for the data to compute KDEs over
    - label_col: Column name for the swara labels
    - wrap_mod_12: Whether to wrap x-axis values mod 12 before KDE overlap computation
    - bandwidth: Optional bandwidth to use in KDE
    - grid_res: Number of points in the KDE grid
    - plot: Whether to show the heatmap of the distance matrix
    - compute_weighted: Whether to return weighted average using semitone distance
    - swara_to_pc: Dict mapping swara labels to pitch class (0–11)
    - weight_power: Use 1/distance^power as weights

    Returns:
    - distance_matrix: pd.DataFrame of Bhattacharyya distances
    - avg_distance: float, unweighted average (off-diagonal)
    - weighted_avg (optional): float, distance weighted by pitch proximity
    """
    from itertools import combinations

    kde_dict = {}
    labels = sorted(df[label_col].unique())

    # Build KDEs for each swara
    for label in labels:
        values = df[df[label_col] == label][value_col].dropna().values
        if len(values) < 2:
            continue
        kde = gaussian_kde(values, bw_method=bandwidth)
        kde_dict[label] = kde

    x_grid = np.linspace(0, 12 if wrap_mod_12 else df[value_col].max(), grid_res)

    def eval_kde(kde, x_grid, wrap_mod_12):
        if wrap_mod_12:
            return kde(x_grid) + kde(x_grid - 12) + kde(x_grid + 12)
        else:
            return kde(x_grid)

    kde_vals = {label: eval_kde(kde, x_grid, wrap_mod_12) for label, kde in kde_dict.items()}

    # Compute pairwise Bhattacharyya distances
    n = len(labels)
    distances = np.zeros((n, n))

    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if i == j:
                continue
            p = kde_vals[l1]
            q = kde_vals[l2]
            bc = simpson(np.sqrt(p * q), x_grid)
            distances[i, j] = -np.log(bc + 1e-12)  # avoid log(0)

    distance_matrix = pd.DataFrame(distances, index=labels, columns=labels)

    # Compute unweighted average (off-diagonal)
    avg_distance = distance_matrix.values[np.triu_indices(n, k=1)].mean()

    # Compute weighted average if requested
    weighted_avg = None
    if compute_weighted:
        weighted_sum = 0
        total_weight = 0
        for i, l1 in enumerate(labels):
            for j, l2 in enumerate(labels):
                if i >= j:
                    continue
                circ_dist = min(abs(l1 - l2), 12 - abs(l1 - l2))
                if circ_dist == 0:
                    continue
                weight = 1 / (circ_dist ** weight_power)
                weighted_sum += weight * distances[i, j]
                total_weight += weight
        weighted_avg = weighted_sum / total_weight if total_weight > 0 else None

    # Plot
    if plot:
        plt.figure(figsize=(8, 6))
        sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap="viridis", 
                    cbar_kws={"label": "Bhattacharyya Distance"})
        plt.title(f"Swara Distribution Similarity ({value_col})")
        plt.xlabel("Swara")
        plt.ylabel("Swara")
        plt.tight_layout()
        plt.show()

    if compute_weighted:
        return distance_matrix, avg_distance, weighted_avg
    else:
        return distance_matrix, avg_distance, None

def plot_bhattacharyya_heatmaps(
    df,
    stat_params=[
        ('mean_octinv', True),
        ('median_octinv', True),
        ('stdev', False),
        ('range', False),
        ('skew', False),
        ('kurtosis', False)
    ],
    n_cols=2,
    grid_res=500,
    bandwidth=None,
    vmin=0,
    vmax=5,
    figsize=(8, 10),
    compute_weighted = True,
    weight_power = 1
):
    """
    Compute and plot Bhattacharyya distance heatmaps for multiple statistics,
    with a clean shared colorbar off to the right.
    """
    n_plots = len(stat_params)
    n_rows = (n_plots + n_cols - 1) // n_cols

    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(n_rows, n_cols + 1, width_ratios=[1]*n_cols + [0.03], wspace=0.2)

    axes = []
    for i in range(n_plots):
        row = i // n_cols
        col = i % n_cols
        ax = fig.add_subplot(gs[row, col])
        axes.append(ax)

    # Add one axis for the colorbar
    cbar_ax = fig.add_subplot(gs[:, -1])  # last column spans all rows

    for i, (value_col, wrap_mod) in enumerate(stat_params):
        ax = axes[i]
        matrix, avg_dist, weighted_avg = compute_kde_bhattacharyya_matrix(
            df,
            value_col=value_col,
            wrap_mod_12=wrap_mod,
            grid_res=grid_res,
            bandwidth=bandwidth,
            compute_weighted=compute_weighted,
            weight_power = weight_power,
            plot=False
        )

        sns.heatmap(matrix, ax=ax, vmin=vmin, vmax=vmax, cmap="viridis",
                    annot=False, cbar=(i == 0), cbar_ax=cbar_ax if i == 0 else None,
                    square=True)
        ax.set_title(f"{value_col} (score: {weighted_avg:.2f})")
        ax.set_xlabel("")
        ax.set_ylabel("")

    # Hide any unused axes (if total subplots < rows*cols)
    for j in range(len(stat_params), n_rows * n_cols):
        fig.add_subplot(gs[j // n_cols, j % n_cols]).axis("off")

    cbar_ax.set_ylabel("Bhattacharyya Distance", rotation=270, labelpad=15)
    plt.tight_layout(rect=[0, 0, 0.96, 1])  # leave room on the right
    plt.show()

In [ ]:
# FUNCTIONS

def compute_raga_stat_distance_table(
    df,
    label_col='label',
    raga_col='raga',
    stat_params=[
        ('mean_octinv', True),
        ('median_octinv', True),
        ('stdev', False),
        ('range', False),
        ('skew', False),
        ('kurtosis', False)
    ],
    weight_power=1
):
    """
    Compute a raga × statistic table of weighted Bhattacharyya distances,
    with an average row and average column added.

    Returns:
    - df_summary: Pandas DataFrame with ragas as rows, stats as columns, and averages
    - latex_table: String containing LaTeX tabular environment
    """
    raga_list = sorted(df[raga_col].unique())
    stat_names = [s[0] for s in stat_params]
    data = []

    for raga in raga_list:
        subdf = df[df[raga_col] == raga]
        row = []
        for value_col, wrap_mod in stat_params:
            try:
                result = compute_kde_bhattacharyya_matrix(
                    subdf,
                    value_col=value_col,
                    label_col=label_col,
                    wrap_mod_12=wrap_mod,
                    plot=False,
                    compute_weighted=True,
                    weight_power=weight_power
                )
                _, _, weighted_avg = result
                row.append(round(weighted_avg, 2) if weighted_avg is not None else np.nan)
            except Exception:
                row.append(np.nan)
        data.append(row)

    # Build DataFrame
    df_summary = pd.DataFrame(data, index=raga_list, columns=stat_names)

    # Add row-wise mean (across stats) as a new column
    df_summary["Mean across stats"] = df_summary.mean(axis=1).round(2)

    # Add column-wise mean (across ragas) as a new row
    mean_row = df_summary.mean(axis=0).round(2)
    mean_row.name = "Mean across ragas"
    df_summary = pd.concat([df_summary, pd.DataFrame([mean_row])])

    # Convert NaNs to dashes for LaTeX output
    df_for_latex = df_summary.fillna("—")

    latex_table = df_for_latex.to_latex(
        index=True,
        float_format="%.2f",
        na_rep="—",
        column_format="l" + "c" * len(df_for_latex.columns),
        caption="Weighted Bhattacharyya distances between swaras for each statistic across ragas.",
        label="tab:weighted_bhattacharyya"
    )

    return df_summary, latex_table


In [ ]:
# TEST

sridf = all_segments[all_segments['raga'] == 'sri']
plot_bhattacharyya_heatmaps(sridf)

In [ ]:
# TEST
wbd_summary_df, wbd_latex_code = compute_raga_stat_distance_table(
    all_segments,
)

display(wbd_summary_df)

with open("weighted_bhattacharyya_table.tex", "w") as f:
    f.write(wbd_latex_code)

#### Total Exclusive Area

In [ ]:
# FUNCTIONS
def compute_exclusive_area_with_plot(
    df,
    value_col='mean_octinv',
    label_col='label',
    grid_res=1000,
    bandwidth=None,
    plot=True
):
    """
    Computes the total exclusive area between KDEs of swaras and optionally plots:
    1. All KDEs (after wrapping)
    2. Max and second max KDE values
    3. Exclusive area function to be integrated

    Returns:
    - total_exclusive_area: float
    """
    labels = sorted(df[label_col].unique())
    kde_dict = {}

    # Build KDEs
    for label in labels:
        values = df[df[label_col] == label][value_col].dropna().values
        if len(values) < 2:
            continue
        kde = gaussian_kde(values, bw_method=bandwidth)
        kde_dict[label] = kde

    # Grid
    x_grid = np.linspace(0, 12, grid_res)

    # Evaluate KDEs with wrapping
    kde_vals = []
    for label in labels:
        kde = kde_dict[label]
        kde_eval = kde(x_grid) + kde(x_grid - 12) + kde(x_grid + 12)
        kde_vals.append(kde_eval)

    kde_stack = np.vstack(kde_vals)  # shape: (n_labels, grid_res)

    # Sort values at each x
    sorted_vals = np.sort(kde_stack, axis=0)
    max_vals = sorted_vals[-1]
    second_max_vals = sorted_vals[-2] if len(labels) > 1 else np.zeros_like(max_vals)

    # Exclusive area function
    exclusive_area_curve = max_vals - second_max_vals

    # Area under the curve
    total_exclusive_area = simpson(exclusive_area_curve, x_grid)

    # Plot if requested
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))

        # Plot 1: All KDEs
        for kde_eval in kde_vals:
            axes[0].plot(x_grid, kde_eval, alpha=0.7)
        axes[0].set_title("All Wrapped KDEs")
        axes[0].set_xlabel("Semitone")
        axes[0].set_ylabel("Density")

        # Plot 2: Max and second max
        axes[1].plot(x_grid, max_vals, label='Primary Swara Density', color='blue')
        axes[1].plot(x_grid, second_max_vals, label='Secondary Swara Density', color='orange')
        axes[1].fill_between(x_grid, second_max_vals, max_vals, color='lightblue', alpha=0.3)
        axes[1].set_title("Top 2 KDE Curves")
        axes[1].set_xlabel("Semitone")
        axes[1].legend()

        # Plot 3: Exclusive area function
        axes[2].plot(x_grid, exclusive_area_curve, color='green')
        axes[2].fill_between(x_grid, 0, exclusive_area_curve, color='green', alpha=0.3)
        axes[2].set_title(f"Total Exclusive Area: {total_exclusive_area:.4f}")
        axes[2].set_xlabel("Semitone")
        axes[2].set_ylabel("Difference (max - 2nd max)")

        plt.tight_layout()
        plt.show()

    return total_exclusive_area

In [ ]:
# FUNCTIONS
def compute_and_format_exclusive_area_table(
    df,
    raga_col='raga',
    label_col='label',
    stat_params=[
        'mean_octinv',
        'median_octinv',
        'stdev',
        'range',
        'skew',
        'kurtosis'
    ],
    normalize_by_swara_count=True,
    grid_res=1000,
    bandwidth=None
):
    """
    Computes a raga × statistic table of exclusive area values (normalized if desired),
    adds mean rows/columns, and returns both the DataFrame and LaTeX string.

    Parameters:
    - df: Full DataFrame with pitch values, swara labels, and raga info
    - raga_col: Column name for raga identifiers
    - label_col: Column name for swara labels
    - stat_params: List of value column names to evaluate
    - normalize_by_swara_count: Whether to normalize by number of swaras in the raga
    - grid_res: Number of x-grid evaluation points for KDE
    - bandwidth: Optional KDE bandwidth override

    Returns:
    - df_summary: DataFrame with averages added
    - latex_str: LaTeX tabular string
    """
    raga_list = sorted(df[raga_col].unique())
    summary_data = []

    for raga in raga_list:
        subdf = df[df[raga_col] == raga]
        row = []
        swaras_in_raga = subdf[label_col].nunique()

        for stat in stat_params:
            try:
                area = compute_exclusive_area_with_plot(
                    subdf,
                    value_col=stat,
                    label_col=label_col,
                    grid_res=grid_res,
                    bandwidth=bandwidth,
                    plot=False
                )
                if normalize_by_swara_count and swaras_in_raga > 1:
                    area /= swaras_in_raga
                row.append(round(area, 4))
            except Exception:
                row.append(np.nan)

        summary_data.append(row)

    df_summary = pd.DataFrame(summary_data, index=raga_list, columns=stat_params)

    # Add row-wise mean (across stats)
    df_summary["Mean across stats"] = df_summary.mean(axis=1).round(4)

    # Add column-wise mean (across ragas)
    mean_row = df_summary.mean(axis=0).round(4)
    mean_row.name = "Mean across ragas"
    df_summary = pd.concat([df_summary, pd.DataFrame([mean_row])])

    # Convert to LaTeX
    df_for_latex = df_summary.fillna("—")

    latex_str = df_for_latex.to_latex(
        index=True,
        float_format="%.4f",
        na_rep="—",
        column_format="l" + "c" * len(df_for_latex.columns),
        caption="Normalized exclusive area between swaras for each statistic across ragas.",
        label="tab:exclusive_area"
    )

    return df_summary, latex_str



In [ ]:
# TEST
ex_exclusive_area = compute_exclusive_area_with_plot(
    ex_segdf, 
    value_col='mean_octinv', 
    label_col='label'
)

print(f"Exclusive Area: {ex_exclusive_area:.4f}")

In [ ]:
# TEST

tea_summary_df, tea_latex_code = compute_and_format_exclusive_area_table(
    all_segments,
    raga_col='raga',
    label_col='label',
    normalize_by_swara_count=True
)

display(tea_summary_df)

with open("exclusive_area_table.tex", "w") as f:
    f.write(tea_latex_code)

### N-Gram Studies

#### Context-Dependent Mean Shifting

In [ ]:
# FUNCTIONS

def bigram_plots(df):    
    # Compute contextual distances
    prev_means = df['mean'].shift(1)
    next_means = df['mean'].shift(-1)

    # Distances to previous and next note
    df['dist_to_prev'] = abs(df['mean'] - prev_means)
    df['dist_to_next'] = abs(df['mean'] - next_means)
    df['context_span'] = abs(prev_means - next_means)

    # Plot 1: Spread vs contextual distances
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].scatter(df['dist_to_prev'], df['stdev'], alpha=0.6)
    axes[0].set_title("Spread vs. Distance to Previous Note")
    axes[0].set_xlabel("Distance to Previous")
    axes[0].set_ylabel("Standard Deviation")

    axes[1].scatter(df['dist_to_next'], df['stdev'], alpha=0.6)
    axes[1].set_title("Spread vs. Distance to Next Note")
    axes[1].set_xlabel("Distance to Next")
    axes[1].set_ylabel("Standard Deviation")

    axes[2].scatter(df['context_span'], df['stdev'], alpha=0.6)
    axes[2].set_title("Spread vs. Span Between Context Notes")
    axes[2].set_xlabel("Span Between Context Notes")
    axes[2].set_ylabel("Standard Deviation")

    plt.tight_layout()
    plt.show()

    # Add signed distance from previous note (not circular)
    df['signed_dist_from_prev'] = df['mean'] - prev_means

    # Add direction of mean skew (mean - label)
    # Simulate 'mean' column around the label ± small bias
    df['mean_minus_label'] = df['mean_octinv'] - df['label']

    # Rebuild filtered, aligned arrays for regression (drop rows with any NaNs)
    df_clean = df[['signed_dist_from_prev', 'stdev', 'mean_minus_label']].dropna()

    X = df_clean['signed_dist_from_prev'].values.reshape(-1, 1)
    y1 = df_clean['stdev'].values
    y2 = df_clean['mean_minus_label'].values

    # Fit models
    model1 = LinearRegression().fit(X, y1)
    model2 = LinearRegression().fit(X, y2)

    y1_pred = model1.predict(X)
    y2_pred = model2.predict(X)

    # Compute R² values
    r2_1 = model1.score(X, y1)
    r2_2 = model2.score(X, y2)

    # Updated regression labels with R²
    reg_eq1 = f"y = {model1.coef_[0]:.3f}x + {model1.intercept_:.3f}\n$R^2$ = {r2_1:.3f}"
    reg_eq2 = f"y = {model2.coef_[0]:.3f}x + {model2.intercept_:.3f}\n$R^2$ = {r2_2:.3f}"

    # Plot 2: Mean shift vs signed distance with regression
    plt.figure(figsize=(6, 4))
    plt.scatter(X, y2, alpha=0.6, color='orange', label='Data')
    plt.plot(X, y2_pred, color='red', linestyle='--', label=reg_eq2)
    plt.axhline(0, linestyle='--', color='gray')
    plt.axvline(0, linestyle='--', color='gray')
    plt.title("Mean Shift vs. Signed Distance from Previous Note")
    plt.xlabel("Signed Distance from Previous Note")
    plt.ylabel("Segment Mean - Label")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# TEST
bigram_plots(ex_segdf) # one sample
bigram_plots(sridf) # all samples in sri

#### Anchor Pairs

In [ ]:
# GLOBAL VALUES

anchor_pairs_by_raga = {}

for raga, note_dict in ragadicts.items():
    semitones_in_raga = sorted(note_dict.values())
    anchors = []
    for s in semitones_in_raga:
        # Check if a neighbor semitone exists in the raga
        if (s - 1) % 12 in semitones_in_raga or (s + 1) % 12 in semitones_in_raga:
            anchors.append(s)
    anchor_pairs_by_raga[raga] = anchors

print(anchor_pairs_by_raga)

In [ ]:
# FUNCTIONS
def circular_dist(x, y):
    return np.minimum(np.abs(x - y), 12 - np.abs(x - y))

def plot_anchor_effect(ax, anchor_df, anchor_means, mean_all, title):
    sns.violinplot(data=all_segments, x='label', y='mean_octinv', inner=None, color='lightgray', linewidth=1, ax=ax)
    sns.violinplot(data=anchor_df, x='label', y='mean_octinv', inner=None, color='skyblue', linewidth=1, ax=ax)

    # Make violins translucent
    for violin in ax.collections:
        violin.set_alpha(0.5)

    # Step lines
    ax.step(mean_all.index, mean_all.values, where='mid', color='black', linewidth=1, label='Mean of All Notes')
    ax.step(anchor_means.index, anchor_means.values, where='mid', color='blue', linestyle='--', linewidth=1, label='Mean of Anchored Notes')
    # ax.step(range(12), range(12), where='mid', color='red', linewidth=1, label='Semitone Labels')

    # Titles and legend
    ax.set_title(title)
    ax.set_ylabel("Pitch")
    ax.legend(handles=[
        Patch(facecolor='lightgray', edgecolor='k', label='All Notes', alpha=0.5),
        Patch(facecolor='skyblue', edgecolor='k', label='Anchored Notes', alpha=0.5),
        plt.Line2D([0], [0], color='black', lw=1, label='Mean (All Notes)'),
        plt.Line2D([0], [0], color='blue', lw=1, label='Mean (Anchored Notes)'),
        # plt.Line2D([0], [0], color='red', lw=1, label='Semitone Labels')
    ], title="Legend")

In [ ]:
# DATAFRAME MODIFICATIONS
all_segments['prev_label'] = all_segments['label'].shift(1)
all_segments['next_label'] = all_segments['label'].shift(-1)

anchor_rows = []
for raga in all_segments['raga'].unique():
    anchor_labels = anchor_pairs_by_raga[raga]
    subset = all_segments[(all_segments['raga'] == raga) & (all_segments['label'].isin(anchor_labels))].copy()
    anchor_rows.append(subset)

anchor_prev_df = all_segments[(circular_dist(all_segments['prev_label'], all_segments['label']) == 1)].copy()
anchor_next_df = all_segments[(circular_dist(all_segments['next_label'], all_segments['label']) == 1)].copy()

# Reusable: mean of all notes
mean_all = all_segments.groupby('label')['mean_octinv'].mean().reindex(range(12))

# Compute anchored means
mean_anchor_prev = anchor_prev_df.groupby('label')['mean_octinv'].mean().reindex(range(12))
mean_anchor_next = anchor_next_df.groupby('label')['mean_octinv'].mean().reindex(range(12))

In [ ]:
# TEST

# Create subplots: one for prev anchors, one for next anchors
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
# Top: Anchored notes preceded by other anchor
plot_anchor_effect(ax1, anchor_prev_df, mean_anchor_prev, mean_all, "Effect on Notes Preceded by Anchor")

# Bottom: Anchored notes followed by other anchor
plot_anchor_effect(ax2, anchor_next_df, mean_anchor_next, mean_all, "Effect on Notes Followed by Anchor")

ax2.set_xlabel("Semitone (Label)")
plt.tight_layout()
plt.show()

In [ ]:
# TEST
# Compare effect of preceding versus following anchors

# Initialize lists to collect results
results = []

# Loop through all swara labels (0 through 11)
for label in range(12):
    # Extract values for this label from all datasets
    all_vals = all_segments[all_segments['label'] == label]['mean_octinv'].dropna()
    prev_vals = anchor_prev_df[anchor_prev_df['label'] == label]['mean_octinv'].dropna()
    next_vals = anchor_next_df[anchor_next_df['label'] == label]['mean_octinv'].dropna()

    # Skip label if not enough data
    if len(all_vals) < 2 or len(prev_vals) < 2 or len(next_vals) < 2:
        continue

    # Welch's t-test
    stat_prev, pval_prev = ttest_ind(prev_vals, all_vals, equal_var=False)
    stat_next, pval_next = ttest_ind(next_vals, all_vals, equal_var=False)

    # Cohen's d (Prev vs All)
    pooled_std_prev = np.sqrt((np.std(prev_vals, ddof=1) ** 2 + np.std(all_vals, ddof=1) ** 2) / 2)
    cohens_d_prev = abs(np.mean(prev_vals) - np.mean(all_vals)) / pooled_std_prev

    # Cohen's d (Next vs All)
    pooled_std_next = np.sqrt((np.std(next_vals, ddof=1) ** 2 + np.std(all_vals, ddof=1) ** 2) / 2)
    cohens_d_next = abs(np.mean(next_vals) - np.mean(all_vals)) / pooled_std_next

    # Append results
    results.append({
        "Label": label,
        "Mean (All)": np.mean(all_vals),
        "Mean (Prev Anchor)": np.mean(prev_vals),
        "Mean (Next Anchor)": np.mean(next_vals),
        "p-value (Prev vs All)": pval_prev,
        "Cohen's d (Prev)": cohens_d_prev,
        "p-value (Next vs All)": pval_next,
        "Cohen's d (Next)": cohens_d_next
    })

# Convert to DataFrame
anchor_comparison_df = pd.DataFrame(results)

# Add a summary row with averages across all swaras
anchor_summary_row = anchor_comparison_df.mean(numeric_only=True)
anchor_summary_row["Label"] = "Mean"
comparison_df = pd.concat([anchor_comparison_df, pd.DataFrame([anchor_summary_row])], ignore_index=True)

# Display the result
display(anchor_comparison_df)

In [ ]:
# TEST
# Plot results of statistical comparison between two classes of anchors

anchor_plot_df = anchor_comparison_df[anchor_comparison_df['Label'] != "Mean"]

labels = anchor_plot_df['Label']
cohens_d_prev = anchor_plot_df["Cohen's d (Prev)"]
cohens_d_next = anchor_plot_df["Cohen's d (Next)"]

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(labels, cohens_d_prev, marker='o', label="Prev Anchor", color='blue')
plt.plot(labels, cohens_d_next, marker='s', label="Next Anchor", color='green')
plt.axhline(0.2, color='gray', linestyle='dotted', linewidth=1)
plt.axhline(0.5, color='gray', linestyle='dotted', linewidth=1)
plt.axhline(0.8, color='gray', linestyle='dotted', linewidth=1)
plt.text(11.2, 0.2, 'Small', va='center', fontsize=8, color='gray')
plt.text(11.2, 0.5, 'Medium', va='center', fontsize=8, color='gray')
plt.text(11.2, 0.8, 'Large', va='center', fontsize=8, color='gray')

plt.title("Cohen's d Effect Size of Anchoring (Prev vs Next)")
plt.xlabel("Swara Label")
plt.ylabel("Cohen's d")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 2. Tonic and Label Estimation

### Label Estimation

In [ ]:
# FUNCTIONS

def plot_pitch_hist(pitch_midi, bins=100, ax=None):
    ax.hist(pitch_midi, bins=bins, density=True, alpha=0.7, color='b', edgecolor='black')
    ax.set_xlabel('Semitone')
    ax.set_ylabel('Probability Density')
    
def plot_kde(pitch_range, kde_values, ax=None):
    ax.plot(pitch_range, kde_values, color='r', linewidth=2)
    ax.set_xlabel('Semitone')
    ax.set_ylabel('Density')

def get_pitch_dist(arr, bins=100, bw_method=0.05, display=True, ax=None, vlines = None, ret_kde = False):
    '''
    Inputs:
    1. p0: fundamental pitch contour (ARR)
    2. sr: sampling rate of original audio (INT)
    3. hop_length: hop_length used to compute f0 (INT)
    4. filt_ker: kernel size used for smoothing f0 (INT)
    5. display: (BOOL)

    Outputs:
    1. pitch_range: x-axis, linearly spaced points between min and max frequencies present (ARR)
    2. kde_values: probability density assigned to each x value in the pitch_range (ARR)
    3. if display: histogram of pitches with overlaid kde curve
    '''
    clean = arr[~np.isnan(arr)]
    kde = stats.gaussian_kde(clean, bw_method)
    pitch_range = np.linspace(min(clean), max(clean), 200)
    kde_values = kde(pitch_range)

    if ax is None:
        fig, ax = plt.subplots()
        
    if display:
        plot_pitch_hist(clean, bins, ax=ax)
        plot_kde(pitch_range, kde_values, ax=ax)
        if vlines:
            for label, pos in vlines.items():
                ax.axvline(pos, color='gray', linestyle='--', linewidth=0.8)
                ax.text(pos, ax.get_ylim()[1]*0.9, label, rotation=90,
                    verticalalignment='top', horizontalalignment='right',
                    fontsize=8, color='gray')
        ax.set_title(f'Pitch Distribution (bw={bw_method})')

    if ret_kde:
        return pitch_range, kde_values
    
def compute_average_pitch_per_frame(p0, sr, hop_length = 1024, frame_dur=0.025, overlap=0.5):
    '''
    Inputs:
    1. p0: fundamental pitch contour (ARR)
    2. sr: sampling rate of the original audio (INT)
    3. hop_length: hop length used to compute f0 (INT)
    4. frame_dur: length (in seconds) of each frame (FLOAT)
    5: overlap: proportion of overlap between frames (number 0-1) (FLOAT)

    Outputs:
    1. avg_p0_per_frame: average pitch per frame (ARR)
    2. psr: effective sampling rate ofo the p0 contour = sr / hop_length (FLOAT)
    3. new_hop_length: hop_length used on p0 to generate averages (INT)
    '''

    # Effective sampling rate of the f0 contour being passed in
    psr = sr/hop_length
    print(sr, hop_length, psr)
    print(frame_dur*psr)
    # Define frame length and hop length (50% overlap)
    frame_length = int(frame_dur * psr)  # 25ms window
    print(frame_length)
    new_hop_length = max(int(frame_length * (1 - overlap)), 1) # 50% overlap

    # Compute average f0 per frame
    avg_p0_per_frame = []
    num_frames = int((len(p0) - frame_length) // new_hop_length) + 1

    for i in range(num_frames):
        start = i * new_hop_length
        end = start + frame_length

        # Extract current frame's f0 values (ignore NaNs)
        frame_p0 = p0[start:end]
        frame_p0 = frame_p0[~np.isnan(frame_p0)]  # Remove NaNs

        # Compute mean f0 if valid values exist, otherwise set NaN
        avg_p0_per_frame.append(np.mean(frame_p0) if len(frame_p0) > 0 else np.nan)

    avg_p0_per_frame = np.array(avg_p0_per_frame)
    # Compute times for each averaged frame
    return avg_p0_per_frame, psr, new_hop_length

In [ ]:
# HYPERPARAMETER TUNING

bandwidths = [0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.5, 1]
swara_locations = {"S": 0, "R2": 2, "G3": 4, "M2": 6, "P": 7, "D2": 9, "N3": 11, "S*": 12}  # adjust as needed

fig, axs = plt.subplots(3, 3, figsize=(18, 8), sharey=True)
axs = axs.flatten()

for ax, bw in zip(axs, bandwidths):
    get_pitch_dist(ex_p0, bw_method=bw, vlines = swara_locations, ax=ax)
fig.suptitle("KDE of Pitch Distributions with Varying Bandwidths", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# TEST

ex_range, ex_values = get_pitch_dist(ex_p0_smooth, vlines = swara_locations, ret_kde = True)
ex_p0_octinv = ex_p0_smooth % 12
ex_range_octinv, ex_values_octinv = get_pitch_dist(ex_p0_octinv, vlines = swara_locations, ret_kde = True, bw_method = 0.05)

#### Naive Peak Picking

#### Gaussian Mixture Models (GMMs)

In [ ]:
# FUNCTIONS

def linear_kde(data, bandwidth=0.05, n_points=500, pad=False, pad_width=12):
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    
    if len(data) == 0:
        raise ValueError("No valid data points to compute KDE.")

    # Padding for pseudo-periodicity (shift up/down an octave)
    if pad:
        data = np.concatenate([data - pad_width, data, data + pad_width])

    # Use min/max of data (w/ small padding) as KDE domain
    data_min, data_max = np.min(data), np.max(data)
    padding = 2  # in semitones
    x = np.linspace(data_min - padding, data_max + padding, n_points)

    # Construct KDE
    kde_func = gaussian_kde(data, bw_method=bandwidth)
    kde = kde_func(x)
    kde /= np.trapz(kde, x)  # Normalize area to 1

    return x, kde

def estimate_tonic_from_linear_kde(midi_pitches, kde_x, kde_vals):
    if len(midi_pitches) == 0:
        raise ValueError("No pitch data available.")
    peaks, _ = find_peaks(kde_vals)
    if len(peaks) < 2:
        idx = np.argmax(kde_vals)
        tonic = kde_x[idx]
        confidence = 1.0
    else:
        peak_vals = kde_vals[peaks]
        top2_idx = peaks[np.argsort(peak_vals)[-2:]]
        top2_vals = peak_vals[np.argsort(peak_vals)[-2:]]
        top2_notes = kde_x[top2_idx]
        diff = (top2_notes[1] - top2_notes[0]) % 12
        tonic = top2_notes[0 if top2_vals[0] > top2_vals[1] else 1]
        other = top2_notes[1 if tonic == top2_notes[0] else 0]
        interval = (other - tonic) % 12
        if abs(interval - 7) > abs(interval - 5):
            tonic = other
        confidence = top2_vals[1] / top2_vals[0] if top2_vals[1] > 0 else 10.0
    return tonic % 12, confidence

In [ ]:
# FUNCTIONS

def negative_log_likelihood_gmm(params, data, k):
    mus = params[:k]
    sigmas = np.exp(params[k:2*k])
    weights = np.exp(params[2*k:])
    weights /= np.sum(weights)
    pdf_vals = np.array([w * norm.pdf(data, mu, sigma) for w, mu, sigma in zip(weights, mus, sigmas)])
    total_pdf = np.sum(pdf_vals, axis=0)
    return -np.sum(np.log(total_pdf + 1e-12))

def gmm_constraints(params, k, min_sep=0.8):
    mus = params[:k]
    constraints = []
    for i in range(k):
        for j in range(i+1, k):
            constraints.append(abs(mus[i] - mus[j]) - min_sep)
    return np.array(constraints)

def fit_gmm_model(data, k, constrain=False, min_sep=0.8):
    data = np.asarray(data)
    data = data[~np.isnan(data)]

    low, high = np.percentile(data, [2, 98])  # avoids outliers
    mus0 = np.linspace(low, high, k)

    sigmas0 = np.log(np.ones(k) * 0.5)
    weights0 = np.zeros(k)
    x0 = np.concatenate([mus0, sigmas0, weights0])
    bounds = [(low, high)] * k + [(np.log(0.1), 0)] * k + [(None, None)] * k
    constraints = [{'type': 'ineq', 'fun': lambda p: gmm_constraints(p, k, min_sep)}] if constrain else None
    res = minimize(
        negative_log_likelihood_gmm, x0, args=(data, k),
        method='SLSQP', bounds=bounds, constraints=constraints,
        options={'disp': False, 'maxiter': 1000}
    )
    mus = res.x[:k]
    sigmas = np.exp(res.x[k:2*k])
    weights = np.exp(res.x[2*k:])
    weights /= np.sum(weights)
    sorted_params = list(zip(mus, sigmas, weights))
    sorted_params.sort(key=lambda x: x[0])
    mus = [x[0] for x in sorted_params]
    sigmas = [x[1] for x in sorted_params]
    weights = [x[2] for x in sorted_params]
    return mus, sigmas, weights, res.fun

def select_gmm_bic(data, min_k=2, max_k=10, constrain=False, min_sep=0.8):
    data = data[~np.isnan(data)]
    best_bic = np.inf
    best_model = None
    n = len(data)
    for k in range(min_k, max_k + 1):
        try:
            mus, sigmas, weights, nll = fit_gmm_model(data, k, constrain, min_sep)
            num_params = 3*k - 1
            bic = 2 * nll + num_params * np.log(n)
            if bic < best_bic:
                best_bic = bic
                best_model = (mus, sigmas, weights)
        except Exception:
            continue
    return best_model


#### Von Mises Mixture Models (VMMs)

In [ ]:
# FUNCTIONS

def midi_to_angle(pitches):
    return (2 * np.pi * (pitches % 12)) / 12

def angle_to_midi(angle):
    return (angle * 12) / (2 * np.pi)

def von_mises_pdf(x, mu, kappa):
    return np.exp(kappa * np.cos(x - mu)) / (2 * np.pi * i0(kappa))

In [ ]:
# FUNCTIONS

def circular_vonmises_kde(angles, kappa=12, n_points=360):
    '''
    takes in data as angles and returns paired thetas with densities
    '''
    theta = np.linspace(0, 2 * np.pi, n_points)
    kde = np.zeros_like(theta)
    norm_const = 1 / (2 * np.pi * i0(kappa) * len(angles))
    for angle in angles:
        kde += np.exp(kappa * np.cos(theta - angle))
    kde *= norm_const
    return theta, kde

def estimate_tonic_from_kde(angles, theta, kde):
    if len(angles) == 0:
        raise ValueError("No valid pitch data to estimate tonic.")
    peaks, _ = find_peaks(kde)
    if len(peaks) < 2:
        # If less than two peaks, just take the global max for tonic
        idx = np.argmax(kde)
        estimated_tonic_angle = theta[idx]
        confidence = 1.0  # full confidence, no ambiguity
    else:
        # Pick two largest peaks
        peak_vals = kde[peaks]
        top2_idx = peaks[np.argsort(peak_vals)[-2:]]  # indices of top 2 peaks
        peak_angles = theta[top2_idx]
        peak_vals_sorted = peak_vals[np.argsort(peak_vals)[-2:]]

        # Compute spacing in semitones between peaks (mod 12)
        semitones = angle_to_midi(peak_angles)
        diff = (semitones[1] - semitones[0]) % 12

        # Heuristic to decide tonic and fifth:
        # The fifth is 7 semitones above tonic.
        # We assume the higher peak value corresponds to tonic.
        tonic_idx = 0 if peak_vals_sorted[0] > peak_vals_sorted[1] else 1
        tonic_angle = peak_angles[tonic_idx]
        other_angle = peak_angles[1 - tonic_idx]
        interval = (other_angle - tonic_angle) % (2 * np.pi)
        interval_semitones = angle_to_midi(interval)

        # If the interval is close to 7 semitones, tonic is tonic_angle
        # If interval close to 5 (4 semitones) — could be tonic/fourth, so adjust accordingly
        # For now, if interval_semitones closer to 7 than 5, keep tonic_angle as tonic.
        # Else swap tonic and other
        if abs(interval_semitones - 7) > abs(interval_semitones - 5):
            # Swap tonic and other
            tonic_angle = other_angle

        estimated_tonic_angle = tonic_angle

        # Simple confidence measure: ratio of top peak to second peak
        confidence = peak_vals_sorted[1] / peak_vals_sorted[0] if peak_vals_sorted[1] > 0 else 10.0

    return estimated_tonic_angle, confidence

In [ ]:
# FUNCTIONS

def negative_log_likelihood(params, x, n_components):
    mus = params[:n_components] % (2 * np.pi)
    kappas = np.exp(params[n_components:2*n_components])
    weights = np.exp(params[2*n_components:])
    weights /= np.sum(weights)
    pdf_vals = np.array([w * von_mises_pdf(x, m, k) for w, m, k in zip(weights, mus, kappas)])
    total_pdf = np.sum(pdf_vals, axis=0)
    return -np.sum(np.log(total_pdf + 1e-12))

def separation_constraints(params, n_components, min_sep):
    mus = params[:n_components] % (2 * np.pi)
    constraints = []
    for i in range(n_components):
        for j in range(i+1, n_components):
            d = np.abs((mus[i] - mus[j] + np.pi) % (2*np.pi) - np.pi)
            constraints.append(d - min_sep)
    return np.array(constraints)

def fit_vmm_model(data, n_components, constrain=False, min_separation_cents=80, min_kappa = 1.25):
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    angles = midi_to_angle(data)
    min_sep = (2 * np.pi * (min_separation_cents / 100)) / 12 if constrain else 0
    mus0 = np.linspace(0, 2*np.pi, n_components, endpoint=False)
    kappas0 = np.log(np.ones(n_components) * 2)
    weights0 = np.zeros(n_components)
    x0 = np.concatenate([mus0, kappas0, weights0])
    
    bounds = (
        [(0, 2*np.pi)] * n_components +
        [(np.log(min_kappa), None)] * n_components +
        [(None, None)] * n_components
    )
    cons = ({
        'type': 'ineq',
        'fun': lambda p: separation_constraints(p, n_components, min_sep)
    } if constrain else None)

    res = minimize(
        negative_log_likelihood, x0,
        args=(angles, n_components),
        method='SLSQP', bounds=bounds,
        constraints=cons, options={'disp': False, 'maxiter': 1000}
    )

    mus = res.x[:n_components] % (2 * np.pi)
    kappas = np.exp(res.x[n_components:2*n_components])
    weights = np.exp(res.x[2*n_components:])
    weights /= np.sum(weights)
    sorted_params = list(zip(mus, kappas, weights))
    sorted_params.sort(key=lambda x: x[0])
    mus = [x[0] for x in sorted_params]
    kappas = [x[1] for x in sorted_params]
    weights = [x[2] for x in sorted_params]
    return mus, kappas, weights, res.fun # mu is in radians, not semitones

def select_vmm_bic(data, min_k=2, max_k=10, constrain=False, min_separation_cents=50):
    data = data[~np.isnan(data)]
    angles = midi_to_angle(data)
    n = len(angles)
    best_bic = np.inf
    best_model = None
    for k in range(min_k, max_k+1):
        try:
            mus, kappas, weights, nll = fit_vmm_model(data, k, constrain, min_separation_cents)
            num_params = 3*k - 1
            bic = 2*nll + num_params*np.log(n)
            if bic < best_bic:
                best_bic = bic
                best_model = (mus, kappas, weights)
        except Exception:
            continue
    return best_model # mu is still in radians, not semitones

#### Label Estimation Dataframe Generation

In [ ]:
# GLOBAL VALUES

kde_models = ['lin_kde', 'gmm', 'gmm_padded', 'gmm_constrained', 'gmm_var', 'gmm_var_padded', 'gmm_var_constrained', 'circ_kde', 'vmm', 'vmm_constrained', 'vmm_var', 'vmm_var_constrained']

In [ ]:
# FUNCTIONS

def add_pitch_model_to_df(df, model='vmm'):
    """
    Adds columns to df for the specified pitch model.
    """
    new_col = f"{model}_components"
    pdf_col = f"{model}_pdfs"
    df[new_col] = None
    df[pdf_col] = None

    # bic_col = f"{model}_bic" if model.startswith("vmm") else None

    df = df.copy()  # don't modify original
    for idx, row in df.iterrows():
        n_components = len(row['note_locs'])
        try:
            if model == 'circ_kde':
                tonic_angle, confidence = estimate_tonic_from_linear_kde(row['angles'], row['thetas'], row['circ_kde'])
                df.at[idx, f"{model}_tonic_est"] = tonic_angle
                df.at[idx, f"{model}_conf"] = confidence

            elif model == 'vmm':
                mus, kappas, weights, nll = fit_vmm_model(row['midi_pitches'], n_components)
                df.at[idx, new_col] = list(zip(mus, kappas, weights))
                df.at[idx, pdf_col] = [(weights[k] * vonmises.pdf(row['thetas'], kappas[k], loc=mus[k]), f"μ={angle_to_midi(mus[k]):.2f}, κ={kappas[k]:.1f}") for k in range(len(mus))]
                # df.at[idx, bic_col] = bic
                # converts mu into semitones when plotting but not in tonic_df
            
            elif model == 'vmm_constrained':
                mus, kappas, weights, nll = fit_vmm_model(row['midi_pitches'], n_components, constrain=True)
                df.at[idx, new_col] = list(zip(mus, kappas, weights))
                df.at[idx, pdf_col] = [(weights[k] * vonmises.pdf(row['thetas'], kappas[k], loc=mus[k]), f"μ={angle_to_midi(mus[k]):.2f}, κ={kappas[k]:.1f}") for k in range(len(mus))]
                # df.at[idx, bic_col] = bic

            elif model == 'vmm_var':
                mus, kappas, weights = select_vmm_bic(row['midi_pitches'], constrain=False)
                df.at[idx, new_col] = list(zip(mus, kappas, weights))
                df.at[idx, pdf_col] = [(weights[k] * vonmises.pdf(row['thetas'], kappas[k], loc=mus[k]), f"μ={angle_to_midi(mus[k]):.2f}, κ={kappas[k]:.1f}") for k in range(len(mus))]
                # df.at[idx, bic_col] = bic

            elif model == 'vmm_var_constrained':
                mus, kappas, weights = select_vmm_bic(row['midi_pitches'], constrain=True)
                df.at[idx, new_col] = list(zip(mus, kappas, weights))
                df.at[idx, pdf_col] = [(weights[k] * vonmises.pdf(row['thetas'], kappas[k], loc=mus[k]), f"μ={angle_to_midi(mus[k]):.2f}, κ={kappas[k]:.1f}") for k in range(len(mus))]
                # df.at[idx, bic_col] = bic

            elif model == 'lin_kde':
                tonic, confidence = estimate_tonic_from_kde(row['midi_pitches'], row['xvals'], row['lin_kde'])
                df.at[idx, f"{model}_tonic_est"] = tonic
                df.at[idx, f"{model}_conf"] = confidence
            
            elif model == 'gmm':
                mus, sigmas, weights, nll = fit_gmm_model(row['midi_pitches'], n_components, constrain=False)
                # TODO: sort by mu value? do this everywhere its needed
                df.at[idx, new_col] = list(zip(mus, sigmas, weights))
                df.at[idx, pdf_col] = [(weights[i] * norm.pdf(row['xvals'], loc=mus[i], scale=sigmas[i]), f"μ={mus[i]:.2f}, σ={sigmas[i]:.2f}") for i in range(len(mus))]
                # df.at[idx, bic_col] = bic
            
            elif model == 'gmm_padded':
                mus, sigmas, weights, nll = fit_gmm_model(row['midi_padded'], n_components, constrain=False)
                # TODO: sort by mu value? do this everywhere its needed
                df.at[idx, new_col] = list(zip(mus, sigmas, weights))
                df.at[idx, pdf_col] = [(weights[i] * norm.pdf(row['xvals_padded'], loc=mus[i], scale=sigmas[i]), f"μ={mus[i]:.2f}, σ={sigmas[i]:.2f}") for i in range(len(mus))]
                # df.at[idx, bic_col] = bic
            
            elif model == 'gmm_var_padded':
                mus, sigmas, weights = select_gmm_bic(row['midi_padded'], constrain=False)
                # TODO: sort by mu value? do this everywhere its needed
                df.at[idx, new_col] = list(zip(mus, sigmas, weights))
                df.at[idx, pdf_col] = [(weights[i] * norm.pdf(row['xvals_padded'], loc=mus[i], scale=sigmas[i]), f"μ={mus[i]:.2f}, σ={sigmas[i]:.2f}") for i in range(len(mus))]
                # df.at[idx, bic_col] = bic

            elif model == 'gmm_constrained':
                mus, sigmas, weights, nll = fit_gmm_model(row['midi_pitches'], n_components, constrain=True)
                df.at[idx, new_col] = list(zip(mus, sigmas, weights))
                df.at[idx, pdf_col] = [(weights[i] * norm.pdf(row['xvals'], loc=mus[i], scale=sigmas[i]), f"μ={mus[i]:.2f}, σ={sigmas[i]:.2f}") for i in range(len(mus))]
                # df.at[idx, bic_col] = bic
            
            elif model == 'gmm_var':
                mus, sigmas, weights = select_gmm_bic(row['midi_pitches'], constrain=False)
                # TODO: sort by mu value? do this everywhere its needed
                df.at[idx, new_col] = list(zip(mus, sigmas, weights))
                df.at[idx, pdf_col] = [(weights[i] * norm.pdf(row['xvals'], loc=mus[i], scale=sigmas[i]), f"μ={mus[i]:.2f}, σ={sigmas[i]:.2f}") for i in range(len(mus))]
                # df.at[idx, bic_col] = bic
            
            elif model == 'gmm_var_constrained':
                mus, sigmas, weights = select_gmm_bic(row['midi_pitches'], constrain=True)
                df.at[idx, new_col] = list(zip(mus, sigmas, weights))
                df.at[idx, pdf_col] = [(weights[i] * norm.pdf(row['xvals'], loc=mus[i], scale=sigmas[i]), f"μ={mus[i]:.2f}, σ={sigmas[i]:.2f}") for i in range(len(mus))]
                # df.at[idx, bic_col] = bic

        except Exception as e:
            print(f"Skipping {row['artist']} / {row['raga']} due to error: {e}")
            df.at[idx, new_col] = None

    return df

In [ ]:
# FUNCTIONS
def find_top_k_peaks_from_local_maxima(xvals, kde_vals, k):
    """
    Find the top-k local maxima from the KDE curve.

    Parameters:
    - xvals: array of x-values (e.g., MIDI pitch)
    - kde_vals: array of KDE values aligned with xvals
    - k: number of peaks to return

    Returns:
    - List of up to k x-values corresponding to top local maxima
    """
    kde_vals = np.asarray(kde_vals)
    xvals = np.asarray(xvals)

    # Step 1: Find local maxima
    peak_indices, _ = find_peaks(kde_vals)

    # Step 2: Rank peaks by height
    peak_heights = kde_vals[peak_indices]
    top_k_indices = np.argsort(-peak_heights)[:k]  # descending order

    # Step 3: Select and return corresponding xvals
    top_k_peaks = xvals[peak_indices[top_k_indices]]
    return sorted(top_k_peaks)

In [ ]:
# TEST

kde_df = pd.DataFrame(columns=['artist', 'raga', 'actual_tonic_hz', 'actual_tonic_midi',  'midi_pitches', 'midi_padded', 'xvals', 'lin_kde', 'xvals_padded', 'lin_kde_padded', 'note_locs', 'actual_tonic_angle', 'angles', 'thetas', 'circ_kde', 'note_angles'])

for key, value in dataset.items():
    # extract metadata from file name
    item, x, sr = value
    artist = item.split("-")[3]
    raga = item.split("-")[5]

    # extract audio content and pitch contour
    xtonic_hz = tonics[artist]
    xtonic_midi = lb.hz_to_midi(xtonic_hz)
    xtonic_angle = midi_to_angle(xtonic_midi)
    xf0, xvflags, xvprobs = get_f0(x, sr, xtonic_hz, display = False) # CHANGED: used to include duration = 80?
    xp0_noton = get_p0(xf0, sr, display = False)
    xp0_smooth = smooth_voiced_sections(xp0_noton, sr, xvflags, filter_size = 7, display = False) # CHANGED: made all instances of noton into smooth
    angles = midi_to_angle(xp0_smooth[~np.isnan(xp0_smooth)])

    xvals, lin_kde = linear_kde(xp0_smooth)
    xp0_padded = np.concatenate([xp0_smooth - 12, xp0_smooth, xp0_smooth + 12])
    xvals_padded, lin_kde_padded = linear_kde(xp0_smooth, pad = True)
    theta, circ_kde = circular_vonmises_kde(angles)

    ragadict = ragadicts[raga]
    note_locs = np.sort([(xtonic_midi + semitones) for semitones in ragadict.values()])
    note_angles = midi_to_angle(note_locs)

    kde_df.loc[len(kde_df)] = [artist, raga, xtonic_hz, xtonic_midi, xp0_smooth, xp0_padded, xvals, lin_kde, xvals_padded, lin_kde_padded, note_locs, xtonic_angle, angles, theta, circ_kde, note_angles]

for model in kde_models:
    kde_df = add_pitch_model_to_df(kde_df, model)

In [ ]:
kde_df["naive_peaks"] = kde_df.apply(
    lambda row: find_top_k_peaks_from_local_maxima(row["xvals"], row["lin_kde"], len(row["note_locs"])),
    axis=1
)

#### Label Estimation Plotting

In [ ]:
# FUNCTIONS

def plot_pitch_model_components(
    ax,
    x_grid,
    pdfs=None,
    kde=None,
    hist_data=None,
    title="",
    tonic_line=None,
    actual_tonic_line=None,
    show_labels=True,
    vlines=[],
    polar=False,
):
    """
    General function to plot KDEs, GMM/VMM component PDFs, and histograms.

    Parameters:
    - ax: matplotlib axis (polar or cartesian)
    - x_grid: angle grid (for circular) or linear pitch grid
    - pdfs: list of (pdf array, label) tuples
    - kde: array of KDE values (same length as x_grid)
    - hist_data: raw pitch/angle data for histogram
    - tonic_line: value to mark as estimated tonic (angle or pitch)
    - actual_tonic_line: ground truth tonic
    - show_labels: include legend
    - vlines: extra vertical lines to draw (e.g., note locations)
    - polar: whether this is a polar plot
    """
    if ax is None:
        fig, ax = plt.subplots(subplot_kw={'projection': 'polar'} if polar else {}, figsize=(7, 5))

    if kde is not None:
        ax.plot(x_grid, kde, label='KDE', color='teal', linewidth=2)

    if hist_data is not None:
        if polar:
            hist_bins = np.linspace(0, 2 * np.pi, 25)
        else:
            hist_bins = np.linspace(min(x_grid), max(x_grid), 60)

        counts, _ = np.histogram(hist_data, bins=hist_bins)
        centers = hist_bins[:-1] + np.diff(hist_bins) / 2

        # Normalize heights to match the scale of kde/pdf
        if counts.max() > 0:
            if kde is not None:
                ref_height = kde.max()
            elif pdfs and len(pdfs) > 0:
                ref_height = max([p.max() for p, _ in pdfs])
            else:
                ref_height = 1.0
            norm_heights = counts / counts.max() * ref_height
        else:
            norm_heights = counts

        bar_kwargs = dict(width=np.diff(hist_bins)[0], alpha=0.3, color='gray', label='Histogram')
        if polar:
            ax.bar(centers, norm_heights, **bar_kwargs)
        else:
            ax.bar(centers, norm_heights, align='center', **bar_kwargs)

    if pdfs:
        for pdf, lbl in pdfs:
            ax.plot(x_grid, pdf, linestyle='--', label=lbl, alpha=0.7)
        total_pdf = sum(p for p, _ in pdfs)
        ax.plot(x_grid, total_pdf, color='orange', linewidth=2, label='Total PDF')

    if tonic_line is not None:
        ax.axvline(tonic_line, color='red', linestyle='--', label='Estimated Tonic')

    if actual_tonic_line is not None:
        ax.axvline(actual_tonic_line, color='green', linestyle='--', label='Actual Tonic')

    for v in vlines:
        ax.axvline(v, color='black', linestyle='--', linewidth=0.8)

    if polar:
        ax.set_xticks(np.linspace(0, 2 * np.pi, 12, endpoint=False))
        ax.set_xticklabels([str(i) for i in range(12)])

    ax.set_title(title)
    if not polar:
        ax.set_xlabel("MIDI Pitch")
        ax.set_ylabel("Density")

    if show_labels:
        ax.legend(fontsize=7)

def plot_pitch_model_grid(df, model='circ_kde', polar = False):
    ragas = sorted(df['raga'].unique())
    artists = sorted(df['artist'].unique())
    if polar:
        fig, axes = plt.subplots(len(artists), len(ragas), subplot_kw={'projection': 'polar'}, figsize=(4*len(ragas), 4*len(artists)), squeeze=False)
    else:
        fig, axes = plt.subplots(len(artists), len(ragas), figsize=(4 * len(ragas), 4 * len(artists)),squeeze=False)

    for idx, row in df.iterrows():
        i, j = artists.index(row['artist']), ragas.index(row['raga'])
        ax = axes[i, j]
        try:
            if model == 'circ_kde':
                plot_pitch_model_components(ax, row['thetas'], [], kde=row['circ_kde'], hist_data=row['angles'], title=f"{row['artist']} / {row['raga']}\nConf: {row['circ_kde_conf']:.2f}", tonic_line = row['circ_kde_tonic_est'], actual_tonic_line = row['actual_tonic_angle'], polar = polar)

            elif model == 'vmm':
                plot_pitch_model_components(ax, row['thetas'], row['vmm_pdfs'], kde = row['circ_kde'], hist_data=None, title=f"{row['artist']} / {row['raga']}", vlines = row['note_angles'], polar = polar)

            elif model == 'vmm_constrained':
                plot_pitch_model_components(ax, row['thetas'], row['vmm_constrained_pdfs'], kde = row['circ_kde'], hist_data=row['angles'], title=f"{row['artist']} / {row['raga']}", vlines = row['note_angles'], polar = polar)

            elif model == 'vmm_var':
                plot_pitch_model_components(ax, row['thetas'], row['vmm_var_pdfs'], kde = row['circ_kde'], hist_data=None, title=f"{row['artist']} / {row['raga']}", vlines = row['note_angles'], polar = polar)

            elif model == 'vmm_var_constrained':
                plot_pitch_model_components(ax, row['thetas'], row['vmm_var_constrained_pdfs'], kde = row['circ_kde'], hist_data=row['angles'], title=f"{row['artist']} / {row['raga']}", vlines = row['note_angles'], polar = polar)

            elif model == 'lin_kde':
                plot_pitch_model_components(ax, row['xvals'], [], kde=row['lin_kde'], hist_data=row['midi_pitches'], title=f"{row['artist']} / {row['raga']}\nConf: {row['lin_kde_conf']:.2f}", tonic_line = row['lin_kde_tonic_est'], actual_tonic_line = row['actual_tonic_midi'], polar = polar)
            
            elif model == 'gmm':
                plot_pitch_model_components(ax, row['xvals'], row['gmm_pdfs'], kde = row['lin_kde'], hist_data=None, title=f"{row['artist']} / {row['raga']}", vlines = row['note_locs'], polar = polar)

            elif model == 'gmm_constrained':
                plot_pitch_model_components(ax, row['xvals'], row['gmm_constrained_pdfs'], kde = row['lin_kde'], hist_data=row['midi_pitches'], title=f"{row['artist']} / {row['raga']}", vlines = row['note_locs'], polar = polar)

            elif model == 'gmm_var':
                plot_pitch_model_components(ax, row['xvals'], row['gmm_var_pdfs'], kde = row['lin_kde'], hist_data=None, title=f"{row['artist']} / {row['raga']}", vlines = row['note_locs'], polar = polar)

            elif model == 'gmm_padded':
                plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_padded_pdfs'], kde = None, hist_data=None, title=f"{row['artist']} / {row['raga']}", vlines = row['note_locs'], polar = polar)

            elif model == 'gmm_var_padded':
                plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_var_padded_pdfs'], kde = None, hist_data=None, title=f"{row['artist']} / {row['raga']}", vlines = row['note_locs'], polar = polar)

            elif model == 'gmm_var_constrained':
                plot_pitch_model_components(ax, row['xvals'], row['gmm_var_constrained_pdfs'], kde = row['lin_kde'], hist_data=row['midi_pitches'], title=f"{row['artist']} / {row['raga']}", vlines = row['note_locs'], polar = polar)

        except Exception as e:
            ax.set_visible(False)
            print(f"Error processing {row['artist']} / {row['raga']}: {e}")

    plt.tight_layout()
    plt.show()

def plot_model_comparisons_grid(tonic_df, row_index):
    """
    Creates a grid of plots comparing all pitch modeling methods for a given sample.
    """
    row = tonic_df.iloc[row_index]

    fig, axes = plt.subplots(2, 5, figsize=(25, 10))
    axes = axes.flatten()

    plot_titles_and_calls = [
        ("GMM with Known n_c", lambda ax: plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_pdfs'],
                                                                      kde=row['lin_kde'], title="GMM with Known $n_c$",
                                                                      vlines=row['note_locs'], polar=False)),
        ("Padded GMM with Known n_c", lambda ax: plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_padded_pdfs'],
                                                                             kde=row['lin_kde_padded'], title="GMM-P with Known $n_c$",
                                                                             vlines=row['note_locs'], polar=False)),
        ("Constrained GMM with Known n_c", lambda ax: plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_constrained_pdfs'],
                                                                                  kde=row['lin_kde'], title="GMM-C with Known $n_c$",
                                                                                  vlines=row['note_locs'], polar=False)),
        ("VMM with Known n_c", lambda ax: plot_pitch_model_components(ax, row['thetas'], row['vmm_pdfs'],
                                                                      kde=row['circ_kde'], title="VMM with Known $n_c$",
                                                                      vlines=row['note_angles'], polar=True)),
        ("Constrained VMM with Known n_c", lambda ax: plot_pitch_model_components(ax, row['thetas'], row['vmm_constrained_pdfs'],
                                                                                  kde=row['circ_kde'], title="VMM-C with Known $n_c$",
                                                                                  vlines=row['note_angles'], polar=True)),
        ("GMM with Variable n_c", lambda ax: plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_var_pdfs'],
                                                                         kde=row['lin_kde'], title="GMM with Variable $n_c$",
                                                                         vlines=row['note_locs'], polar=False)),
        ("Padded GMM with Variable n_c", lambda ax: plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_var_padded_pdfs'],
                                                                                kde=row['lin_kde_padded'], title="GMM-P with Variable $n_c$",
                                                                                vlines=row['note_locs'], polar=False)),
        ("Constrained GMM with Variable n_c", lambda ax: plot_pitch_model_components(ax, row['xvals_padded'], row['gmm_var_constrained_pdfs'],
                                                                                     kde=row['lin_kde'], title="GMM-C with Variable $n_c$",
                                                                                     vlines=row['note_locs'], polar=False)),
        ("VMM with Variable n_c", lambda ax: plot_pitch_model_components(ax, row['thetas'], row['vmm_var_pdfs'],
                                                                         kde=row['circ_kde'], title="VMM with Variable $n_c$",
                                                                         vlines=row['note_angles'], polar=True)),
        ("Constrained VMM with Variable n_c", lambda ax: plot_pitch_model_components(ax, row['thetas'], row['vmm_var_constrained_pdfs'],
                                                                                     kde=row['circ_kde'], title="VMM-C with Variable $n_c$",
                                                                                     vlines=row['note_angles'], polar=True)),
    ]

    for ax, (_, plot_call) in zip(axes, plot_titles_and_calls):
        plot_call(ax)

    # Hide any unused axes
    for ax in axes[len(plot_titles_and_calls):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# TEST

for i in range(5):
    plot_model_comparisons_grid(kde_df, row_index = i)

In [ ]:
# TEST

for model in kde_models:
    if 'circ' in model or 'vmm' in model:
        polar = True
    else:
        polar = False
    plot_pitch_model_grid(kde_df, model, polar = polar)

#### Ground Truth Generation

In [ ]:
# FUNCTIONS

# all_segments["label"] = all_segments["label"].astype(int)

def compute_mu_sep(group):
    # Sort swaras in cyclic order
    group = group.sort_values("swara_label").reset_index(drop=True)
    
    # Cyclic shift of mu values within the group
    mu_vals = group["mu"].values
    mu_next = np.roll(mu_vals, -1)  # cyclically shift left
    mu_sep = (mu_next - mu_vals) % 12    # difference to next

    label_vals = group["swara_label"].values
    label_next = np.roll(label_vals, -1)
    label_sep = (label_next - label_vals) % 12
    
    group["mu_sep"] = mu_sep
    group["label_sep"] = label_sep
    return group

def all_component_stats(df, stat = 'p0', remove_outliers = False, plot = True):
    # Storage for both Gaussian and von Mises results
    gaussian_records = []
    vonmises_records = []

    # Generate one combined plot per raga-artist pair for both models
    for (raga, artist), group in df.groupby(["raga", "artist"]):
        if plot:
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.set_title(f"Gaussian Fits for Raga: {raga}, Artist: {artist}")
            
            fig_vm, ax_vm = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(8, 6))
            ax_vm.set_title(f"Von Mises Fits (mod 12) for Raga: {raga}, Artist: {artist}")

        for label in range(12):
            if stat == 'p0':
                swara_segments = group[group["label"] == label]["p0"].values
                if len(swara_segments) == 0:
                    continue
                pitches = np.concatenate(swara_segments)
                pitches = pitches[~np.isnan(pitches)]
            else:
                pitches = group[group["label"] == label][stat].values
                if len(pitches) == 0:
                    continue
                pitches = pitches[~np.isnan(pitches)]
            
            if remove_outliers:
                lower, upper = np.percentile(pitches, [2.5, 97.5])
                pitches = pitches[(pitches >= lower) & (pitches <= upper)]

            # Gaussian fit
            mu = np.mean(pitches)
            sigma = np.std(pitches)
            x_vals = np.linspace(pitches.min(), pitches.max(), 200)
            if plot:
                ax.plot(x_vals, norm.pdf(x_vals, mu, sigma), label=f"Swara {label} ($\mu$={mu:.2f}, $\sigma$={sigma:.2f})")
                gaussian_records.append({
                    "raga": raga,
                    "artist": artist,
                    "swara_label": label,
                    "mu": mu,
                    "sigma": sigma
                })

            # Von Mises fit
            angles = midi_to_angle(pitches)
            mu_vm = np.angle(np.sum(np.exp(1j * angles))) % (2 * np.pi)
            R = np.abs(np.sum(np.exp(1j * angles))) / len(angles)
            kappa = max(1e-3, (R * (2 - R**2)) / (1 - R**2))  # Approximation for kappa
            theta = np.linspace(0, 2 * np.pi, 500)

            if plot:
                ax_vm.plot(theta, vonmises.pdf(theta, kappa, loc=mu_vm), label=f"Swara {label} ($\mu$={angle_to_midi(mu_vm):.2f}, $\sigma$={kappa:.2f})")

            vonmises_records.append({
                "raga": raga,
                "artist": artist,
                "swara_label": label,
                "mu": angle_to_midi(mu_vm) % 12, # THIS IS IN SEMITONES, NOT RADIANS
                "kappa": kappa
            })

        if plot:
            ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.0))
            ax_vm.legend(loc='upper right', bbox_to_anchor=(1.35, 1.0))
            plt.tight_layout()

    # Create summary dataframes
    gaussian_df = pd.DataFrame(gaussian_records)
    vonmises_df = pd.DataFrame(vonmises_records)
    vonmises_df = vonmises_df.groupby(["raga", "artist"], group_keys=False).apply(compute_mu_sep)

    return gaussian_df, vonmises_df

def compute_weights(merged, vonmises_df):
    weights = []

    for _, row in vonmises_df.iterrows():
        raga = row["raga"]
        artist = row["artist"]
        swara_label = row["swara_label"]

        # All segments for this raga-artist-swara
        swara_segments = merged[
            (merged["raga"] == raga) &
            (merged["artist"] == artist) &
            (merged["label"] == swara_label)
        ]["p0"].values

        # Total pitch count for this swara
        swara_pitches = np.concatenate([p[~np.isnan(p)] for p in swara_segments if p is not None])
        swara_count = len(swara_pitches)

        # All segments for this raga-artist
        all_segments = merged[
            (merged["raga"] == raga) &
            (merged["artist"] == artist)
        ]["p0"].values

        # Total pitch count for all swaras
        all_pitches = np.concatenate([p[~np.isnan(p)] for p in all_segments if p is not None])
        total_count = len(all_pitches)

        weight = swara_count / total_count if total_count > 0 else 0
        weights.append(weight)

    vonmises_df["weight"] = weights
    return vonmises_df

In [ ]:
# TEST
# Generate ground truth dataframes
gaussian_df, vonmises_df = all_component_stats(all_segments, 'p0', False, False)
vonmises_df = compute_weights(all_segments, vonmises_df)

#### Musically-Informed Parameter Constraints

In [ ]:
def plot_col_dist(vmm_df, colname, percentiles=[2.5, 5, 50], ax=None, title=None, color='orange'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    ax.hist(vmm_df[colname], bins=50, color=color, edgecolor='black', alpha=0.7)

    colors = ['red', 'green', 'blue']
    labels = [f"{p}th percentile" for p in percentiles]

    ymin, ymax = ax.get_ylim()
    y_offset = -0.05 * ymax  # small offset below x-axis

    for p, c, label in zip(percentiles, colors, labels):
        val = np.percentile(vmm_df[colname].dropna(), p)
        ax.axvline(val, color=c, linewidth=1.5, label=label)
        ax.text(val, y_offset, f'{val:.2f}', color=c,
                ha='center', va='top', fontsize=9, rotation=90)

    ax.set_xlabel(colname)
    ax.set_ylabel("Frequency")
    if title:
        ax.set_title(title)
    ax.legend()
    ax.grid(True)

def plot_mu_sep_comparison(vonmises_df):
    close_df = vonmises_df[vonmises_df["label_sep"] == 1]

    fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    plot_col_dist(
        vonmises_df,
        colname='mu_sep',
        ax=axs[0],
        title=r"$\mu$ Separation (all adjacent swaras)",
        color='orange'
    )

    plot_col_dist(
        close_df,
        colname='mu_sep',
        ax=axs[1],
        title=r"$\mu$ Separation (adjacent swaras 1 semitone apart)",
        color='seagreen'
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# TEST

plot_mu_sep_comparison(vonmises_df)

plot_col_dist(vonmises_df, 'kappa')

plot_col_dist(vonmises_df, 'weight')

#### Evaluation

In [ ]:
# FUNCTIONS

def circular_mse(true_mus, pred_mus):
    """
    Compute mean squared circular error from each true_mu to nearest pred_mu.
    """
    errors = []
    for mu in true_mus:
        dists = np.abs((mu - pred_mus)) % 12
        dists = np.minimum(dists, 12 - dists)
        errors.append(np.min(dists) ** 2)
    return np.mean(errors)


def circular_assignment_mse(true_mus, pred_mus):
    """
    Compute minimum circular MSE with unique assignments using Hungarian algorithm.
    """
    cost_matrix = np.zeros((len(true_mus), len(pred_mus)))
    for i, t in enumerate(true_mus):
        for j, p in enumerate(pred_mus):
            d = np.abs(t - p) % 12
            d = min(d, 12 - d)
            cost_matrix[i, j] = d**2

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    return np.mean(cost_matrix[row_ind, col_ind])


def shape_mse(true_comps, pred_comps, model="gmm"):
    """
    Compute MSE between true and predicted components based on shape (mu, sigma/kappa, weight).
    """
    cost_matrix = np.zeros((len(true_comps), len(pred_comps)))

    for i, (t_mu, t_spread, t_w) in enumerate(true_comps):
        for j, (p_mu, p_spread, p_w) in enumerate(pred_comps):
            d_mu = np.abs(t_mu - p_mu) % 12
            d_mu = min(d_mu, 12 - d_mu)
            mse = d_mu**2 + (t_spread - p_spread)**2 + (t_w - p_w)**2
            cost_matrix[i, j] = mse

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    return np.mean(cost_matrix[row_ind, col_ind])


In [ ]:
# FUNCTIONS

def evaluate_all_models(kde_df, vmm_df):
    models = {
        "naive": "naive",
        "gmm": "gmm_components",
        "gmm_constrained": "gmm_constrained_components",
        "gmm_padded": "gmm_padded_components",
        "vmm": "vmm_components",
        "vmm_constrained": "vmm_constrained_components"
    }

    mse_nearest = {m: [] for m in models}
    mse_assign = {m: [] for m in models}
    mse_shape = {m: [] for m in models if m != "naive"}  # Naive has no shape

    for idx, row in kde_df.iterrows():
        artist, raga = row["artist"], row["raga"]
        ground = vmm_df[(vmm_df["artist"] == artist) & (vmm_df["raga"] == raga)]
        true_mus = ground["mu"].values
        true_weights = ground["weight"].values
        true_kappas = ground["kappa"].values

        for model_key, col_name in models.items():
            if model_key == "naive":
                peaks = np.array(sorted(row['naive_peaks']))  # assumed to be added
                pred_mus = peaks
                mse_nearest[model_key].append(circular_mse(true_mus, pred_mus))
                mse_assign[model_key].append(circular_assignment_mse(true_mus, pred_mus))
                continue

            comps = row[col_name]
            if "gmm" in model_key:
                pred_comps = np.array([(mu, sigma, w) for mu, sigma, w in comps])
                pred_mus = pred_comps[:, 0]
                pred_spread = pred_comps[:, 1]  # sigma
            else:
                pred_comps = np.array([(angle_to_midi(mu), kappa, w) for mu, kappa, w in comps])
                pred_mus = pred_comps[:, 0]
                pred_spread = pred_comps[:, 1]  # kappa

            mse_nearest[model_key].append(circular_mse(true_mus, pred_mus))
            mse_assign[model_key].append(circular_assignment_mse(true_mus, pred_mus))
            true_comps = list(zip(true_mus, true_kappas if "vmm" in model_key else true_kappas / np.sqrt(2), true_weights))
            pred_comps = list(zip(pred_mus, pred_spread, pred_comps[:, 2]))
            mse_shape[model_key].append(shape_mse(true_comps, pred_comps, model=model_key))

    # Aggregate results
    data = []
    for m in models:
        row = {
            "Model": m,
            "MSE (Nearest)": np.mean(mse_nearest[m]),
            "MSE (Assignment)": np.mean(mse_assign[m])
        }
        # if m in mse_shape:
        #     row["MSE (Shape)"] = np.mean(mse_shape[m])
        # else:
        #     row["MSE (Shape)"] = np.nan
        data.append(row)

    return pd.DataFrame(data)


In [ ]:
# FUNCTIONS

def gen_latex_table(df, caption, label, fname):
    latex_table = df.to_latex(float_format="%.3f", index=True,
        caption=caption,
        label=label)
    with open(fname, "w") as f:
        f.write(latex_table)

def evaluate_models_by_raga_all_metrics(tonic_df, vmm_snapped_df):
    from collections import defaultdict

    models = {
        "naive": "naive",
        "gmm": "gmm_components",
        "gmm_constrained": "gmm_constrained_components",
        "gmm_padded": "gmm_padded_components",
        "vmm": "vmm_components",
        "vmm_constrained": "vmm_constrained_components"
    }

    ragas = sorted(tonic_df['raga'].unique())
    mse_assign_by_raga = {model: defaultdict(list) for model in models}
    mse_nearest_by_raga = {model: defaultdict(list) for model in models}

    for idx, row in tonic_df.iterrows():
        raga = row["raga"]
        artist = row["artist"]
        ground = vmm_snapped_df[(vmm_snapped_df["artist"] == artist) & (vmm_snapped_df["raga"] == raga)]
        true_mus = ground["mu"].values

        for model_key, col_name in models.items():
            if model_key == "naive":
                pred_mus = np.array(sorted(row['naive_peaks']))
            else:
                comps = row[col_name]
                if "gmm" in model_key:
                    pred_mus = np.array([mu for mu, sigma, w in comps])
                else:
                    pred_mus = np.array([angle_to_midi(mu) for mu, kappa, w in comps])

            mse_assign = circular_assignment_mse(true_mus, pred_mus)
            mse_nearest = circular_mse(true_mus, pred_mus)

            mse_assign_by_raga[model_key][raga].append(mse_assign)
            mse_nearest_by_raga[model_key][raga].append(mse_nearest)

    # Construct final DataFrames
    assign_table = pd.DataFrame(index=models.keys(), columns=ragas)
    nearest_table = pd.DataFrame(index=models.keys(), columns=ragas)

    for model in models:
        for raga in ragas:
            a_vals = mse_assign_by_raga[model][raga]
            n_vals = mse_nearest_by_raga[model][raga]
            assign_table.loc[model, raga] = np.mean(a_vals) if a_vals else np.nan
            nearest_table.loc[model, raga] = np.mean(n_vals) if n_vals else np.nan

    # Ensure tables are float
    assign_table = assign_table.astype(float)
    nearest_table = nearest_table.astype(float)

    # First, compute row-wise mean (across ragas for each method)
    assign_table["Mean"] = assign_table.mean(axis=1)
    nearest_table["Mean"] = nearest_table.mean(axis=1)

    # Then, compute column-wise mean (across methods for each raga)
    assign_table.loc["Mean"] = assign_table.mean(axis=0)
    nearest_table.loc["Mean"] = nearest_table.mean(axis=0)


    return assign_table, nearest_table


In [ ]:
# TEST

component_eval_df = evaluate_all_models(kde_df, vonmises_df)

sns.heatmap(component_eval_df.set_index("Model"), annot=True, fmt=".3f", cmap="Blues")
plt.title("Model Evaluation Metrics")
plt.show()

assign_table, nearest_table = evaluate_models_by_raga_all_metrics(kde_df, vonmises_df)

gen_latex_table(assign_table, "MSE-B", "tab:mseB", "mseB.tex")
gen_latex_table(nearest_table, "MSE-A", "tab:mseA", "mseA.tex")

In [ ]:
# HYPERPARAMETER TUNING

min_separation_grid = np.linspace(30, 100, 8)  # 30, 53.3, 76.6, 100
min_kappa_grid = np.linspace(0.75, 2.5, 8)  # 0.5 to 5.0

mse_results_ground = pd.DataFrame(index=min_separation_grid, columns=min_kappa_grid)

for min_sep in min_separation_grid:
    for min_kap in min_kappa_grid:
        mse_total = 0
        for idx, row in kde_df.iterrows():
            artist, raga = row["artist"], row["raga"]
            ground = vonmises_df[(vonmises_df["artist"] == artist) & (vonmises_df["raga"] == raga)]
            true_angles = ground["mu"].values * 2 * np.pi / 12
            n_components = len(row['note_locs'])
            mus, kappas, weights, nll = fit_vmm_model(row['midi_pitches'], n_components, constrain=True,
                                                      min_separation_cents=min_sep, min_kappa=min_kap)
            comp_angles = mus
            cost = np.zeros((len(true_angles), len(comp_angles)))
            for i, t in enumerate(true_angles):
                for j, m in enumerate(comp_angles):
                    diff = np.abs((t - m) % (2 * np.pi))
                    cost[i, j] = min(diff, 2 * np.pi - diff)**2
            row_ind, col_ind = linear_sum_assignment(cost)
            mse = np.mean([cost[i, j] for i, j in zip(row_ind, col_ind)])
            mse_total += mse
        mse_avg = mse_total / len(kde_df)
        mse_results_ground.at[min_sep, min_kap] = mse_avg

# Convert to float for plotting
mse_results_ground = mse_results_ground.astype(float)

# Plot heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(mse_results_ground, annot=True, fmt=".3f", cmap="viridis")
plt.xlabel("Minimum Kappa")
plt.ylabel("Minimum Separation (cents)")
plt.title("MSE-B for Constrained VMM Grid Search")
plt.tight_layout()
plt.show()


In [ ]:
# EVAL

var_models = {
    "GMM": "gmm_var_components",
    "GMM-C": "gmm_var_constrained_components",
    "VMM": "vmm_var_components",
    "VMM-C": "vmm_var_constrained_components"
}

# Skip execution if real data is not present
if not kde_df.empty:
    ragas = sorted(kde_df["raga"].unique())
    mae_table = pd.DataFrame(index=var_models.keys(), columns=ragas)

    for model_key, col_name in var_models.items():
        for raga in ragas:
            errors = []
            sub_df = kde_df[kde_df["raga"] == raga]
            for _, row in sub_df.iterrows():
                true_n = len(row["note_locs"])
                pred_comps = row[col_name]
                if pred_comps is not None:
                    pred_n = len(pred_comps)
                    errors.append(np.abs(pred_n - true_n))
            mae = np.mean(errors) if errors else np.nan
            mae_table.loc[model_key, raga] = mae

    mae_table = mae_table.astype(float)

    plt.figure(figsize=(12, 6))
    sns.heatmap(mae_table, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("MAE of $n_c$ by Raga and Model")
    plt.xlabel("Raga")
    plt.ylabel("Model")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
kde_df_sorted = kde_df.sort_values(by="raga").reset_index(drop=True)

# Create a dataframe to hold signed errors for each sample
signed_errors = pd.DataFrame(index=kde_df_sorted.index, columns=var_models.keys())

for model_key, col_name in var_models.items():
    for idx, row in kde_df_sorted.iterrows():
        true_n = len(row["note_locs"])
        pred_comps = row[col_name]
        if pred_comps is not None:
            pred_n = len(pred_comps)
            signed_errors.loc[idx, model_key] = pred_n - true_n
        else:
            signed_errors.loc[idx, model_key] = np.nan

# Convert to float for plotting
signed_errors = signed_errors.astype(float)

# Compute tick positions for each raga
raga_labels = kde_df_sorted["raga"].values
tick_positions = []
tick_labels = []
prev_raga = None
for i, raga in enumerate(raga_labels):
    if raga != prev_raga:
        tick_positions.append(i)
        tick_labels.append(raga)
        prev_raga = raga

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    signed_errors.T,
    cmap="coolwarm",
    center=0,
    annot=False,
    cbar_kws={"label": "Signed Error in $n_c$"},
    xticklabels=False
)
ax.set_title("Signed Error of Estimated $n_c$")
ax.set_xlabel("Raga")
ax.set_ylabel("Model")

# Add raga labels at tick positions
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha="right")

plt.tight_layout()
plt.show()


In [ ]:
se_df = signed_errors.T
col_list = list(se_df)
print(col_list)
se_df['sum'] = se_df[col_list].sum(axis=1)/28
display(se_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# ----------------- helpers -----------------
def hz_to_midi(f):
    f = np.asarray(f, dtype=float)
    return 69.0 + 12.0 * np.log2(f / 440.0)

def circ_dist12(x, y):
    """Circular distance on a 12-semitone ring."""
    d = np.abs(np.asarray(x, dtype=float) - np.asarray(y, dtype=float)) % 12.0
    return np.minimum(d, 12.0 - d)

def angles_to_semitones_rel_tonic(angles, tonic_hz):
    """
    Convert absolute angles (radians, 0..2π) to semitone offsets RELATIVE to the tonic.
    Assumes angles were created from pitch-class (mod-12) via midi_to_angle.
    """
    angles = np.asarray(angles, dtype=float)
    tonic_pc = np.mod(hz_to_midi(tonic_hz), 12.0)                 # tonic pitch class in semitones
    tonic_angle = 2.0 * np.pi * (tonic_pc / 12.0)                 # tonic angle on the circle
    rel_angle = np.mod(angles - tonic_angle, 2.0*np.pi)           # rotate so tonic maps to 0 angle
    semis = (12.0 / (2.0*np.pi)) * rel_angle                      # map to [0,12)
    return semis

def midi_to_semitones_rel_tonic(midi_vals, tonic_hz):
    midi_vals = np.asarray(midi_vals, dtype=float)
    return np.mod(midi_vals - hz_to_midi(tonic_hz), 12.0)

def hz_to_semitones_rel_tonic(freq_hz, tonic_hz):
    return np.mod(hz_to_midi(freq_hz) - hz_to_midi(tonic_hz), 12.0)

def comps_to_semis_rel_tonic(components, mu_domain, tonic_hz):
    """
    components: list of (mu, spread, weight)
    mu_domain: 'angle' | 'midi' | 'hz'
    returns: np.ndarray of μ in semitones relative to tonic, in [0,12)
    """
    if components is None or len(components) == 0:
        return np.array([], dtype=float)
    mus = np.array([c[0] for c in components], dtype=float)

    if mu_domain == 'angle':
        semi = angles_to_semitones_rel_tonic(mus, tonic_hz)
    elif mu_domain == 'midi':
        semi = midi_to_semitones_rel_tonic(mus, tonic_hz)
    elif mu_domain == 'hz':
        semi = hz_to_semitones_rel_tonic(mus, tonic_hz)
    else:
        raise ValueError("mu_domain must be one of {'angle','midi','hz'}")
    return semi

# ----------------- grid builder (your matching logic) -----------------
def build_confusion_grid_tolerance_by_label(
    kde_df,
    vonmises_df,
    tonics,              # dict: artist -> tonic_hz
    model_col,           # e.g. 'vmm_components' or 'gmm_var_components'
    mu_domain='angle',   # 'angle' | 'midi' | 'hz'
    tol=0.4,
    sample_order='raga'  # 'as_is' | 'raga'
):
    """
    Returns:
      grid: (12, N) with codes {0=TN, 1=FP, 2=FN, 3=TP}
      sample_index: list of row indices in plotted order
    Matching logic:
      For each semitone label s present in the ground truth:
        - Let μ_gt be that row's ground-truth μ (semitones rel. tonic).
        - If any predicted μ is within tol (circular) of μ_gt: TP, else FN.
      After TPs are assigned, remaining unmatched predicted μ are FPs:
        - Round each unmatched μ_pred to nearest int r in [0..11];
        - If r NOT in the raga’s ground-truth label set, mark cell [r, col] as FP (do not override FN/TP).
      All remaining cells are TN.
    """
    # Column order
    if sample_order == 'raga':
        sample_index = kde_df.sort_values('raga').index.tolist()
    else:
        sample_index = kde_df.index.tolist()

    N = len(sample_index)
    grid = np.zeros((12, N), dtype=int)

    for c, idx in enumerate(sample_index):
        row = kde_df.loc[idx]
        raga = row['raga']
        artist = row['artist']
        tonic_hz = tonics.get(artist, None)
        if tonic_hz is None:
            continue

        # Ground truth rows for this sample
        gt_rows = vonmises_df[(vonmises_df['raga'] == raga) & (vonmises_df['artist'] == artist)]
        # GT label set (integers in 0..11)
        gt_label_set = set(gt_rows['swara_label'].astype(int).tolist())

        # Ground truth μ values (already tonic-normalized semitones per your dataset)
        # If your gt μ are guaranteed in [0,12), the next mod is safe and harmless.
        gt_mu_by_label = {}
        for _, r in gt_rows.iterrows():
            s = int(r['swara_label'])
            gt_mu_by_label[s] = np.mod(float(r['mu']), 12.0)

        # Predicted μ (semitones relative to tonic)
        pred_components = row.get(model_col, None)
        pred_semis = comps_to_semis_rel_tonic(pred_components, mu_domain, tonic_hz)
        used_pred = np.zeros(len(pred_semis), dtype=bool)

        # 1) For each ground-truth label s, try to match μ_pred to μ_gt within tol
        for s in sorted(gt_label_set):
            mu_gt = gt_mu_by_label[s]
            if pred_semis.size == 0:
                grid[s, c] = 2  # FN
                continue
            dists = circ_dist12(pred_semis, mu_gt)
            # mask out already-used preds
            dists_masked = np.where(used_pred, np.inf, dists)
            j = np.argmin(dists_masked)
            if dists_masked[j] <= tol:
                grid[s, c] = 3   # TP
                used_pred[j] = True
            else:
                grid[s, c] = 2   # FN

        # 2) Remaining unmatched preds are candidate FPs
        #    Round each unmatched pred to nearest int r; if r NOT in raga, mark FP at [r, c]
        unmatched_idx = np.where(~used_pred)[0]
        for j in unmatched_idx:
            mu_pred = pred_semis[j]
            r = int(np.mod(np.rint(mu_pred), 12))  # nearest label
            if r not in gt_label_set:
                # Only mark FP if the cell isn't already TP/FN
                if grid[r, c] == 0:
                    grid[r, c] = 1  # FP
            # If r is in gt_label_set, do nothing here: the FN/TP status for that s was already decided above.

        # 3) All other cells remain TN (0)
    return grid, sample_index

# ----------------- plotting (unchanged) -----------------
def plot_confusion_heatmap(
    grid,
    kde_df,
    sample_index,
    title="Tolerance-based match: semitone × sample (TP/FP/FN/TN)",
    show_raga_block_labels=True
):
    # TP=3, FN=2, FP=1, TN=0
    colors = [
        "#BFE5DB",  # TN = pale teal (soft good)
        "#E69F00",  # FP = amber/orange (warning)
        "#A23B72",  # FN = plum/red-purple (error)
        "#009E73",  # TP = strong teal (success)
    ]
    cmap = ListedColormap(colors)
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

    fig, ax = plt.subplots(figsize=(1.0 + 0.28*grid.shape[1], 6))
    ax.imshow(grid, aspect='auto', cmap=cmap, norm=norm, origin='lower')

    ax.set_yticks(np.arange(12))
    ax.set_yticklabels([str(i) for i in range(12)])
    ax.set_ylabel("Numerical Semitone Label (0–11)")
    ax.set_title(title)

    if show_raga_block_labels:
        df_sorted = kde_df.loc[sample_index].reset_index()
        ragas = df_sorted['raga'].tolist()
        tick_pos, tick_lab = [], []
        prev = None
        for i, r in enumerate(ragas):
            if r != prev:
                tick_pos.append(i)
                tick_lab.append(r)
                prev = r
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_lab, rotation=45, ha='right')
        ax.set_xlabel("Samples (grouped by raga)")
    else:
        ax.set_xticks([])
        ax.set_xlabel("Samples")

    from matplotlib.patches import Patch
    legend_patches = [
        Patch(facecolor=colors[3], label='TP'),
        Patch(facecolor=colors[2], label='FN'),
        Patch(facecolor=colors[1], label='FP'),
        Patch(facecolor=colors[0], label='TN'),
    ]
    ax.legend(handles=legend_patches, loc='upper right', frameon=True)

    plt.tight_layout()
    plt.show()


In [ ]:
# Example: visualize a VMM model stored as angle μ (radians)
grid, idx_order = build_confusion_grid_tolerance_by_label(
    kde_df,
    vonmises_df,
    tonics,
    model_col='vmm_var_components', # list of (mu_angle, kappa, weight)
    mu_domain='angle',          # 'angle' | 'midi' | 'hz'
    tol=0.5,                    # semitone tolerance
    sample_order='raga'         # group columns by raga
)
plot_confusion_heatmap(grid, kde_df, idx_order, title='Component Accuracy of VMM-C with Variable $n_c$ (tol=0.5)')


### Tonic Estimation

In [ ]:
vonmises_df

In [ ]:
# FUNCTIONS
def plot_kappa_vs_weight(vonmises_df):
    # Define swara category
    def categorize(label):
        if label == 0:
            return "Tonic"
        elif label == 7:
            return "Fifth"
        else:
            return "Other"

    vonmises_df["swara_category"] = vonmises_df["swara_label"].apply(categorize)

    # Set up colors
    colors = {"Tonic": "darkgreen", "Fifth": "darkorange", "Other": "gray"}

    # Plot
    plt.figure(figsize=(8, 6))
    for category, group in vonmises_df.groupby("swara_category"):
        plt.scatter(group["kappa"], group["weight"],
                    label=category,
                    color=colors[category],
                    s=80, alpha=0.8, edgecolor='black')

    plt.xlabel(r"$\kappa$ (concentration)")
    plt.ylabel("Weight (proportion of pitch points)")
    plt.title(r"$\kappa$ vs Weight by Swara Type")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_boosted_score(vonmises_df):
    # First: base score for each label
    vonmises_df["base_score"] = vonmises_df["kappa"] * vonmises_df["weight"]

    # Create a lookup table: (raga, artist, label) → score
    key_to_score = {
        (row["raga"], row["artist"], row["swara_label"]): row["base_score"]
        for _, row in vonmises_df.iterrows()
    }

    # Compute boosted score
    boosted_scores = []
    for _, row in vonmises_df.iterrows():
        key = (row["raga"], row["artist"], row["swara_label"])
        fifth_label = (row["swara_label"] + 7) % 12
        fifth_key = (row["raga"], row["artist"], fifth_label)
        fifth_score = key_to_score.get(fifth_key, 0)
        boosted_scores.append(row["base_score"] + fifth_score)

    vonmises_df["boosted_score"] = boosted_scores

    # Categorize tonic, fifth, other
    def categorize(label):
        if label == 0:
            return "Tonic"
        elif label == 7:
            return "Fifth"
        else:
            return "Other"

    vonmises_df["swara_category"] = vonmises_df["swara_label"].apply(categorize)

    # Plot
    colors = {"Tonic": "darkgreen", "Fifth": "darkorange", "Other": "gray"}
    plt.figure(figsize=(8, 6))
    for category, group in vonmises_df.groupby("swara_category"):
        plt.scatter(group["kappa"], group["weight"],
                    label=category,
                    color=colors[category],
                    s=100 * group["boosted_score"],  # bubble size encodes score
                    alpha=0.7, edgecolor='black')

    plt.xlabel(r"$\kappa$ (concentration)")
    plt.ylabel("Weight (proportion of pitch points)")
    plt.title(r"Boosted Score: $\kappa \cdot w$ + fifth's $\kappa \cdot w$")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_boosted_score_violin(vonmises_df):
    # Step 1: Base score = kappa * weight
    vonmises_df["base_score"] = vonmises_df["kappa"] * vonmises_df["weight"]

    # Step 2: Build lookup table for (raga, artist, label) → score
    key_to_score = {
        (row["raga"], row["artist"], row["swara_label"]): row["base_score"]
        for _, row in vonmises_df.iterrows()
    }

    # Step 3: Compute boosted score = own score + score of fifth (label + 7 mod 12)
    boosted_scores = []
    for _, row in vonmises_df.iterrows():
        key = (row["raga"], row["artist"], row["swara_label"])
        fifth_label = (row["swara_label"] + 7) % 12
        fifth_key = (row["raga"], row["artist"], fifth_label)
        fifth_score = key_to_score.get(fifth_key, 0)
        boosted_scores.append(row["base_score"] + fifth_score)

    vonmises_df["boosted_score"] = boosted_scores

    # Step 4: Swara type
    def categorize(label):
        if label == 0:
            return "Tonic"
        elif label == 7:
            return "Fifth"
        else:
            return "Other"

    vonmises_df["swara_category"] = vonmises_df["swara_label"].apply(categorize)

    # Step 5: Plot violinplot
    plt.figure(figsize=(8, 6))
    sns.violinplot(data=vonmises_df, x="swara_category", y="boosted_score", palette={
        "Tonic": "darkgreen",
        "Fifth": "darkorange",
        "Other": "gray"
    })

    plt.ylabel(r"Boosted Score: $\kappa \cdot w$ + fifth's $\kappa \cdot w$")
    plt.xlabel("Swara Type")
    plt.title("Distribution of Boosted Scores by Swara Type")
    plt.grid(True, axis='y')
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_mu_distance_from_tonic(vonmises_df):
    # Step 1: Build (raga, artist) → tonic mu lookup
    tonic_mu_map = {
        (row["raga"], row["artist"]): row["mu"]
        for _, row in vonmises_df[vonmises_df["swara_label"] == 0].iterrows()
    }

    # Step 2: Compute distance to tonic mu for each row
    distances = []
    for _, row in vonmises_df.iterrows():
        key = (row["raga"], row["artist"])
        tonic_mu = tonic_mu_map.get(key, None)
        if tonic_mu is not None:
            dist = (row['mu'] + (6 -row['swara_label'])) % 12 - (6 -row['swara_label'])
            distances.append(dist)
        else:
            distances.append(np.nan)  # if no tonic found, skip

    vonmises_df["dist_from_tonic"] = distances

    # Step 3: Plot violinplot by swara label (as categorical x-axis)
    plt.figure(figsize=(12, 6))
    sns.violinplot(data=vonmises_df, x="swara_label", y="dist_from_tonic", inner="point", color='mediumpurple')

    plt.xlabel("Swara Label")
    plt.ylabel(r"Ground Truth Component Means")
    plt.title("Distribution of Swara Means")
    plt.grid(True, axis='y')
    plt.tight_layout()
    plt.show()


In [ ]:
# TEST

plot_kappa_vs_weight(vonmises_df)

plot_boosted_score(vonmises_df)

plot_boosted_score_violin(vonmises_df)

In [ ]:
# TEST

plot_mu_distance_from_tonic(vonmises_df)

#### Harmonic Filters

In [ ]:
# FUNCTIONS
# Generate harmonic filters and score according to true tonic information

def build_vmm_harmonic_filters(
    sr,
    vmm_df,
    tonic_hz,
    n_fft=4096,
    num_harmonics=6,
    fixed_kappa=20,
    harmonic_decay=True,
    decay_alpha=0.5,
    normalize=True,
    col = 'mu'
):
    """
    Build normalized harmonic filters for each VMM component, 
    including harmonics for both the component note and its fifth above.

    Parameters:
    - sr: sampling rate
    - vmm_df: DataFrame with 'swara_label' column in semitones
    - tonic_hz: tonic frequency in Hz
    - n_fft: FFT size for frequency resolution
    - num_harmonics: number of harmonics per note
    - fixed_kappa: von Mises spread
    - harmonic_decay: whether to apply decay to higher harmonics
    - decay_alpha: decay factor for harmonic strength
    - normalize: whether to L2 normalize each filter

    Returns:
    - filter_bank: dict mapping index to filter array
    - frequencies: frequency axis for each filter bin
    """
    frequencies = lb.fft_frequencies(sr=sr, n_fft=n_fft)
    filter_bank = {}

    for idx, row in vmm_df.iterrows():
        mu_semitones = row[col]

        # Base pitch and its fifth (7 semitones above)
        target_semitones = [mu_semitones] #, (mu_semitones + 7) % 12]
        harmonic_filter = np.zeros_like(frequencies)

        for semitone in target_semitones:
            base_freq = tonic_hz * (2 ** (semitone / 12))

            for h in range(1, num_harmonics + 1):
                harmonic_freq = h * base_freq
                harmonic_kappa = fixed_kappa

                angle = 2 * np.pi * (np.log2(frequencies / harmonic_freq))
                angle = np.nan_to_num(angle, nan=0.0, posinf=0.0, neginf=0.0)

                weight = np.exp(-decay_alpha * (h - 1)) if harmonic_decay else 1.0
                bump = weight * vonmises.pdf(angle, harmonic_kappa)

                harmonic_filter += bump

        if normalize:
            norm = np.linalg.norm(harmonic_filter)
            if norm > 0:
                harmonic_filter /= norm

        try:
            filter_bank[row['swara_label']] = harmonic_filter
        except:
            filter_bank[idx] = harmonic_filter

    return filter_bank, frequencies

def plot_filters(freqs, filter_bank, vmm_df, ax=None, tonic_midi=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))

    midis = lb.hz_to_midi(freqs)

    for label, filt in filter_bank.items():
        # Find the corresponding row for this swara_label
        row = vmm_df[vmm_df["swara_label"] == label].iloc[0]

        mu = row["mu"]
        kappa = row["kappa"]

        ax.plot(midis, filt, label=f'Swara {label} (μ={mu:.3f}, κ={kappa:.3f})')

    ax.set_xlabel("MIDI Pitch")
    ax.set_ylabel("Filter Magnitude")
    ax.set_title("Harmonic Filters Shaped by VMM Components")
    
    if tonic_midi is not None:
        ax.axvline(tonic_midi, linestyle='--', color='gray')

    ax.legend()
    ax.grid(False)
    plt.show()

def score_vmm_filters_against_spectrum(audio, filt_bank, n_fft=4096, hop_length=1024, use_median=True):
    """
    Scores von Mises harmonic filters against the average power spectrum of an audio signal.

    Parameters:
    - audio: 1D numpy array of audio signal
    - sr: sampling rate
    - vmm_df: dataframe with columns ['mu', 'kappa', 'swara_label']
    - tonic_hz: tonic frequency in Hz
    - n_fft: FFT size
    - hop_length: hop size for STFT
    - use_median: whether to use median (vs mean) for spectrum summarization

    Returns:
    - scores: dict mapping VMM component index to similarity score
    - avg_spectrum: normalized average spectrum
    - filter_bank: dict mapping component index to harmonic filter
    - frequencies: frequency bins corresponding to the filters and spectrum
    """
    # Compute power spectrogram
    S = np.abs(lb.stft(audio, n_fft=n_fft, hop_length=hop_length))**2
    spectrum = np.median(S, axis=1) if use_median else np.mean(S, axis=1)
    spectrum /= np.linalg.norm(spectrum)  # L2 normalize

    # Score each filter
    scores = {}
    for idx, filt in filt_bank.items():
        print(idx)
        score = np.dot(spectrum, filt)
        scores[idx] = score

    return scores, spectrum

def plot_spec_scores(score_dict, vmm_df, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    # Sort scores by descending value
    sorted_scores = sorted(score_dict.items(), key=lambda x: -x[1])
    labels, values = zip(*sorted_scores)  # swara_labels are now the keys

    # Optionally confirm these are in vmm_df
    rows = vmm_df[vmm_df["swara_label"].isin(labels)]

    ax.bar(range(len(values)), values, tick_label=labels, color='skyblue', edgecolor='black')
    ax.set_xlabel("Swara Label")
    ax.set_ylabel("Similarity Score")
    ax.set_title("Swara-wise Harmonic Match to Spectrum")
    ax.grid(False)
    plt.show()

In [ ]:
# TEST

ex_vmm_df = vonmises_df[(vonmises_df['raga'] == 'kalyani') & (vonmises_df['artist'] == 'badrinarayanan')]
display(ex_vmm_df)

ex_filt_bank, ex_freqs = build_vmm_harmonic_filters(
                    sr=sr,
                    vmm_df=ex_vmm_df,
                    tonic_hz=ex_tonic_hz,
                    n_fft=4096,
                    num_harmonics=6,
                    fixed_kappa=15,
                    harmonic_decay=True,
                    decay_alpha=0.5,
                    normalize=False
                )
plot_filters(ex_freqs, ex_filt_bank, ex_vmm_df, tonic_midi = lb.hz_to_midi(ex_tonic_hz))
ex_scores, ex_spectrum = score_vmm_filters_against_spectrum(ex_audio, ex_filt_bank)
plot_spec_scores(ex_scores, ex_vmm_df)

#### Fixed Semitone Grid Estimation

In [ ]:
# FUNCTIONS

def find_best_grid_shift_assignment(
    vmm_df,
    steps=200,
    plot=True,
    distance_fn=None,
    ax=None
):
    """
    Finds the optimal shift for a 12-tone grid minimizing custom cost between VMM components and gridlines.
    
    Parameters:
    - vmm_df: DataFrame with 'mu' and optionally 'kappa'
    - distance_fn: function(mu, grid_val, kappa) → scalar cost
    - ax: optional matplotlib Axes object to plot into (for multi-plot layouts)
    """
    mus = vmm_df['mu'].values
    kappas = vmm_df['kappa'].values if 'kappa' in vmm_df.columns else np.ones_like(mus)
    shifts = np.linspace(0, 1, steps, endpoint=False)
    mses = []
    assignments_per_shift = []

    if distance_fn is None:
        # Default: squared circular distance weighted by kappa
        def distance_fn(mu, g, k): 
            diff = np.abs(mu - g) % 12
            diff = np.minimum(diff, 12 - diff)
            return k * diff**2

    for shift in shifts:
        grid = (np.arange(12) + shift) % 12
        dist_matrix = np.zeros((len(mus), 12))
        for i, mu in enumerate(mus):
            for j, g in enumerate(grid):
                dist_matrix[i, j] = distance_fn(mu, g, kappas[i])

        row_ind, col_ind = linear_sum_assignment(dist_matrix)
        mse = dist_matrix[row_ind, col_ind].sum()
        mses.append(mse)
        assignments_per_shift.append((row_ind, col_ind, grid))

    best_idx = np.argmin(mses)
    best_shift = shifts[best_idx]
    best_mse = mses[best_idx]
    best_row_ind, best_col_ind, best_grid = assignments_per_shift[best_idx]
    best_assignments = best_grid[best_col_ind]

    if plot:
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(shifts, mses, label="Objective vs Shift")
        ax.axvline(best_shift, color='red', linestyle='--', label=f"Best Shift = {best_shift:.3f}")
        ax.set_xlabel("Grid Shift (in semitones)")
        ax.set_ylabel("Total Assignment Cost")
        ax.set_title("Optimal Unique 12-Tone Grid Alignment")
        ax.legend()
        ax.grid(True)

    return best_shift, best_mse, shifts, mses, best_assignments

def compare_distance_functions(vmm_df, distance_fns, steps=200):
    """
    Plots the Objective vs Shift curve for multiple distance functions.

    Parameters:
    - vmm_df: DataFrame with 'mu' and optionally 'kappa'
    - distance_fns: list of (name, fn) tuples
    - steps: number of shift candidates
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    for ax, (name, fn) in zip(axes, distance_fns):
        find_best_grid_shift_assignment(
            vmm_df,
            steps=steps,
            plot=True,
            distance_fn=fn,
            ax=ax
        )
        ax.set_title(name)

    plt.tight_layout()
    plt.show()

In [ ]:
# DISTANCE FUNCTIONS

def dist_squared(mu, g, kappa):
    diff = np.abs(mu - g) % 12
    diff = np.minimum(diff, 12 - diff)
    return diff**2

def dist_pow3(mu, g, kappa):
    diff = np.abs(mu - g) % 12
    diff = np.minimum(diff, 12 - diff)
    return diff**3

def dist_kappa_weighted(mu, g, kappa):
    diff = np.abs(mu - g) % 12
    diff = np.minimum(diff, 12 - diff)
    return kappa**2 * diff

def dist_kappa_bounded(mu, g, kappa):
    diff = np.abs(mu - g) % 12
    diff = np.minimum(diff, 12 - diff)
    return (min(kappa, 10))**2 * diff

def dist_log_weighted(mu, g, kappa):
    diff = np.abs(mu - g) % 12
    diff = np.minimum(diff, 12 - diff)
    return np.log1p(kappa) * diff**2

In [ ]:
def assign_snapped_mu(vmm_df, shift):
    """
    Assign each VMM component to a unique gridline on the shifted 12-tone grid
    and add a new column 'snapped_mu' to the dataframe.

    Parameters:
    - vmm_df: DataFrame with 'mu' values in [0, 12)
    - shift: optimal grid shift (in semitones)

    Returns:
    - vmm_df: same DataFrame with added 'snapped_mu' column
    """
    mus = vmm_df['mu'].values
    grid = (np.arange(12) + shift) % 12

    # Compute cost matrix with circular distances
    cost_matrix = np.zeros((len(mus), 12))
    for i, mu in enumerate(mus):
        dists = np.abs(mu - grid) % 12
        cost_matrix[i] = np.minimum(dists, 12 - dists) ** 2

    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # Assign snapped mu values using matched grid positions
    snapped_mu = np.full_like(mus, fill_value=np.nan)
    for i, j in zip(row_ind, col_ind):
        snapped_mu[i] = grid[j]

    vmm_df['snapped_mu'] = snapped_mu
    return vmm_df

def add_snapped_mu_column(vonmises_df, steps=200, plot=True, dist_fn = None):
    """
    For each raga-artist group in vonmises_df, compute the best 12-tone grid alignment
    and assign snapped_mu using Hungarian matching.

    Parameters:
    - vonmises_df: DataFrame with ['raga', 'artist', 'mu']
    - steps: number of shift candidates to evaluate
    - plot: whether to show MSE vs shift plots

    Returns:
    - updated_df: copy of vonmises_df with 'snapped_mu' column
    """
    updated_dfs = []
    mse = 0
    baseline = 0
    count = 0

    for (raga, artist), group in vonmises_df.groupby(["raga", "artist"]):
        best_shift, *_ = find_best_grid_shift_assignment(group, distance_fn=dist_fn, steps=steps, plot=plot)
        error = min(best_shift, 1 - best_shift)
        mse += error*error
        base_error = min([min(mu, 12-mu) for mu in group['mu']])
        baseline += base_error*base_error
        # Use assign_snapped_mu to append the snapped_mu column
        snapped_group = assign_snapped_mu(group.copy(), best_shift)
        updated_dfs.append(snapped_group)
        count += 1

    return pd.concat(updated_dfs, ignore_index=True), mse / count, baseline / count

In [ ]:
# TEST

distance_functions = [
    ("MSE-B", dist_squared),
    ("Pow3", dist_pow3),
    ("Kappa-Weighted", dist_kappa_weighted),
    ("Kappa-Log", dist_log_weighted)
]

compare_distance_functions(ex_vmm_df, distance_functions)

for dfn in [dist_squared, dist_pow3, dist_kappa_weighted, dist_log_weighted]:
    vmm_snapped_df, mse, base_mse = add_snapped_mu_column(vonmises_df, dist_fn = dfn, plot = False)
    print(mse, base_mse)

#### Evaluation

In [ ]:
# FUNCTIONS

def evaluate_tonic_detection(score_dicts, top_k=1):
    """
    Evaluate tonic detection accuracy across multiple samples.

    Parameters:
    - score_dicts: list of dicts {swara_label: score}
    - true_labels: list of ground truth tonic labels (usually all 0)
    - top_k: evaluate Top-K accuracy

    Returns:
    - top_k_acc: proportion of samples where true label is in top K
    - mean_rank: average rank of true tonic
    - mean_gap: avg (tonic score - max non-tonic score)
    """
    top_k_hits = 0
    total_rank = 0
    total_gap = 0

    for scores in score_dicts:
        sorted_items = sorted(scores.items(), key=lambda x: -x[1])
        labels_ranked = [label for label, score in sorted_items]
        rank = labels_ranked.index(0)
        total_rank += rank

        if rank < top_k:
            top_k_hits += 1

        tonic_score = scores.get(0, 0)
        other_scores = [value for key, value in scores.items() if key != 0]
        max_other = max(other_scores) if other_scores else 0
        total_gap += tonic_score - max_other

    top_k_acc = top_k_hits / len(score_dicts)
    mean_rank = total_rank / len(score_dicts)
    mean_gap = total_gap / len(score_dicts)

    return {
        "Top-{} Accuracy".format(top_k): top_k_acc,
        "Mean Rank": mean_rank,
        "Mean Gap": mean_gap
    }


In [ ]:
# HYPERPARAMETER TUNING

# Initialize storage
top1_grid_results = []
top2_grid_results = []
top3_grid_results = []

# Inside your grid search loop:
for num_harmonics in [2, 3, 4, 5, 6, 7, 8, 9, 10]:
    for fixed_kappa in [30, 40, 50, 60, 70, 80, 90, 100, 110]:
        for decay_alpha in [0.0]:
            print(f"Trying: harmonics={num_harmonics}, kappa={fixed_kappa}, alpha={decay_alpha}")
            all_scores = []
            all_true_labels = []

            for key, value in dataset.items():
                item, x, sr = value
                artist = item.split("-")[3]
                raga = item.split("-")[5]

                tonic_hz = tonics.get(artist, None)
                vmm_sample = vmm_snapped_df[(vmm_snapped_df["raga"] == raga) & (vmm_snapped_df["artist"] == artist)]

                if tonic_hz is None or vmm_sample.empty:
                    continue

                # Build filters with current hyperparams
                filt_bank, freqs = build_vmm_harmonic_filters(
                    sr=sr,
                    vmm_df=vmm_sample,
                    tonic_hz=tonic_hz,
                    n_fft=4096,
                    num_harmonics=num_harmonics,
                    fixed_kappa=fixed_kappa,
                    harmonic_decay=True,
                    decay_alpha=decay_alpha,
                    normalize=False,
                    col = 'snapped_mu'
                )

                # Score spectrum with filters
                scores, spectrum = score_vmm_filters_against_spectrum(x, filt_bank)
                all_scores.append(scores)

            # Compute metrics
            run1_metrics = evaluate_tonic_detection(all_scores, top_k=1)
            run2_metrics = evaluate_tonic_detection(all_scores, top_k=2)
            run3_metrics = evaluate_tonic_detection(all_scores, top_k=3)

            # Log into dataframe row
            top1_grid_results.append({
                "num_harmonics": num_harmonics,
                "fixed_kappa": fixed_kappa,
                "decay_alpha": decay_alpha,
                **run1_metrics
            })

            top2_grid_results.append({
                "num_harmonics": num_harmonics,
                "fixed_kappa": fixed_kappa,
                "decay_alpha": decay_alpha,
                **run2_metrics
            })

            top3_grid_results.append({
                "num_harmonics": num_harmonics,
                "fixed_kappa": fixed_kappa,
                "decay_alpha": decay_alpha,
                **run3_metrics
            })

results1_df = pd.DataFrame(top1_grid_results)
results2_df = pd.DataFrame(top2_grid_results)
results3_df = pd.DataFrame(top3_grid_results)

pivot1_df = results1_df.pivot(index="num_harmonics", columns="fixed_kappa", values="Top-1 Accuracy")
pivot2_df = results2_df.pivot(index="num_harmonics", columns="fixed_kappa", values="Top-2 Accuracy")
pivot3_df = results3_df.pivot(index="num_harmonics", columns="fixed_kappa", values="Top-3 Accuracy")


# Plot heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(pivot1_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Top-1 Accuracy'})
plt.title("Grid Search: Top-1 Accuracy by Num Harmonics and Fixed Kappa")
plt.xlabel("Fixed Kappa")
plt.ylabel("Number of Harmonics")
plt.tight_layout()
plt.show()

# Plot heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(pivot2_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Top-2 Accuracy'})
plt.title("Grid Search: Top-2 Accuracy by Num Harmonics and Fixed Kappa")
plt.xlabel("Fixed Kappa")
plt.ylabel("Number of Harmonics")
plt.tight_layout()
plt.show()

# Plot heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(pivot3_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Top-3 Accuracy'})
plt.title("Grid Search: Top-3 Accuracy by Num Harmonics and Fixed Kappa")
plt.xlabel("Fixed Kappa")
plt.ylabel("Number of Harmonics")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create figure with 3 subplots side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True, sharex=True, sharey=True)

# Common settings
heatmap_kwargs = dict(
    annot=True, fmt=".2f", cmap="YlGnBu",
    vmin=0, vmax=0.75,
    cbar=False  # turn off individual colorbars
)

# --- Heatmap 1 ---
sns.heatmap(pivot1_df, ax=axes[0], **heatmap_kwargs)
axes[0].set_title("Top-1 Accuracy")

# --- Heatmap 2 ---
sns.heatmap(pivot2_df, ax=axes[1], **heatmap_kwargs)
axes[1].set_title("Top-2 Accuracy")

# --- Heatmap 3 ---
sns.heatmap(pivot3_df, ax=axes[2], **heatmap_kwargs)
axes[2].set_title("Top-3 Accuracy")

# One shared colorbar
cbar = fig.colorbar(axes[0].collections[0], ax=axes, location="right", shrink=0.9, pad=0.02)
cbar.set_label("Accuracy")

plt.show()


## 3. Contour Segmentation

In [ ]:
# FUNCTIONS

def plot_segments(p0, sr, breakpoints, hop_length = 1024):
    '''
    Inputs:
    1. f0: fundamental pitch contour (ARR)
    2. sr: sampling rate of the original audio (INT)
    3. breakpoints: segment boundaries as indices of f0 (ARR)
    4. hop_length: hop length used to compute f0 (INT)

    Outputs:
    1. plot of f0 with vertical boundaries at breakpoints
    '''
    
    # Compute time axis for f0 contour
    print(p0.shape)
    times = lb.times_like(p0, sr=sr, hop_length=hop_length)
    print('have times')
    # Plot f0 contour
    plt.figure(figsize=(12, 6))
    plt.plot(times, p0, label="p0 Contour", color='b', linewidth=2)

    # Add vertical lines for segment boundaries
    for idx in breakpoints:
        if 0 <= idx < len(times):  # Ensure the index is valid
            plt.axvline(times[idx], color='r', linestyle='-', linewidth=0.1, label="Segment Boundary" if idx == breakpoints[0] else "")

    # Labels and legend
    plt.xlabel("Time (s)")
    plt.ylabel("Semitone (Relative to Tonic)")
    plt.title("Fundamental Pitch Contour with Segment Boundaries")
    plt.legend()
    plt.show()

### Evaluation

In [ ]:
def evaluate_segmentation(predicted, ground_truth, tolerance=2, return_matches=False):
    """
    Evaluate segmentation using precision, recall, F1, and a distance-based cost.

    Args:
        predicted (list or np.array): Predicted breakpoint indices.
        ground_truth (list or np.array): Ground truth breakpoint indices.
        tolerance (int): Allowed distance for matching.
        return_matches (bool): Whether to return matched/unmatched indices.

    Returns:
        metrics (dict): precision, recall, f1, and mse_cost
        (optional) matches (dict): matched_pairs, unmatched_predicted, unmatched_ground_truth
    """
    predicted = np.array(predicted)
    ground_truth = np.array(ground_truth)

    matched_pred = set()
    matched_gt = set()
    matched_pairs = []

    for i, gt in enumerate(ground_truth):
        distances = np.abs(predicted - gt)
        within_tol = np.where(distances <= tolerance)[0]
        if len(within_tol) > 0:
            closest_idx = within_tol[np.argmin(distances[within_tol])]
            if closest_idx not in matched_pred:
                matched_pred.add(closest_idx)
                matched_gt.add(i)
                matched_pairs.append((predicted[closest_idx], gt))

    TP = len(matched_pairs)
    FP = len(predicted) - len(matched_pred)
    FN = len(ground_truth) - len(matched_gt)

    precision = TP / (TP + FP) if TP + FP > 0 else 0.0
    recall = TP / (TP + FN) if TP + FN > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    if TP > 0:
        distances = [(p - g) ** 2 for p, g in matched_pairs]
        mse_cost = np.mean(distances)
    else:
        mse_cost = float('inf')  # No matches → infinite alignment cost

    results = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mse_cost': mse_cost
    }

    if return_matches:
        unmatched_pred = [p for i, p in enumerate(predicted) if i not in matched_pred]
        unmatched_gt = [g for i, g in enumerate(ground_truth) if i not in matched_gt]
        matches = {
            'matched_pairs': matched_pairs,
            'false_pos': unmatched_pred,
            'false_neg': unmatched_gt
        }
        return results, matches

    return results


## 4. Tempo Estimation

### Akshara Distribution Studies

In [ ]:
# FUNCTIONS

def plot_duration_analysis_with_normality(segment_tuples, plot = True):
    """
    Given a list of (frame_start, frame_end) tuples, plot:
    1. A histogram of durations with Gaussian overlay
    2. A scatterplot of durations vs index with a linear regression trendline and equation
    Also prints the p-value from a Shapiro-Wilk test of normality.
    """

    # Compute durations
    durations = [end - start for start, end in segment_tuples]
    X = np.arange(len(durations)).reshape(-1, 1)
    y = np.array(durations)

    # Fit linear regression
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    slope = model.coef_[0]
    intercept = model.intercept_
    regression_eq = f"y = {slope:.3f}x + {intercept:.3f}"

    # Compute Gaussian curve
    mu, sigma = np.mean(durations), np.std(durations)
    hist_vals, bin_edges = np.histogram(durations, bins='auto', density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    gaussian_curve = stats.norm.pdf(bin_centers, mu, sigma)

    # Run Shapiro-Wilk normality test
    stat, p_value = stats.shapiro(durations)

    if plot:
        # Plot
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))

        # Histogram + Gaussian
        ax[0].hist(durations, bins='auto', density=True, alpha=0.5, color='skyblue', edgecolor='black')
        ax[0].plot(bin_centers, gaussian_curve, color='red', label='Gaussian Fit')
        ax[0].set_title("Distribution of Akshara Durations\nShapiro-Wilk p-value = {:.4f}".format(p_value))
        ax[0].set_xlabel("Duration (seconds)")
        ax[0].set_ylabel("Density")
        ax[0].legend()

        # Scatter + regression
        ax[1].scatter(X, y, color='darkblue', label='Durations')
        ax[1].plot(X, y_pred, color='red', linestyle='--', label='Linear Fit')
        ax[1].text(0.05, 0.95, regression_eq, transform=ax[1].transAxes,
                fontsize=10, verticalalignment='top', color='red')
        ax[1].set_title("Akshara Durations Over Time")
        ax[1].set_xlabel("Akshara Index")
        ax[1].set_ylabel("Duration (seconds)")
        ax[1].set_ylim((0, 0.5))
        ax[1].legend()

        plt.tight_layout()
        plt.show()

    return mu, sigma, p_value

In [ ]:
# DATAFRAME AUGMENTATION

beat_mus = []
beat_sigmas = []
beat_pvals = []

for key, value in dataset.items():
    # extract metadata from file name
    item, x, sr = value
    artist = item.split("-")[3]
    raga = item.split("-")[5]

    print(raga, artist)
    # extract audio content and pitch contour
    xtonic_hz = tonics[artist]
    xf0, xvflags, xvprobs = get_f0(x, sr, xtonic_hz, display = False)
    xp0 = get_p0(xf0, sr, tonic = xtonic_hz, display = False)
    xp0_smooth = smooth_voiced_sections(xp0, sr, xvflags, filter_size = 7, display = False)
    
    # get beat annotations from sonic visualizer
    xsvl = 'carnatic_varnam_1.0/Notations_Annotations/annotations/taalas/' + raga + '/' + artist + '.svl'
    xparsed_svl = cut_reps(parse_svl(xsvl))
    xbeat_frames = subdivide(xparsed_svl, 2, len(x))
    # Example usage
    mu, sigma, p_val = plot_duration_analysis_with_normality(xbeat_frames/sr, plot = True)
    beat_mus.append(mu)
    beat_sigmas.append(sigma)
    beat_pvals.append(p_val)
    print(f"Shapiro-Wilk p-value: {p_val:.4f}")

kde_df['beat_mu'] = beat_mus
kde_df['beat_sigma'] = beat_sigmas
kde_df['beat_pval'] = beat_pvals

In [ ]:
# FUNCTIONS

def linregression(x, y):
    # Convert to numpy arrays
    x = np.array(x)
    y = np.array(y)

    # Perform linear regression
    slope, intercept, r_value, p_value, std_err = linregress(x, y)

    # Regression line
    y_pred = slope * x + intercept

    # Plot data points
    plt.scatter(x, y, label='Data', color='blue')

    # Plot regression line
    plt.plot(x, y_pred, color='red', label=f'Best Fit: y = {slope:.2f}x + {intercept:.2f}')

    # Labels and legend
    plt.xlabel('Akshara Mean Duration')
    plt.ylabel('Standard Deviation of Akshara Duration')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Print regression stats
    print(f"Slope: {slope:.3f}, Intercept: {intercept:.3f}")
    print(f"R²: {r_value**2:.3f}, p-value: {p_value:.3e}")

In [ ]:
# TEST

ex_mu, ex_sigma, ex_pval = plot_duration_analysis_with_normality(ex_beat_frames/ex_sr)
print(f"Shapiro-Wilk p-value: {ex_pval:.4f}", ex_mu, ex_sigma)

In [ ]:
linregression(beat_mus, beat_sigmas)

### Syllable Detection

In [ ]:
def detect_syllable_onsets(audio, sr, hop_length=1024, backtrack=True, pre_max=30, post_max=30, delta=0.2, wait=20):
    """
    Detect syllable onset times in an audio signal using librosa's onset detection.

    Parameters:
    - audio: np.ndarray, raw audio signal
    - sr: int, sampling rate
    - hop_length: int, number of samples between onset frames
    - backtrack: bool, adjust onset to nearest local minimum in energy
    - pre_max, post_max, delta, wait: onset detection parameters

    Returns:
    - onset_times: np.ndarray of times (in seconds) corresponding to detected onsets
    """

    onset_frames = lb.onset.onset_detect(
        y=audio,
        sr=sr,
        hop_length=hop_length,
        backtrack=backtrack,
        pre_max=pre_max,
        post_max=post_max,
        delta=delta,
        wait=wait
    )

    onset_times = lb.frames_to_time(onset_frames, sr=sr, hop_length=hop_length)
    return onset_times

def times_to_indices(times, sr = 44100, hop_length = 1024):
    return [int(t * sr / hop_length) for t in times]

def add_clicks_to_audio(audio, sr, onset_times, click_freq=1000.0, click_duration=0.03, output_path=None):
    """
    Overlay click sounds at specified onset times on the original audio.
    
    Parameters:
    - audio: np.ndarray, original audio waveform
    - sr: int, sample rate of the audio
    - onset_times: list of onset times in seconds
    - click_freq: frequency of the click tone in Hz
    - click_duration: duration of each click in seconds
    - output_path: optional path to save the output as a .wav file

    Returns:
    - audio_with_clicks: np.ndarray, combined audio
    """
    # Generate click track
    click_track = lb.clicks(times=onset_times, sr=sr, click_freq=click_freq, click_duration=click_duration, length=len(audio))
    
    # Combine original audio with click track
    audio_with_clicks = audio + click_track

    # Normalize to avoid clipping
    max_val = np.max(np.abs(audio_with_clicks))
    if max_val > 1.0:
        audio_with_clicks /= max_val

    # Optionally write to file
    if output_path:
        sf.write(output_path, audio_with_clicks, sr)

    return audio_with_clicks


In [ ]:
# HYPERPARAMETER TUNING

def grid_search_onset_params(
    audio, sr, kbps_gt,
    delta_vals=None,
    wait_vals=None,
    eval_tolerance=2,
    hop_length=1024,  # only used if your helper needs it elsewhere
    verbose=True
):
    """
    Sweeps (delta, wait) for onset detection and evaluates against ground truth.
    Assumes the following helpers already exist in your notebook:
      - detect_syllable_onsets(y, sr, delta, wait) -> times (sec)
      - times_to_indices(times) -> frame indices (or sample indices, as your evaluate_segmentation expects)
      - evaluate_segmentation(pred_inds, gt_inds, tolerance, return_matches=True) -> dict w/ precision, recall, f1, mse_cost
    """
    if delta_vals is None:
        # 10 reasonably spaced thresholds; tune if your ODF scale differs
        delta_vals = np.linspace(0.03, 0.25, 10)
    if wait_vals is None:
        # wait is in frames; pick ~5–23 by 2s (10 values)
        wait_vals = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50], dtype=int)

    P = np.full((len(delta_vals), len(wait_vals)), np.nan)   # precision
    R = np.full((len(delta_vals), len(wait_vals)), np.nan)   # recall
    F = np.full((len(delta_vals), len(wait_vals)), np.nan)   # f1
    E = np.full((len(delta_vals), len(wait_vals)), np.nan)   # mse_cost (lower is better)

    for i, d in enumerate(delta_vals):
        for j, w in enumerate(wait_vals):
            try:
                konset_times = detect_syllable_onsets(audio, sr, delta=float(d), wait=int(w))
                konsets_inds  = times_to_indices(konset_times)
                res_onset, _  = evaluate_segmentation(konsets_inds, kbps_gt, tolerance=eval_tolerance, return_matches=True)

                P[i, j] = res_onset.get('precision', np.nan)
                R[i, j] = res_onset.get('recall', np.nan)
                F[i, j] = res_onset.get('f1', np.nan)
                E[i, j] = res_onset.get('mse_cost', np.nan)
            except Exception as ex:
                if verbose:
                    print(f"[warn] delta={d:.3f}, wait={w}: {ex}")

    # Wrap as DataFrames for convenience
    idx = pd.Index([round(float(x), 3) for x in delta_vals], name="delta")
    cols = pd.Index(wait_vals, name="wait (frames)")
    dfP = pd.DataFrame(P, index=idx, columns=cols)
    dfR = pd.DataFrame(R, index=idx, columns=cols)
    dfF = pd.DataFrame(F, index=idx, columns=cols)
    dfE = pd.DataFrame(E, index=idx, columns=cols)

    # Report best settings
    def best_of(df, higher_is_better=True):
        if higher_is_better:
            pos = np.unravel_index(np.nanargmax(df.values), df.shape)
        else:
            pos = np.unravel_index(np.nanargmin(df.values), df.shape)
        return df.index[pos[0]], df.columns[pos[1]], df.values[pos]

    bP = best_of(dfP, True)
    bR = best_of(dfR, True)
    bF = best_of(dfF, True)
    bE = best_of(dfE, False)

    if verbose:
        print(f"Best precision at delta={bP[0]}, wait={bP[1]} → {bP[2]:.3f}")
        print(f"Best recall    at delta={bR[0]}, wait={bR[1]} → {bR[2]:.3f}")
        print(f"Best F1        at delta={bF[0]}, wait={bF[1]} → {bF[2]:.3f}")
        print(f"Lowest MSE     at delta={bE[0]}, wait={bE[1]} → {bE[2]:.3f}")

    # Plot four heatmaps (separate figures; no seaborn)
    def plot_heatmap(df, title, vmin=None, vmax=None):
        plt.figure()
        im = plt.imshow(df.values, aspect='auto', vmin=vmin, vmax=vmax)
        plt.title(title)
        plt.xlabel(df.columns.name or "wait")
        plt.ylabel(df.index.name or "delta")
        # tick labels
        plt.xticks(ticks=np.arange(len(df.columns)), labels=df.columns, rotation=45)
        plt.yticks(ticks=np.arange(len(df.index)), labels=df.index)
        plt.colorbar(im)
        plt.tight_layout()
        plt.show()

    # For better visual comparability, clamp PRF to [0,1]
    plot_heatmap(dfP, "Precision")
    plot_heatmap(dfR, "Recall")
    plot_heatmap(dfF, "F1-Score")
    plot_heatmap(dfE, "MSE cost")

    return {"precision": dfP, "recall": dfR, "f1": dfF, "mse": dfE}


In [ ]:
# TEST

ex_onset_times = detect_syllable_onsets(ex_audio, ex_sr, delta = 0.05)
ex_onsets_inds = times_to_indices(ex_onset_times)
Audio(add_clicks_to_audio(ex_audio, ex_sr, ex_onset_times), rate = ex_sr)

### IOI Computation

In [ ]:
# FUNCTIONS

def beat_score_scaled(candidate, iois, tol, scale='sqrt'):
    iois = np.asarray(iois)
    k = np.round(iois / candidate)
    error = np.abs(iois - k * candidate)

    if scale == 'linear':
        tol_scaled = tol * k
    elif scale == 'sqrt':
        tol_scaled = tol * np.sqrt(k)
    elif scale == 'log':
        tol_scaled = tol * np.log1p(k)
    # TODO: insert other things here for scaling function
    else:
        raise ValueError("Unsupported scale type.")

    return np.mean(error <= tol_scaled)

def estimate_beat_from_onsets(onsets, min_beat=0.2, max_beat=0.50, sr=44100, tol=0.05, scale='sqrt', plot=True):
    onsets = np.asarray(onsets)
    iois = np.diff(onsets)

    # Candidate beat durations to test
    candidates = np.linspace(min_beat, max_beat, 500)
    scores = [beat_score_scaled(b, iois, tol=0.05*b, scale=scale) for b in candidates]

    best_idx = np.argmax(scores)
    best_beat_duration = candidates[best_idx]
    best_bpm = 60.0 / best_beat_duration

    if plot:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(candidates, scores)
        ax.axvline(best_beat_duration, color='r', linestyle='--', label=f"Best Beat Length: {best_beat_duration:.2f}")
        ax.set_xlabel("Beat Duration (seconds)")
        ax.set_ylabel("Proportion of IOIs matched")
        ax.set_title("Beat Estimation via Scaled Tolerance Matching")
        ax.legend()
        plt.tight_layout()
        plt.show()

    return best_beat_duration, scores, candidates


In [ ]:
kde_df.head(6)

In [ ]:
ex_beat_est, _, _ = estimate_beat_from_onsets(ex_onset_times)

In [ ]:
# TODO: get mean precision/recall/f1 across dataset for hyperparameter tuning

In [ ]:
onset_time_list = []
onset_ind_list = []
beat_dur_ests = []
for key, value in dataset.items():
    # extract metadata from file name
    item, x, sr = value
    artist = item.split("-")[3]
    raga = item.split("-")[5]
    xonset_times = detect_syllable_onsets(x, sr, delta = 0.054, wait = 35)
    xonset_inds = times_to_indices(xonset_times)
    onset_time_list.append(xonset_times)
    onset_ind_list.append(xonset_inds)
    beat_est, _, _ = estimate_beat_from_onsets(xonset_times, min_beat = 0.25, max_beat = 0.50, plot=False)
    beat_dur_ests.append(beat_est)
kde_df['onset_times'] = onset_time_list
kde_df['onset_inds'] = onset_ind_list
kde_df['beat_dur_ests'] = beat_dur_ests

In [ ]:
import matplotlib.pyplot as plt

# Extract ground truth and estimates
x = kde_df['beat_mu']        # ground truth mean akshara duration
y = kde_df['beat_dur_ests']   # predicted durations

plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.7, edgecolor='k')

# Plot the reference line y = x
lims = [
    min(min(x), min(y)),
    max(max(x), max(y))
]
plt.plot(lims, lims, 'r--', linewidth=2, label='y = x')

plt.xlabel("Ground truth mean akṣara duration (s)")
plt.ylabel("Predicted mean akṣara duration (s)")
plt.title("Tempo Estimation Accuracy")
plt.legend()
plt.axis('equal')  # ensures 1:1 aspect
plt.tight_layout()
plt.show()


## 5. Dynamic Programming

### Preprocessing

In [ ]:
all_segments['duration'] = all_segments['time_end'] - all_segments['time_start']

In [ ]:
all_segments.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Parameters
FRAME_SEC = 0.023  # frame duration in seconds (adjust if needed)

# Select 9 tonic (0-labeled) segments
segments = all_segments[all_segments["label"] == 6].sample(9, random_state=0)
p0_list = segments["p0"].to_list()
means = segments["mean_octinv"].to_numpy()

# Global y-limits for consistency across 9 plots
y_min = np.nanmin([np.nanmin(seg) for seg in p0_list])
y_max = np.nanmax([np.nanmax(seg) for seg in p0_list])
y_lims = (y_min, y_max)

# Figure setup with constrained_layout
fig = plt.figure(figsize=(12, 4), constrained_layout=True)
outer = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1])

# --- Left: 3x3 grid of contours ---
subfig_left = fig.add_subfigure(outer[0, 0])
gs_left = subfig_left.add_gridspec(3, 3, wspace=0.05, hspace=0.08)

for i, seg in enumerate(p0_list):
    r, c = divmod(i, 3)
    ax = subfig_left.add_subplot(gs_left[r, c])
    x = np.arange(len(seg)) * FRAME_SEC
    ax.plot(x, seg, linewidth=1)
    ax.axhline(np.nanmean(seg), color="purple", linestyle="--", linewidth=1)
    ax.set_ylim(y_lims)
    ax.set_xticks([]); ax.set_yticks([])

subfig_left.suptitle("Segment Contours ($M_2$)", fontsize=11)

# --- Middle: histogram of means ---
ax_hist = fig.add_subplot(outer[0, 1])
ax_hist.hist(means, bins=10, color="skyblue", edgecolor="black")
ax_hist.set_title("Statistic Histogram", fontsize=11)
ax_hist.set_xlabel("Mean (octave-invariant semitone)")
ax_hist.set_ylabel("Count")

# --- Right: KDE of means ---
ax_kde = fig.add_subplot(outer[0, 2])
sns.kdeplot(means, fill=True, ax=ax_kde, color="purple")
ax_kde.set_title("Statistic KDE", fontsize=11)
ax_kde.set_xlabel("Mean (octave-invariant semitone)")
ax_kde.set_ylabel("Density")

plt.show()


In [ ]:
# FUNCTIONS
# input filepath -> smoothed pitch contour
def preprocess(filepath):
    x, sr = lb.load(filepath, sr = 44100)
    xf0, xvflags, xvprobs = get_f0(x, sr, tonic = None, duration = 80, display = False) # TODO: remove duration
    xp0 = get_p0(xf0, sr, display = False)
    xp0_smooth = np.array(smooth_voiced_sections(xp0, sr, xvflags, filter_size = 7, display = False))
    times = np.array([i * 1024/sr for i in range(len(xp0_smooth))])
    return xp0_smooth, times

In [ ]:
# TEST
test_example = 'carnatic_varnam_1.0/Audio/223579__gopalkoduri__carnatic-varnam-by-prasanna-in-abhogi-raaga.mp3'
# 'carnatic_varnam_1.0/Audio/223597__gopalkoduri__carnatic-varnam-by-vignesh-in-sahana-raaga.mp3'
# 'carnatic_varnam_1.0/Audio/223604__gopalkoduri__carnatic-varnam-by-ramakrishnamurthy-in-sri-raaga.mp3'
# 'carnatic_varnam_1.0/Audio/223603__gopalkoduri__carnatic-varnam-by-badrinarayanan-in-sri-raaga.mp3'
# 'carnatic_varnam_1.0/Audio/223595__gopalkoduri__carnatic-varnam-by-ramakrishnamurthy-in-sahana-raaga.mp3' 
# 'carnatic_varnam_1.0/Audio/223590__gopalkoduri__carnatic-varnam-by-badrinarayanan-in-mohanam-raaga.mp3'
test_ex_audio, test_ex_sr = lb.load(test_example, sr = 44100)

test_p0, test_times = preprocess(test_example)
test_audio = clip_sample(test_ex_audio, sr = test_ex_sr, duration = 80)

### Component Estimation

In [ ]:
def compute_vmmc(p0_smooth, nc = None):
    angles = midi_to_angle(p0_smooth[~np.isnan(p0_smooth)])
    theta, kde = circular_vonmises_kde(angles)
    if nc:
        mus, kappas, weights, nll = fit_vmm_model(p0_smooth, nc, constrain = True)
    else:
        mus, kappas, weights = select_vmm_bic(p0_smooth, constrain=True)
    pdfs = [(weights[k] * vonmises.pdf(theta, kappas[k], loc=mus[k]), f"μ={angle_to_midi(mus[k]):.2f}, κ={kappas[k]:.1f}") for k in range(len(mus))]
    plot_pitch_model_components(ax, theta, pdfs, kde = kde, hist_data=angles, title=f"VMM-C Decomposition", polar = polar)
    return mus, kappas, weights

In [ ]:
# TEST

test_components = compute_vmmc(test_p0, nc = 5)

In [ ]:
print(test_components)

In [ ]:
def gen_vmm_df(components, dist_fn, tonic_hz = None):
    mus, kappas, weights = components
    vmm_df = pd.DataFrame(columns=['radians', 'mu', 'kappa', 'weight'])
    vmm_df['radians'] = mus
    vmm_df['mu'] = [angle_to_midi(mu) for mu in mus]
    vmm_df['kappa'] = kappas
    vmm_df['weight'] = weights
    if tonic_hz:
        best_shift = lb.hz_to_midi(tonic_hz) % 12
    else:
        best_shift, best_mse, shifts, mses, best_assignments = find_best_grid_shift_assignment(vmm_df, distance_fn = dist_fn)
    vmm_df = assign_snapped_mu(vmm_df, 0) # TODO: Change to use best_shift
    return vmm_df

In [ ]:
# TEST
test_vmm_df = gen_vmm_df(test_components, dist_squared)

In [ ]:
display(test_vmm_df)

### Tonic Estimation

In [ ]:
def compute_drone_tonic(vmm_df):
    # Build filters with current hyperparams
    filt_bank, freqs = build_vmm_harmonic_filters(
        sr=sr,
        vmm_df=vmm_df,
        tonic_hz=lb.midi_to_hz(24),
        n_fft=4096,
        num_harmonics=10,
        fixed_kappa=90,
        harmonic_decay=True,
        decay_alpha=decay_alpha,
        normalize=False,
        col = 'snapped_mu'
    )
    print(filt_bank.keys())
    # Score spectrum with filters
    scores, spectrum = score_vmm_filters_against_spectrum(x, filt_bank)
    return scores


In [ ]:
test_scores = compute_drone_tonic(test_vmm_df)

In [ ]:
print(test_scores)

### Label Estimation

In [ ]:
def compute_labels(vmm_df):
    scores = compute_drone_tonic(vmm_df)
    # TODO: finish coding this
    pass

### Contour Segmentation

In [ ]:
# FUNCTIONS
# pitch contour -> breakpoint indices of pitch contour

def segment_der(p0):
    diffs = [abs(p0[i+1] - p0[i]) for i in range(len(p0) - 1)]
    thresh = np.nanmean(diffs)
    bps_der = [i for i in range(len(diffs)) if diffs[i] >= thresh]
    return bps_der

def segment_der2(p0):
    nan_locs = np.where(np.isnan(p0))[0]
    nan_set = set(list(nan_locs))
    nan_starts = [nan for nan in nan_locs if (nan - 1) not in nan_set]

    diffs = [abs(p0[i+1] - p0[i]) for i in range(len(p0) - 1)]
    thresh = np.nanmean(diffs)
    bps_der = [i for i in range(len(diffs)) if diffs[i] >= thresh]

    der_set = set(bps_der)
    der_starts = [der for der in bps_der if (der - 1) not in der_set]
    bps_der2 = sorted(list(set(der_starts).union(set(nan_starts))))
    return bps_der2

def segment_opt(p0):
    bps_opt = [i for i in range(1, len(p0)-1) if ((p0[i] > p0[i-1] and p0[i] > p0[i + 1]) or (p0[i] < p0[i-1] and p0[i] < p0[i + 1]))]
    return bps_opt

def segment_opt2(p0):
    nan_locs = np.where(np.isnan(p0))[0]
    nan_set = set(list(nan_locs))
    nan_starts = [nan for nan in nan_locs if (nan - 1) not in nan_set]

    bps_opt = [i for i in range(1, len(p0)-1) if ((p0[i] > p0[i-1] and p0[i] > p0[i + 1]) or (p0[i] < p0[i-1] and p0[i] < p0[i + 1]))]
    bps_opt2 = sorted(list(set(bps_opt).union(set(nan_starts))))
    return bps_opt2

def segment_opt3(p0):
    nan_locs = np.where(np.isnan(p0))[0]
    nan_set = set(list(nan_locs))
    nan_starts = [nan for nan in nan_locs if (nan - 1) not in nan_set]

    opt_expanded = [i for i in range(1, len(p0) - 1) if ((p0[i] >= p0[i-1] and p0[i] > p0[i + 1]) or (p0[i] <= p0[i-1] and p0[i] < p0[i + 1]))]
    bps_opt3 = sorted(list(set(opt_expanded).union(set(nan_starts))))
    return bps_opt3

def segment_hybrid(p0):
    diffs = [abs(p0[i+1] - p0[i]) for i in range(len(p0) - 1)]
    thresh = np.nanmean(diffs)
    bps_der = [i for i in range(len(diffs)) if diffs[i] >= thresh]

    bps_opt = [i for i in range(1, len(p0)-1) if ((p0[i] > p0[i-1] and p0[i] > p0[i + 1]) or (p0[i] < p0[i-1] and p0[i] < p0[i + 1]))]
    
    tot = set(bps_der)
    tot.update(set(bps_opt))
    bps_hybrid = sorted(list(tot))
    return bps_hybrid

def segment_known(p0):
    bps = None # Insert your breakpoint indices here
    return bps

In [ ]:
test_theta = (test_p0 % 12) * (2*np.pi/12)

test_bps = segment_opt3(test_p0)
test_bps = np.array(sorted(set([0, *test_bps, len(theta)-1])))

### Tempo Estimation

In [ ]:
# FUNCTIONS
# filepath -> akshara duration in seconds

def tempo_estimation(filepath):
    x, sr = lb.load(filepath, sr = 44100)
    xonset_times = detect_syllable_onsets(x, sr, delta = 0.054, wait = 35)
    xonset_inds = times_to_indices(xonset_times)
    onset_time_list.append(xonset_times)
    onset_ind_list.append(xonset_inds)
    beat_est, _, _ = estimate_beat_from_onsets(xonset_times, min_beat = 0.25, max_beat = 0.50, plot=False)
    return beat_est

In [ ]:
test_beat_est = tempo_estimation(test_example)

In [ ]:
print(test_beat_est)

### DP Button

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from scipy.special import logsumexp, softplus

# ---------- Circular utilities ----------
def circ_mean(theta):
    """Circular mean in [0, 2π), ignoring NaNs. Returns np.nan if no finite samples."""
    theta = np.asarray(theta)
    valid = np.isfinite(theta)
    if not np.any(valid):
        return np.nan
    z = np.exp(1j * theta[valid])
    m = np.angle(np.mean(z))
    return (m + 2*np.pi) % (2*np.pi)

def circ_resultant_length(theta):
    """Mean resultant length R∈[0,1], ignoring NaNs. Returns np.nan if no finite samples."""
    theta = np.asarray(theta)
    valid = np.isfinite(theta)
    if not np.any(valid):
        return np.nan
    z = np.exp(1j * theta[valid])
    return np.abs(np.mean(z))


def circ_dist(a, b):
    # shortest signed distance a-b on circle, result in [-pi, pi]
    d = (a - b + np.pi) % (2*np.pi) - np.pi
    return d

# ---------- VMM utilities ----------
@dataclass
class VMM:
    mu: np.ndarray     # shape (K,), radians
    kappa: np.ndarray  # shape (K,), >0
    weight: np.ndarray # shape (K,), >=0, sum ~ 1

    @classmethod
    def from_df(cls, vmm_df: pd.DataFrame):
        # gets infor form dataframe
        w = vmm_df['weight'].to_numpy(dtype=float)
        w = w / (w.sum() + 1e-12)
        return cls(
            mu=vmm_df['radians'].to_numpy(dtype=float), # CHANGED: from mu to radians, mu will now be semitone
            kappa=vmm_df['kappa'].to_numpy(dtype=float),
            weight=w
        )

    def logpdf_mixture(self, theta: np.ndarray) -> np.ndarray:
        """
        Return log( sum_k w_k * VM_k(theta) ) for each theta.
        """
        # compute log w_k + log VM_k(theta) in a (len(theta), K) matrix
        K = len(self.mu)
        log_w = np.log(self.weight + 1e-18)
        # Broadcast over samples x components
        # vonmises.pdf(theta, kappa, loc=mu)
        # For numerical stability, compute per-component logpdf then logsumexp
        logpdf_components = np.stack([
            log_w[k] + vonmises.logpdf(theta, self.kappa[k], loc=self.mu[k])
            for k in range(K)
        ], axis=1)  # shape (n, K)
        return logsumexp(logpdf_components, axis=1)  # shape (n,)

    def component_posteriors(self, theta: np.ndarray) -> np.ndarray:
        """
        Return responsibilities p(k | theta_t) as (n, K).
        """
        K = len(self.mu)
        log_w = np.log(self.weight + 1e-18)
        logpdf_components = np.stack([
            log_w[k] + vonmises.logpdf(theta, self.kappa[k], loc=self.mu[k])
            for k in range(K)
        ], axis=1)
        logZ = logsumexp(logpdf_components, axis=1, keepdims=True)
        return np.exp(logpdf_components - logZ)  # (n, K)

    def best_label_for_segment(self, theta: np.ndarray) -> int:
        """
        MAP component for the segment, ignoring NaNs. If no finite frames, fall back to the
        heaviest-weight component (this label won't be used if the candidate is disqualified).
        """
        theta = np.asarray(theta)
        valid = np.isfinite(theta)
        if not np.any(valid):
            return int(np.argmax(self.weight))

        theta = theta[valid]
        K = len(self.mu)
        scores = []
        for k in range(K):
            s = np.log(self.weight[k] + 1e-18) + np.sum(
                vonmises.logpdf(theta, self.kappa[k], loc=self.mu[k])
            )
            scores.append(s)
        return int(np.argmax(scores))


# ---------- Segment scoring terms ----------
@dataclass
class ScoringParams:
    aksh_dur: float          # seconds per akshara (T)
    # duration bounds in aksharas (hard gating)
    min_r: float = 0.5           # e.g., at least half an akshara
    max_r: float = 8.0           # e.g., cap long notes at 8 aksharas
    # existing weights
    alpha_align: float = 1.0
    delta_cap: float = 0.25
    beta_long: float = 0.5
    r_long: float = 4.0
    gamma_stab: float = 0.8
    r_stab: float = 3.0
    lambda_entropy: float = 0.0
    label_mode: str = "summary"  # "mixture_ll" | "summary"

    # NEW: for single-component MAP scoring (used when label_mode == "mixture_ll")
    include_weight_prior: bool = False   # if True, add log(weight_k) to the MAP score

def akshara_alignment_cost(duration, params: ScoringParams):
    r = duration / params.aksh_dur
    dist_to_int = abs(r - np.round(r))
    return params.alpha_align * min(dist_to_int, params.delta_cap) # min enforces a flat error between 0.25 and 0.5 off from nearest multiple

def long_duration_cost(duration, params: ScoringParams):
    r = duration / params.aksh_dur
    return params.beta_long * softplus(r - params.r_long)

def stability_cost(theta_seg, duration, params: ScoringParams):
    # keep only finite frames
    theta_seg = np.asarray(theta_seg)
    valid = np.isfinite(theta_seg)
    if not np.any(valid):
        return np.inf  # no voiced/finite frames → disqualify this candidate

    theta_seg = theta_seg[valid]
    R = circ_resultant_length(theta_seg)  # already NaN-safe
    if not np.isfinite(R):
        return np.inf

    r = duration / params.aksh_dur
    val = params.gamma_stab * softplus((r - params.r_stab) * (1.0 - R))
    return float(val) if np.isfinite(val) else np.inf

def label_cost(theta_seg, vmm: VMM, params: ScoringParams):
    # NaN-safe: drop non-finite frames
    theta_seg = np.asarray(theta_seg)
    valid = np.isfinite(theta_seg)
    if not np.any(valid):
        return np.inf
    theta_seg = theta_seg[valid]

    if params.label_mode == "mixture_ll":
        # single-component MAP, returned as a *cost* to minimize
        K = len(vmm.mu)
        scores = []
        for k in range(K):
            ll = np.sum(vonmises.logpdf(theta_seg, vmm.kappa[k], loc=vmm.mu[k]))
            if params.include_weight_prior:
                ll += np.log(vmm.weight[k] + 1e-18)   # add once per segment
            scores.append(ll)

        cost = -float(np.max(scores))                 # negate to get a cost
        return cost if np.isfinite(cost) else np.inf

    elif params.label_mode == "summary":
        # (keep your existing summary-mode; with the corrected kappa usage if you updated it)
        m = circ_mean(theta_seg)
        if not np.isfinite(m):
            return np.inf
        d2 = (circ_dist(m, vmm.mu))**2
        if not np.all(np.isfinite(d2)):
            return np.inf
        # treat kappa as precision (multiplicative); optional prior term could be added here too
        score = np.min(d2 * (vmm.kappa + 1e-6) - np.log(vmm.weight + 1e-18))
        L = max(1.0, len(theta_seg) / 100.0)
        val = score * L
        return float(val) if np.isfinite(val) else np.inf

    else:
        raise ValueError("Unknown label_mode")


def segment_label_map_single_component(theta_seg, vmm: VMM, include_weight_prior: bool = False) -> int:
    theta_seg = np.asarray(theta_seg)
    valid = np.isfinite(theta_seg)
    if not np.any(valid):
        return int(np.argmax(vmm.weight))  # harmless fallback; segment will be disqualified upstream
    theta_seg = theta_seg[valid]

    best_k = 0
    best_s = -np.inf
    for k in range(len(vmm.mu)):
        s = np.sum(vonmises.logpdf(theta_seg, vmm.kappa[k], loc=vmm.mu[k]))
        if include_weight_prior:
            s += np.log(vmm.weight[k] + 1e-18)
        if s > best_s:
            best_s = s
            best_k = k
    return int(best_k)



def transition_cost(last_label: Optional[int], curr_label: Optional[int], tau: float = 0.0):
    if tau <= 0.0 or last_label is None or curr_label is None:
        return 0.0
    # Example: small penalty for large interval jumps on the circle of 12 (if your labels map to pitch classes)
    # If your labels map directly to VMM components, you can define a label→pc map and use that here.
    # Placeholder: no-op unless tau>0 and mapping added.
    # TODO: incorporate in the future
    return 0.0

# ---------- Dynamic Programming ----------
def dynamic_programming_segmentation(
    times: np.ndarray,
    theta: np.ndarray,
    breakpoints: np.ndarray,
    vmm_df: pd.DataFrame,
    params: ScoringParams,
    max_lookback_aksharas: Optional[float] = 8.0,  # prune search
    transition_tau: float = 0.0,
) -> Dict:
    """
    Returns:
      {
        'segments': List[(start_idx, end_idx)],
        'labels':   List[int],            # component indices for each segment
        'total_score': float,
        'DP': np.ndarray,                 # DP array for debugging
      }
    """
    vmm = VMM.from_df(vmm_df)
    B = np.asarray(breakpoints, dtype=int)
    N = len(B) - 1  # number of candidate *intervals* is N
    # map breakpoint index c in [0..N] to absolute sample index B[c]

    # Precompute segment-level caches for speed
    # Get durations and per-segment label costs for all j<i (sparse if we prune)
    DP = np.full(N+1, np.inf)
    prev = np.full(N+1, -1, dtype=int)
    seg_label = [[None]*(N+1) for _ in range(N+1)]  # store winning label per (j,i) if you want
    # start at breakpoint 0 (assumed start of track)
    DP[0] = 0.0

    # Helper to bound lookback by time / aksharas
    def ok_lookback(j, i):
        if max_lookback_aksharas is None:
            return True
        d = times[B[i]] - times[B[j]]
        return (d / params.aksh_dur) <= max_lookback_aksharas + 1e-9

    # Optional: keep last assigned label when evaluating transition cost
    # (Here we compute best (j,label) jointly by evaluating segment’s best label only;
    # for label-dependent transitions, you’d expand state to include last_label.)
# --- inside dynamic_programming_segmentation(), replace the inner loop body ---
    for i in range(1, N+1):
        best_val = np.inf
        best_j = -1

        for j in range(max(0, i-200), i):
            if not ok_lookback(j, i):
                continue

            s_idx, e_idx = B[j], B[i]
            theta_seg = theta[s_idx:e_idx]
            duration = times[e_idx] - times[s_idx]
            if duration <= 0:
                continue

            # --- NEW: hard duration constraints in aksharas ---
            r = duration / params.aksh_dur
            if (r < params.min_r) or (r > params.max_r):
                continue

            # costs (unchanged)
            c_tempo = akshara_alignment_cost(duration, params) + long_duration_cost(duration, params)
            c_stab  = stability_cost(theta_seg, duration, params)
            c_label = label_cost(theta_seg, vmm, params)

            if params.label_mode == "mixture_ll":
                label_i = segment_label_map_single_component(theta_seg, vmm, include_weight_prior=params.include_weight_prior)
            else:
                label_i = vmm.best_label_for_segment(theta_seg)  # or your summary-mode choice
            seg_label[j][i] = label_i
            seg_label[j][i] = label_i

            total = DP[j] + (c_tempo + c_stab + c_label)
            if total < best_val:
                best_val = total
                best_j = j

        DP[i] = best_val
        prev[i] = best_j


    # Backtrack
    segments = []
    labels = []
    i = N
    while i > 0:
        j = prev[i]
        if j < 0:
            # fallback: if something went wrong, connect to previous
            j = i-1
        segments.append((B[j], B[i]))
        labels.append(seg_label[j][i])
        i = j

    segments.reverse()
    labels.reverse()

    return {
        'segments': segments,
        'labels': labels,
        'total_score': float(DP[N]),
        'DP': DP,
    }


In [ ]:
# Helper to print how many candidate intervals get filtered by min/max
def count_filtered_candidates(times, breakpoints, params):
    import numpy as np
    B = np.asarray(breakpoints, dtype=int)
    total = 0
    kept = 0
    for i in range(1, len(B)):
        for j in range(max(0, i-200), i):
            d = times[B[i]] - times[B[j]]
            if d <= 0: 
                continue
            r = d / params.aksh_dur
            total += 1
            if (params.min_r <= r <= params.max_r) or (i == len(B)-1 and r <= params.max_r + 2.0):
                kept += 1
    print(f"Candidate intervals considered: {total}, kept after bounds: {kept} ({kept/total:.1%})")

In [ ]:
# Choose params (feel free to tweak)
params = ScoringParams(
    aksh_dur=test_beat_est,
    min_r=1.0,   # >= half an akshara
    max_r=8.0,   # cap very long spans
    alpha_align=1.0,
    beta_long=0.4,
    r_long=3.0,
    gamma_stab=0.8,
    r_stab=4.0,
    lambda_entropy=0.0,
    label_mode="mixture_ll",
    include_weight_prior = False  # default
)

# Sanity report:
count_filtered_candidates(test_times, test_bps, params)

In [ ]:
# Run DP (uses the updated version you have)
dp_out = dynamic_programming_segmentation(
    times=test_times,
    theta=test_theta,
    breakpoints=test_bps,
    vmm_df=test_vmm_df,
    params=params,
    max_lookback_aksharas=8.0,    # pruning; adjust as needed
)

pred_segments = dp_out['segments']   # list of (start_idx, end_idx)
pred_labels   = dp_out['labels']     # list of component ids (same length as segments)


In [ ]:
print(test_beat_est)

In [ ]:
print(test_bps)

In [ ]:
print(dp_out['DP'])

In [ ]:
print(pred_segments)
print(test_beat_est * 44100/1024)
print([(interval[1] - interval[0])/15.62 for interval in pred_segments])

In [ ]:
print(pred_labels)

In [ ]:
test_boundaries = np.array([interval[0] * 1024/44100 for interval in pred_segments])

Audio(add_clicks_to_audio(test_audio, 44100, test_boundaries), rate = 44100)

In [ ]:
# 1) Sanity-print the key settings & frame step
print("aksh_dur (s):", params.aksh_dur)
print("min_r, max_r:", params.min_r, params.max_r)
dt = float(np.median(np.diff(test_times)))
print("median frame dt (s):", dt, " (~", round(params.aksh_dur/dt, 2), "frames per akshara)")

# 2) Check all chosen segments from dp_out for violations
def check_chosen_segments(dp_out, times, params):
    bad = []
    for (s, e) in dp_out['segments']:
        dur = times[e] - times[s]
        r = dur / params.aksh_dur
        if r < params.min_r - 1e-9 or r > params.max_r + 1e-9:
            bad.append((s, e, dur, r))
    print(f"Segments: {len(dp_out['segments'])}, violations: {len(bad)}")
    if bad[:5]:
        print("First few violations (s_idx, e_idx, dur_s, aksharas):")
        for row in bad[:5]:
            print(row)
    return bad

_ = check_chosen_segments(dp_out, test_times, params)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def build_segment_cost_matrix(times, theta, breakpoints, vmm_df, params,
                              max_lookback_aksharas=8.0,
                              allow_final_relax=True, relax_aksharas=2.0):
    """
    Returns:
      C_local: (N+1, N+1) array with cost(j→i) for j<i, else np.nan
      valid_mask: boolean mask where candidates were considered
    """
    B = np.asarray(breakpoints, dtype=int)
    N = len(B) - 1
    vmm = VMM.from_df(vmm_df)

    def ok_lookback(j,i):
        if max_lookback_aksharas is None:
            return True
        d = times[B[i]] - times[B[j]]
        return (d / params.aksh_dur) <= max_lookback_aksharas + 1e-9

    C_local = np.full((N+1, N+1), np.nan, dtype=float)
    valid_mask = np.zeros_like(C_local, dtype=bool)

    for i in range(1, N+1):
        for j in range(max(0, i-200), i):
            if not ok_lookback(j,i):
                continue
            s_idx, e_idx = B[j], B[i]
            duration = times[e_idx] - times[s_idx]
            if duration <= 0:
                continue
            r = duration / params.aksh_dur
            # hard gating with optional final relax
            if allow_final_relax and (i == N):
                if not (params.min_r <= r <= params.max_r + relax_aksharas):
                    continue
            else:
                if not (params.min_r <= r <= params.max_r):
                    continue

            theta_seg = theta[s_idx:e_idx]
            c_tempo = akshara_alignment_cost(duration, params) + long_duration_cost(duration, params)
            c_stab  = stability_cost(theta_seg, duration, params)
            c_label = label_cost(theta_seg, vmm, params)
            C_local[j, i] = c_tempo + c_stab + c_label
            valid_mask[j, i] = True

    return C_local, valid_mask

def segments_to_bp_edges(pred_segments, breakpoints):
    """
    Map (start_idx, end_idx) pairs back to breakpoint indices (j,i).
    """
    B = np.asarray(breakpoints, dtype=int)
    pos = {int(b): k for k, b in enumerate(B)}
    edges = []
    for (s,e) in pred_segments:
        j = pos.get(int(s), None)
        i = pos.get(int(e), None)
        if j is not None and i is not None and j < i:
            edges.append((j,i))
    return edges

def plot_cost_heatmap(C_local, edges_bp=None, title="Segment cost heatmap (local)"):
    """
    C_local: (N+1, N+1) with np.nan for invalid cells; rows=j (prev), cols=i (curr)
    edges_bp: list of (j,i) along optimal path to overlay
    """
    # Mask NaNs for plotting
    M = np.ma.masked_invalid(C_local)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(M, origin='lower', interpolation='nearest', aspect='auto')
    ax.set_title(title)
    ax.set_xlabel("i (current breakpoint index)")
    ax.set_ylabel("j (previous breakpoint index)")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("$C(i, j) = C_{tempo}(i, j) + C_{label}(i, j)$")

    # overlay optimal path
    if edges_bp:
        xs = [i for (_,i) in edges_bp]
        ys = [j for (j,_) in edges_bp]
        ax.scatter(xs, ys, s=40, marker='s', edgecolors='k', facecolors='none', linewidths=1.0)

        # (Optional) connect them in chronological order
        # This draws small line segments roughly from (j,i) to next (j',i')
        for (j1,i1),(j2,i2) in zip(edges_bp[:-1], edges_bp[1:]):
            ax.plot([i1, i2], [j1, j2], linewidth=1.0)

    # Limit axes to the used triangular region
    n = C_local.shape[0]
    ax.set_xlim(-0.5, n-0.5)
    ax.set_ylim(-0.5, n-0.5)
    plt.tight_layout()
    plt.show()

def plot_cumulative_cost_heatmap(C_local, DP, title="Cumulative one-step cost heatmap (DP[j] + local)"):
    """
    Shows DP[j] + C_local[j,i] for valid (j,i). Highlights how state affects choices.
    """
    JI = np.indices(C_local.shape)
    J = JI[0]
    C_cum = C_local + DP[J]  # broadcasts DP over rows
    C_cum[~np.isfinite(C_local)] = np.nan  # keep invalid as NaN
    plot_cost_heatmap(C_cum, edges_bp=None, title=title)


In [ ]:
# 1) Build local cost matrix with the same gating as DP
C_local, valid_mask = build_segment_cost_matrix(
    times=test_times,
    theta=test_theta,
    breakpoints=test_bps,
    vmm_df=test_vmm_df,
    params=params,
    max_lookback_aksharas=8.0,     # match your DP pruning
    allow_final_relax=True,        # match your final-segment relax policy
    relax_aksharas=2.0
)

# 2) Convert DP output segments to breakpoint edges
edges_bp = segments_to_bp_edges(dp_out['segments'], test_bps)

# 3) Plot local cost heatmap + optimal path
plot_cost_heatmap(C_local, edges_bp, title="$C(i, j)$ and Optimal Path")

# 4) (Optional) Plot cumulative one-step costs
plot_cumulative_cost_heatmap(C_local, dp_out['DP'], title="DP[j] + local segment cost (valid j→i)")


In [ ]:
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path

def _write_xml(root, out_path):
    tree = ET.ElementTree(root)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    tree.write(out_path, encoding="utf-8", xml_declaration=True)

def _sec_to_frames(t, sr):  # seconds -> audio sample frames (int)
    return int(round(float(t) * float(sr)))

def export_timevalues_svl(times_sec, values, sample_rate, out_path, units=None, layer_name="PitchContour"):
    times_sec = np.asarray(times_sec, float)
    values = np.asarray(values, float)
    assert times_sec.shape == values.shape

    root = ET.Element("sv")
    layer_attrs = {"type": "timevalues", "name": layer_name, "sampleRate": str(int(sample_rate))}
    if units: layer_attrs["units"] = units
    layer = ET.SubElement(root, "layer", layer_attrs)

    finite = np.isfinite(values)
    for t, v in zip(times_sec[finite], values[finite]):
        ET.SubElement(layer, "point", {
            "frame": str(_sec_to_frames(t, sample_rate)),
            "value": str(float(v))
        })

    _write_xml(root, out_path)
    return out_path

def export_regions_svl(segments, labels, times_sec, sample_rate, out_path, values=None, layer_name="Segments"):
    """
    segments: list of (start_idx, end_idx) in index space of times_sec (end exclusive)
    labels:   same length; ints/strings
    values:   optional per-region Y placement (float); default 0.0
    """
    n = len(times_sec)
    if values is None:
        values = [0.0] * len(segments)

    root = ET.Element("sv")
    layer = ET.SubElement(root, "layer", {
        "type": "regions",
        "name": layer_name,
        "sampleRate": str(int(sample_rate))
    })

    # estimate dt for duration edge coverage (so region is visible)
    if n > 1:
        dt = float(np.median(np.diff(times_sec)))
    else:
        dt = 0.0

    for (s_idx, e_idx), lab, val in zip(segments, labels, values):
        s_idx = int(max(0, min(s_idx, n-1)))
        e_idx = int(max(s_idx+1, min(e_idx, n)))
        t0 = float(times_sec[s_idx])
        t1 = float(times_sec[e_idx-1] + dt)  # cover up to the end of last frame
        f0 = _sec_to_frames(t0, sample_rate)
        dur = max(1, _sec_to_frames(t1, sample_rate) - f0)
        ET.SubElement(layer, "region", {
            "frame": str(f0),
            "duration": str(dur),
            "value": str(float(val)),
            "label": str(lab)
        })

    _write_xml(root, out_path)
    return out_path


In [ ]:
# segments: list of (start_idx, end_idx) in the index space of your `times` array
# labels:   list of ints/strings (same length as segments)
# times:    np.ndarray of seconds aligned to your pitch frames (len = N)
# sr:       your audio sample rate (e.g., 48000.0 if that’s what your WAV uses)

export_regions_svl(pred_segments, pred_labels, test_times, test_ex_sr, "test_notes_regions.svl")

# For pitch contour (e.g., cents or Hz); NaNs are fine (they’ll create gaps):
export_timevalues_svl(test_times, test_p0, sr, "test_pitch_contour.svl", units="cents")  # or "Hz"



In [ ]:
def _intervals_from_breaks_and_labels(break_idxs, labels):
    """Yield (start_idx, end_idx, label) for each predicted interval."""
    if len(break_idxs) != len(labels) + 1:
        raise ValueError("break_idxs must be exactly one longer than labels.")
    b = sorted(set(int(i) for i in break_idxs))
    if len(b) != len(break_idxs):
        raise ValueError("break_idxs contain duplicates after sorting; intervals would be ill-defined.")
    for i, lab in enumerate(labels):
        s, e = b[i], b[i+1]
        if e > s:
            yield (s, e, lab)

def _overlap_len(a_start, a_end, b_start, b_end):
    """Length of intersection of [a_start, a_end) and [b_start, b_end) in index units (>= 0)."""
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def label_time_coverage(
    break_idxs, labels, 
    gt_segments_df: pd.DataFrame, 
    gt_start_col: str = "p0_start", 
    gt_end_col: str = "p0_end", 
    gt_label_col: str = "label"
):
    """
    Compute fraction of total sample duration that is assigned the correct swaram label.
    - break_idxs, labels: DP output intervals and labels (index domain).
    - gt_segments_df: contains GT segments with start/end indices and swaram label.
    Returns dict with coverage_fraction, correct_frames, total_frames.
    """
    # Total duration from GT (robust if GT doesn’t cover leading/trailing silence).
    gt_sorted = gt_segments_df.sort_values(by=gt_start_col).reset_index(drop=True)
    total_frames = int((gt_sorted[gt_end_col] - gt_sorted[gt_start_col]).clip(lower=0).sum())
    if total_frames == 0:
        return {"coverage_fraction": 0.0, "correct_frames": 0, "total_frames": 0}

    # Preload GT segments as tuples for fast overlap checks
    gt_intervals = [
        (int(r[gt_start_col]), int(r[gt_end_col]), r[gt_label_col])
        for _, r in gt_sorted.iterrows()
        if int(r[gt_end_col]) > int(r[gt_start_col])
    ]

    correct = 0
    for ps, pe, plab in _intervals_from_breaks_and_labels(break_idxs, labels):
        # Overlap each predicted interval with GT intervals; add overlap where labels match
        for gs, ge, glab in gt_intervals:
            ov = _overlap_len(ps, pe, gs, ge)
            if ov > 0 and plab == glab:
                correct += ov

    return {
        "coverage_fraction": correct / total_frames if total_frames > 0 else 0.0,
        "correct_frames": int(correct),
        "total_frames": int(total_frames),
    }

import numpy as np

def map_pred_labelidx_to_semitones(pred_label_idx, test_vmm_df, label_col="swara_label"):
    """
    pred_label_idx : 1D array-like of ints (positional indices into test_vmm_df)
    test_vmm_df    : DataFrame with a column (default 'swaram_label') holding semitone labels
    Returns a NumPy array of semitone labels aligned to pred_label_idx order.
    """
    idx = np.asarray(pred_label_idx, dtype=int)
    labels = test_vmm_df[label_col].to_numpy()   # vector of semitone labels, length == len(test_vmm_df)
    if (idx.min() < 0) or (idx.max() >= labels.shape[0]):
        raise IndexError("pred_label_idx contains out-of-range row positions for test_vmm_df.")
    return labels[idx]


In [ ]:
import pandas as pd

# 3) per-swaram breakdown (how much of each GT swaram was labeled correctly)
def overlap_stats_by_label(
    break_idxs, pred_labels_semitone, gt_df,
    gt_start_col="p0_start", gt_end_col="p0_end", gt_label_col="label"
):
    # Reuse the same interval iterator and overlap function from earlier
    def _intervals_from_breaks_and_labels(break_idxs, labels):
        if len(break_idxs) != len(labels) + 1:
            raise ValueError("break_idxs must be exactly one longer than labels.")
        b = sorted(set(int(i) for i in break_idxs))
        if len(b) != len(break_idxs):
            raise ValueError("break_idxs contain duplicates after sorting; intervals would be ill-defined.")
        for i, lab in enumerate(labels):
            s, e = b[i], b[i+1]
            if e > s:
                yield (s, e, lab)

    def _overlap_len(a_start, a_end, b_start, b_end):
        return max(0, min(a_end, b_end) - max(a_start, b_start))

    gt_sorted = gt_df.sort_values(by=gt_start_col).reset_index(drop=True)
    gt_intervals = [
        (int(r[gt_start_col]), int(r[gt_end_col]), r[gt_label_col])
        for _, r in gt_sorted.iterrows()
        if int(r[gt_end_col]) > int(r[gt_start_col])
    ]

    # total frames per GT label
    totals = {}
    for s, e, lab in gt_intervals:
        totals[lab] = totals.get(lab, 0) + (e - s)

    # correct frames per GT label (pred == GT over the overlap)
    correct = {lab: 0 for lab in totals.keys()}
    for ps, pe, plab in _intervals_from_breaks_and_labels(break_idxs, pred_labels_semitone):
        for gs, ge, glab in gt_intervals:
            ov = _overlap_len(ps, pe, gs, ge)
            if ov > 0 and plab == glab:
                correct[glab] = correct.get(glab, 0) + ov

    rows = []
    for lab, tot in totals.items():
        c = int(correct.get(lab, 0))
        rows.append({
            "swaram": lab,
            "correct_frames": c,
            "total_frames": int(tot),
            "coverage_fraction": (c / tot) if tot > 0 else 0.0
        })
    return pd.DataFrame(rows).sort_values("swaram").reset_index(drop=True)

# ---- assume these exist from your pipeline ---


#### BUTTON

In [ ]:
def dp_button(filepath):
    test_example = filepath
    artist = test_example.split("-")[3]
    raga = test_example.split("-")[5]
    row = kde_df[(kde_df['raga'] == raga) & (kde_df['artist'] == artist)]
    
    test_ex_audio, test_ex_sr = lb.load(test_example, sr = 44100)

    test_p0, test_times = preprocess(test_example)
    test_audio = clip_sample(test_ex_audio, sr = test_ex_sr, duration = 80)

    # test_components = compute_vmmc(test_p0, nc = 5)
    # generate vmm df

    test_vmm_df = vonmises_df[(vonmises_df['raga'] == raga) & (vonmises_df['artist'] == artist)]
    test_vmm_df = vonmises_df[(vonmises_df['raga'] == raga) & (vonmises_df['artist'] == artist)]
    tonic_midi = lb.hz_to_midi(tonics[artist])

    test_vmm_df['mu'] = (test_vmm_df['mu'] + tonic_midi) % 12
    test_vmm_df['radians'] = midi_to_angle(test_vmm_df['mu'])

    test_theta = (test_p0 % 12) * (2*np.pi/12)

    test_bps = segment_opt3(test_p0)
    test_bps = np.array(sorted(set([0, *test_bps, len(theta)-1])))

    # test_beat_est = tempo_estimation(test_example)
    test_beat_est = row['beat_mu'].iloc[0]

    params = ScoringParams(
        aksh_dur=test_beat_est,
        min_r=1.0,   # >= half an akshara
        max_r=8.0,   # cap very long spans
        alpha_align=1.0,
        beta_long=0.4,
        r_long=3.0,
        gamma_stab=0.8,
        r_stab=4.0,
        lambda_entropy=0.0,
        label_mode="mixture_ll",
        include_weight_prior = False  # default
        )

    # Run DP (uses the updated version you have)
    dp_out = dynamic_programming_segmentation(
        times=test_times,
        theta=test_theta,
        breakpoints=test_bps,
        vmm_df=test_vmm_df,
        params=params,
        max_lookback_aksharas=8.0,    # pruning; adjust as needed
    )

    pred_segments = dp_out['segments']   # list of (start_idx, end_idx)
    pred_labels   = dp_out['labels']     # list of component ids (same length as segments)

    return pred_segments, pred_labels, test_vmm_df

In [ ]:
i = 0

for key, value in dataset.items():
    i += 1
    if i < 3:
        item, x, sr = value
        artist = item.split("-")[3]
        raga = item.split("-")[5]

        print(raga, artist)
        
        xsegdf = all_segments[(all_segments['raga'] == raga) & (all_segments['artist'] == artist)]
        xbps_gt = xsegdf['p0_start']

        pred_segments, pred_labels, test_vmm_df = dp_button(item)
        last_ind = pred_segments[-1][-1]
        xbps_gt = [gt for gt in xbps_gt if gt <= last_ind]
        res_x, match_x = evaluate_segmentation(pred_segments, xbps_gt, tolerance=5, return_matches=True)
        print(res_x)

        # dp_break_idxs: list[int], length = num_intervals + 1, in pitch-contour index domain
        # pred_label_idx: list[int], DP-chosen row positions into test_vmm_df
        # test_vmm_df: DataFrame with column "swaram_label" (semitones)
        # gt_segments_df: DataFrame with columns ["p0_start", "p0_end", "label"] in pitch-contour index domain
        # ------------------------------------------------

        xsegdf = xsegdf[xsegdf['p0_end'] <= last_ind]
        # 1) map predicted row indices to semitone labels for intervals
        pred_semitones = map_pred_labelidx_to_semitones(pred_labels, test_vmm_df, "swara_label")
        dp_break_idxs = [seg[0] for seg in pred_segments]
        dp_break_idxs.append(pred_segments[-1][-1])
        # 2) overall coverage (fraction of GT duration labeled correctly)
        overall = label_time_coverage(
            dp_break_idxs,
            pred_semitones,          # now semitone labels
            xsegdf,
            gt_start_col="p0_start",
            gt_end_col="p0_end",
            gt_label_col="label"     # GT semitone label column
        )
        print({
            "coverage_fraction": overall["coverage_fraction"],
            "correct_frames": overall["correct_frames"],
            "total_frames": overall["total_frames"],
        })

        per_label_df = overlap_stats_by_label(
            dp_break_idxs, pred_semitones, xsegdf,
            gt_start_col="p0_start", gt_end_col="p0_end", gt_label_col="label"
        )
        print(per_label_df)
        

### Summary

In [ ]:
vonmises_df.head()

In [ ]:
from xml.sax.saxutils import escape

def svl_regions_from_df(
    df,
    *,
    start_col="p0_start",      # start index in pitch-contour units
    end_col="p0_end",          # end index in pitch-contour units
    label_col="label",         # text you want to show in SV (e.g., semitone or swaram)
    sample_rate=44100,
    hop_length=1024,           # convert contour indices -> audio frames
    start_frame_offset=0,      # e.g., 3072 if your existing SV layers begin there
    layer_name="Labeled Segments",
):
    """
    Build an SVL <layer type="regions"> string directly from a dataframe of labeled segments.
    Assumes [start_col, end_col) are contiguous indices in the pitch-contour index domain.
    """
    def idx_to_frame(idx: int) -> int:
        return int(start_frame_offset + idx * hop_length)

    # Clean + sort
    d = df[[start_col, end_col, label_col]].copy()
    d = d.dropna(subset=[start_col, end_col]).sort_values(start_col)
    # Keep only proper intervals
    d = d[d[end_col].astype(int) > d[start_col].astype(int)]

    regions_lines = []
    for _, r in d.iterrows():
        s_idx = int(r[start_col])
        e_idx = int(r[end_col])
        s_fr  = idx_to_frame(s_idx)
        e_fr  = idx_to_frame(e_idx)
        lab   = "" if pd.isna(r[label_col]) else str(r[label_col])
        regions_lines.append(
            f'            <region start="{s_fr}" end="{e_fr}" label="{escape(lab)}" />'
        )

    regions_xml = "\n".join(regions_lines) if regions_lines else ""
    layer_xml = f"""<layer name="{escape(layer_name)}" type="regions">
        <model type="regions" sampleRate="{sample_rate}" resolution="1" notifyOnAdd="false" />
        <dataset>
{regions_xml}
        </dataset>
    </layer>"""
    return layer_xml


In [ ]:
def wrap_layers_in_svl(layers_xml, title="Annotations"):
    joined = "\n\n".join(layers_xml)
    return f"""<?xml version='1.0' encoding='utf-8'?>
<sv>
  <data>
{joined}
  </data>
</sv>
"""

# --- usage ---
# gt_segments_df has columns p0_start, p0_end, and label (or swaram label)
layer = svl_regions_from_df(
    ex_seg_df,
    start_col="p0_start",
    end_col="p0_end",
    label_col="label",      # change to your swaram column if different
    sample_rate=44100,
    hop_length=1024,
    start_frame_offset=0,   # set to 3072 if your other layers start there
    layer_name="GT Labeled Segments"
)

svl_text = wrap_layers_in_svl([layer])
with open("segments_regions.svl", "w", encoding="utf-8") as f:
    f.write(svl_text)
